# Train the order-flow transformer (runs in Colab)

**First run `fetch_itch_colab.ipynb`** (CPU runtime): it adds the market ETF quotes
(`MyDrive/lobgen/market/`) that `v4` reads.

1. Builds the five LOBSTER days for GOOG and INTC from `MyDrive/lobgen/filtered/` (or reuses them
   from Drive) and checks
   that the simulator reproduces every row exactly (same check as on the laptop).
2. A 300-step smoke run checks that GPU training gives finite losses.
3. Trains `v2` (v1's size, with the v2 action encoding and day-so-far features), `full`
   (full size), `v3` (v2 plus order-flow memory inputs) and `v4` (v3 plus the market: the
   stock's own clock-time moves and the QQQ / SMH quotes, the channel news arrives through),
   and optionally the ablations (no day-so-far features, no atoms, no book, short context).
   Checkpoints go to `MyDrive/lobgen/checkpoints/`; finished runs are skipped.
4. Runs the detective on the test day for the baselines and every model (one ticker at a time), saving
   `MyDrive/lobgen/results/detective.json` after each run.
5. Shock test for v3 and v4 (reaction to large market orders; `shock.json`) and tracking test
   (given the market's real path, does the generated stock follow the real one? plus a news
   scenario; `track.json`). v2 and the baselines are already done on the laptop.

**Run:** Runtime -> Change runtime type -> GPU (A100 or L4 if available, T4 works). Then
Runtime -> Run all and approve the Drive prompt. **Resumable:** if the session drops, run all
again: the prepared data is reused from Drive, an interrupted training run continues from its last
evaluation (at most 500 steps lost), and finished runs and detective entries are skipped. To stop
early: interrupt the cell, then Runtime -> Disconnect and delete runtime.

In [ ]:
DRIVE = "/content/drive/MyDrive/lobgen"
REPO = "/content/lobgen"
DATES = ["2025-12-08", "2025-12-09", "2025-12-10", "2025-12-11", "2025-12-12"]
TICKERS = ["GOOG", "INTC"]
STEPS = 20_000
RUNS = {  # name -> extra train.py flags
    "v2": "--d-model 192 --layers 4 --heads 6 --ctx 128",   # v1's size: new encoding + day-so-far
    "full": "",                                             # 5.3M params, ctx 256
    # v4 first: it is the headline model. v3 (= v4 without the market inputs) is the ablation that
    # says whether a v4 gain comes from the market/news channel or from the order-flow memory.
    "v4": "--d-model 192 --layers 4 --heads 6 --ctx 128 --market",  # v3 + market (news channel)
    "v3": "--d-model 192 --layers 4 --heads 6 --ctx 128 --flow",  # v2 + order-flow memory
}
SHOCK_MODELS = ["v4", "v3"]   # shock tests (v2 and baselines: done on the laptop)
TRACK_MODELS = ["v4"]         # tracking test: only models that read the market (v3 cannot track
                              # it, and v4 with the market frozen is already the no-news control)
ETFS = ["QQQ", "SMH"]
RUN_SEED_CHECK = False  # True: v4 once more with another seed (+~35 min training + detective). The
                        # detective CIs cover evaluation noise only; this measures training noise, so
                        # a v4-vs-v3 gap can be told apart from seed luck. Trained last.
if RUN_SEED_CHECK:
    RUNS |= {"v4_seed1": RUNS["v4"] + " --seed 1"}
RUN_ABLATIONS = False   # True: also the full-size ablations below (~1 h each on an A100)
if RUN_ABLATIONS:
    RUNS |= {"no_regime": "--no-regime", "no_atoms": "--no-atoms", "no_book": "--no-book",
             "ctx32": "--ctx 32"}

In [ ]:
from google.colab import drive
drive.mount("/content/drive", timeout_ms=600_000)   # approve the popup: tick "Select all"
import os, subprocess, torch
for d in (f"{DRIVE}/checkpoints", f"{DRIVE}/results", f"{REPO}/src/lobgen", f"{REPO}/scripts"):
    os.makedirs(d, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to GPU")
!pip -q install sortedcontainers numba

In [ ]:
%%writefile /content/lobgen/src/lobgen/__init__.py
"""lobgen: Nasdaq ITCH -> LOBSTER conversion and synthetic order book generation."""

In [ ]:
%%writefile /content/lobgen/src/lobgen/itch.py
"""Streaming reader/filter for Nasdaq TotalView-ITCH 5.0 binary files.

Wire format: each message is a 2-byte big-endian length followed by that many
bytes. Byte 0 of the message is the type char, bytes 1-2 the stock locate code,
3-4 tracking number, 5-10 a 6-byte nanoseconds-since-midnight timestamp.

A full-day file is 4-18 GB gzipped and holds every Nasdaq symbol, so `filter_file`
makes one streaming pass and keeps only the system-event ('S') messages plus
messages whose locate code belongs to a requested ticker. The output is itself a
valid ITCH stream, small enough to parse in pure Python.
"""

from __future__ import annotations

import gzip
import struct
import sys
import time
from pathlib import Path
from typing import Iterator

import numpy as np
from numba import njit

# Message body lengths (excluding the 2-byte length prefix), ITCH 5.0 spec.
MSG_LEN = {
    b"S": 12, b"R": 39, b"H": 25, b"Y": 20, b"L": 26, b"V": 35, b"W": 12,
    b"K": 28, b"J": 35, b"h": 21, b"A": 36, b"F": 40, b"E": 31, b"C": 36,
    b"X": 23, b"D": 19, b"U": 35, b"P": 44, b"Q": 40, b"B": 19, b"I": 50,
    b"N": 20, b"O": 48,
}

_S = ord("S")
_R = ord("R")


@njit(cache=True)
def _scan(buf, n, want_names, locate_map, out):
    """Walk length-prefixed messages in buf[:n]; copy kept ones into out.

    Returns (bytes_consumed, bytes_written, messages_seen). Stops at the last
    complete message so the caller can carry the partial tail to the next chunk.
    locate_map is updated in place when a wanted ticker's 'R' message is seen.
    """
    pos = 0
    w = 0
    seen = 0
    k_names = want_names.shape[0]
    while pos + 2 <= n:
        length = (np.int64(buf[pos]) << 8) | np.int64(buf[pos + 1])
        end = pos + 2 + length
        if end > n:
            break
        t = buf[pos + 2]
        loc = (np.int64(buf[pos + 3]) << 8) | np.int64(buf[pos + 4])
        keep = False
        if t == _S:
            keep = True
        elif t == _R:
            # Stock symbol: 8 bytes at message offset 11 (+2 for prefix).
            for k in range(k_names):
                match = True
                for j in range(8):
                    if buf[pos + 13 + j] != want_names[k, j]:
                        match = False
                        break
                if match:
                    locate_map[loc] = k
                    keep = True
                    break
        elif locate_map[loc] >= 0:
            keep = True
        if keep:
            for i in range(pos, end):
                out[w] = buf[i]
                w += 1
        pos = end
        seen += 1
    return pos, w, seen


def _pad_symbol(sym: str) -> bytes:
    b = sym.encode("ascii")
    if len(b) > 8:
        raise ValueError(f"symbol too long: {sym}")
    return b.ljust(8, b" ")


def filter_file(
    raw_path: str | Path,
    tickers: list[str],
    out_path: str | Path,
    chunk_bytes: int = 1 << 27,
    allow_truncated: bool = False,
    log_every_s: float = 30.0,
) -> dict:
    """Stream a gzipped ITCH file (or '-' for gzipped stdin), keep 'S' + messages for `tickers`.

    Reading stdin lets a remote box run `curl URL | lobgen filter - ...` without storing
    the raw multi-GB day.
    """
    from_stdin = str(raw_path) == "-"
    out_path = Path(out_path)
    want = np.frombuffer(b"".join(_pad_symbol(t) for t in tickers), dtype=np.uint8)
    want = want.reshape(len(tickers), 8).copy()
    locate_map = np.full(1 << 16, -1, dtype=np.int64)

    buf = np.empty(chunk_bytes, dtype=np.uint8)
    out = np.empty(chunk_bytes, dtype=np.uint8)
    mv = memoryview(buf)
    have = 0
    total_seen = total_kept_bytes = total_raw_bytes = 0
    truncated = False
    compressed_size = None if from_stdin else Path(raw_path).stat().st_size
    t0 = last_log = time.time()
    src = gzip.GzipFile(fileobj=sys.stdin.buffer) if from_stdin else gzip.open(raw_path, "rb")

    with src as f, open(out_path, "wb") as o:
        while True:
            try:
                n = f.readinto(mv[have:])
            except EOFError:
                if not allow_truncated:
                    raise
                truncated, n = True, 0
            have += n
            total_raw_bytes += n
            consumed, w, seen = _scan(buf, have, want, locate_map, out)
            o.write(memoryview(out)[:w])
            total_seen += seen
            total_kept_bytes += w
            rem = have - consumed
            if consumed == 0 and have == chunk_bytes:
                raise ValueError(f"message longer than chunk_bytes={chunk_bytes}")
            if rem:
                buf[:rem] = buf[consumed:have].copy()
            have = rem
            now = time.time()
            if now - last_log > log_every_s:
                pct = f"{f.fileobj.tell() / compressed_size:6.1%} compressed read | " if compressed_size else ""
                print(
                    f"  {pct}{total_raw_bytes / 1e9:7.2f} GB raw | {total_seen / 1e6:8.1f} M msgs | "
                    f"{(now - t0) / 60:5.1f} min",
                    file=sys.stderr, flush=True,
                )
                last_log = now
            if n == 0:
                break

    if have and not truncated:
        raise ValueError(f"{have} trailing bytes do not form a complete message")
    found = {t: int(np.flatnonzero(locate_map == k)[0]) if (locate_map == k).any() else None
             for k, t in enumerate(tickers)}
    missing = [t for t, loc in found.items() if loc is None]
    if missing:
        raise ValueError(f"tickers not in stock directory: {missing}")
    return {
        "messages_seen": total_seen,
        "raw_bytes": total_raw_bytes,
        "kept_bytes": total_kept_bytes,
        "locates": found,
        "truncated": truncated,
        "minutes": (time.time() - t0) / 60,
    }


# ---------------------------------------------------------------------------
# Pure-Python message decoding for the (small) filtered stream.
# ---------------------------------------------------------------------------

def load_stream(path: str | Path) -> bytes:
    """Read a filtered ITCH file (.gz ok) into memory."""
    p = Path(path)
    return gzip.decompress(p.read_bytes()) if p.suffix == ".gz" else p.read_bytes()


def iter_messages(src: str | Path | bytes) -> Iterator[bytes]:
    """Yield raw message bodies (without length prefix) from a filtered stream or its path."""
    data = src if isinstance(src, (bytes, bytearray)) else load_stream(src)
    pos, n = 0, len(data)
    while pos + 2 <= n:
        (length,) = struct.unpack_from(">H", data, pos)
        end = pos + 2 + length
        if end > n:
            raise ValueError(f"truncated message at byte {pos}")
        yield data[pos + 2:end]
        pos = end


def header(msg: bytes) -> tuple[str, int, int]:
    """(type char, locate, timestamp_ns)."""
    t = chr(msg[0])
    locate = (msg[1] << 8) | msg[2]
    ts = int.from_bytes(msg[5:11], "big")
    return t, locate, ts


# Body decoders, offsets after the 11-byte common header.
_U64 = struct.Struct(">Q")
_ADD = struct.Struct(">QcI8sI")          # ref, side, shares, stock, price
_EXEC = struct.Struct(">QIQ")            # ref, shares, match
_EXEC_PX = struct.Struct(">QIQcI")       # ref, shares, match, printable, price
_CANCEL = struct.Struct(">QI")           # ref, cancelled shares
_REPLACE = struct.Struct(">QQII")        # orig ref, new ref, shares, price
_TRADE = struct.Struct(">QcI8sIQ")       # ref, side, shares, stock, price, match
_CROSS = struct.Struct(">Q8sIQc")        # shares, stock, price, match, cross type
_HALT = struct.Struct(">8scc4s")         # stock, state, reserved, reason
_DIR = struct.Struct(">8scc")            # stock, market category, FS status


def decode(msg: bytes):
    """Decode one message into (type, locate, ts, fields tuple). Unused types -> fields None."""
    t, locate, ts = header(msg)
    if t in ("A", "F"):
        f = _ADD.unpack_from(msg, 11)
    elif t == "E":
        f = _EXEC.unpack_from(msg, 11)
    elif t == "C":
        f = _EXEC_PX.unpack_from(msg, 11)
    elif t == "X":
        f = _CANCEL.unpack_from(msg, 11)
    elif t == "D":
        f = _U64.unpack_from(msg, 11)
    elif t == "U":
        f = _REPLACE.unpack_from(msg, 11)
    elif t == "P":
        f = _TRADE.unpack_from(msg, 11)
    elif t == "Q":
        f = _CROSS.unpack_from(msg, 11)
    elif t == "H":
        f = _HALT.unpack_from(msg, 11)
    elif t == "R":
        f = _DIR.unpack_from(msg, 11)
    elif t == "S":
        f = (msg[11:12],)
    else:
        f = None
    return t, locate, ts, f

In [ ]:
%%writefile /content/lobgen/src/lobgen/book.py
"""Order-level limit order book. Prices are integer dollars x 10000 (ITCH == LOBSTER units)."""

from __future__ import annotations

from sortedcontainers import SortedDict

BUY, SELL = 1, -1
EMPTY_ASK = 9_999_999_999
EMPTY_BID = -9_999_999_999


class BookError(RuntimeError):
    pass


class OrderBook:
    def __init__(self) -> None:
        self.orders: dict[int, list[int]] = {}  # ref -> [side, price, shares]
        self.bids: SortedDict = SortedDict()     # price -> total shares
        self.asks: SortedDict = SortedDict()

    def _levels(self, side: int) -> SortedDict:
        return self.bids if side == BUY else self.asks

    def add(self, ref: int, side: int, price: int, shares: int) -> None:
        if ref in self.orders:
            raise BookError(f"duplicate order ref {ref}")
        if shares <= 0:
            raise BookError(f"non-positive add size {shares} for ref {ref}")
        self.orders[ref] = [side, price, shares]
        lv = self._levels(side)
        lv[price] = lv.get(price, 0) + shares

    def reduce(self, ref: int, shares: int) -> tuple[int, int, int]:
        """Remove `shares` from an order. Returns (side, price, remaining)."""
        o = self.orders.get(ref)
        if o is None:
            raise BookError(f"unknown order ref {ref}")
        side, price, have = o
        if shares <= 0 or shares > have:
            raise BookError(f"reduce {shares} from ref {ref} holding {have}")
        remaining = have - shares
        lv = self._levels(side)
        left = lv[price] - shares
        if left:
            lv[price] = left
        else:
            del lv[price]
        if remaining:
            o[2] = remaining
        else:
            del self.orders[ref]
        return side, price, remaining

    def delete(self, ref: int) -> tuple[int, int, int]:
        """Remove an order entirely. Returns (side, price, shares removed)."""
        o = self.orders.get(ref)
        if o is None:
            raise BookError(f"unknown order ref {ref}")
        side, price, shares = o
        self.reduce(ref, shares)
        return side, price, shares

    def best_bid(self) -> int | None:
        return self.bids.peekitem(-1)[0] if self.bids else None

    def best_ask(self) -> int | None:
        return self.asks.peekitem(0)[0] if self.asks else None

    def snapshot(self, levels: int) -> list[int]:
        """LOBSTER orderbook row: askP1, askV1, bidP1, bidV1, askP2, ... (4 * levels ints)."""
        row: list[int] = []
        ak, av = self.asks.keys(), self.asks
        bk, bv = self.bids.keys(), self.bids
        na, nb = len(ak), len(bk)
        for i in range(levels):
            if i < na:
                p = ak[i]
                row += (p, av[p])
            else:
                row += (EMPTY_ASK, 0)
            if i < nb:
                p = bk[nb - 1 - i]
                row += (p, bv[p])
            else:
                row += (EMPTY_BID, 0)
        return row

In [ ]:
%%writefile /content/lobgen/src/lobgen/lobster.py
"""Convert a filtered ITCH 5.0 stream into LOBSTER-format message + orderbook files.

LOBSTER message columns: time (s after midnight), type, order id, size, price, direction.
  type 1 submit | 2 partial cancel | 3 delete | 4 visible execution | 5 hidden execution
       6 cross trade | 7 trading halt
  direction: 1 buy limit order, -1 sell limit order (an execution's direction is the
  resting order's side, so a type-4 with direction -1 is a buyer-initiated trade).
  price: integer dollars x 10000 (same as ITCH's 4 implied decimals; never converted to float).

Conversion choices (documented because they change the statistics):
  * The book is rebuilt from the first message of the file so pre-market orders exist
    at the open; only events with start <= t < end are written.
  * ITCH 'U' (replace) -> type 3 delete of the old ref, then type 1 submit of the new ref,
    same timestamp, side inherited from the old order.
  * ITCH 'C' (execution with price) -> type 4 at the resting order's LIMIT price, so every
    type-4 row reduces the book level at its own price. The actual print price differs;
    counts of such rows are in the report.
  * ITCH 'P' (hidden execution) -> type 5, order id 0. Its side flag is unreliable: Nasdaq's
    spec says it has always been 'B' since 2014-07-14. Do not use type-5 direction.
  * ITCH 'Q' (cross) -> type 6, order id 0, direction -1. Cross executions of displayed
    orders also arrive as non-printable 'E'/'C' rows (type 4), so cross volume appears twice.
  * ITCH 'H' state H (halt) -> type 7 price -1; Q (quote only) -> price 0; T (trading) -> 1.
  * Empty book levels: ask 9999999999 / bid -9999999999, size 0.
"""

from __future__ import annotations

from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

from .book import BUY, SELL, OrderBook
from .itch import decode, header, iter_messages, load_stream

SUBMIT, CANCEL, DELETE, EXEC, HIDDEN, CROSS, HALT = 1, 2, 3, 4, 5, 6, 7
NS = 1_000_000_000
_SIDE = {b"B": BUY, b"S": SELL}
_HALT_PRICE = {b"H": -1, b"P": -1, b"Q": 0, b"T": 1}


class _Rows:
    """Preallocated row buffer (a full GOOG day is ~9M rows x 40 cols; doubling would need ~8 GB)."""

    def __init__(self, n: int, width: int) -> None:
        self.a = np.empty((n, width), dtype=np.int64)
        self.n = 0

    def append(self, row) -> None:
        self.a[self.n] = row
        self.n += 1

    def array(self) -> np.ndarray:
        if self.n != len(self.a):
            raise RuntimeError(f"event count mismatch: counted {len(self.a)}, emitted {self.n}")
        return self.a


def _count_events(data: bytes, sym: bytes, start_ns: int, end_ns: int) -> int:
    """Number of LOBSTER rows convert() will emit (a replace 'U' emits two)."""
    locate, n = None, 0
    for raw in iter_messages(data):
        t, loc, ts = header(raw)
        if t == "R" and raw[11:19] == sym:
            locate = loc
        elif loc == locate and start_ns <= ts < end_ns and t in _EMITTING:
            n += 2 if t == "U" else 1
    return n


_EMITTING = frozenset("AFECXDUPQH")


@dataclass
class Report:
    ticker: str
    date: str
    locate: int | None = None
    itch_counts: Counter = field(default_factory=Counter)
    lobster_counts: Counter = field(default_factory=Counter)
    exec_with_price: int = 0
    exec_with_price_nonprintable: int = 0
    exec_with_price_differs: int = 0
    crossed_rows: int = 0
    first_crossed_s: float | None = None
    last_crossed_s: float | None = None
    open_orders_at_end: int = 0
    system_events: list = field(default_factory=list)

    def as_dict(self) -> dict:
        d = dict(self.__dict__)
        d["itch_counts"] = dict(sorted(self.itch_counts.items()))
        d["lobster_counts"] = {int(k): v for k, v in sorted(self.lobster_counts.items())}
        return d


def convert(
    filtered_path: str | Path,
    ticker: str,
    date: str,
    levels: int = 10,
    start_s: int = 34_200,
    end_s: int = 57_600,
) -> tuple[np.ndarray, np.ndarray, Report]:
    """Returns (messages[N,6] with time in ns, book[N, 4*levels], report)."""
    start_ns, end_ns = start_s * NS, end_s * NS
    sym = ticker.encode().ljust(8, b" ")
    data = load_stream(filtered_path)
    n_rows = _count_events(data, sym, start_ns, end_ns)
    book = OrderBook()
    msgs, snaps = _Rows(n_rows, 6), _Rows(n_rows, 4 * levels)
    rep = Report(ticker=ticker, date=date)

    def emit(ts: int, typ: int, ref: int, size: int, price: int, direction: int) -> None:
        if not (start_ns <= ts < end_ns):
            return
        msgs.append((ts, typ, ref, size, price, direction))
        row = book.snapshot(levels)
        snaps.append(row)
        rep.lobster_counts[typ] += 1
        ask, bid = row[0], row[2]
        if row[1] and row[3] and bid >= ask:
            rep.crossed_rows += 1
            s = ts / NS
            rep.first_crossed_s = s if rep.first_crossed_s is None else rep.first_crossed_s
            rep.last_crossed_s = s

    _replay(data, sym, book, emit, rep)
    if rep.locate is None:
        raise ValueError(f"{ticker} not found in stock directory of {filtered_path}")
    rep.open_orders_at_end = len(book.orders)
    return msgs.array(), snaps.array(), rep


def preopen_orders(filtered_path: str | Path, ticker: str, start_s: int = 34_200) -> np.ndarray:
    """Order-level book just before the first `ticker` message at or after start_s.

    Returns int64 [K, 4] rows (ref, side, price, shares) in time priority (the order rows
    entered the book; a replace re-enters at the back). With the LOBSTER messages from
    start_s onward this is enough to rebuild the full order-level book at any later row.
    """
    sym = ticker.encode().ljust(8, b" ")
    book = OrderBook()
    rep = Report(ticker=ticker, date="")
    _replay(load_stream(filtered_path), sym, book, lambda *a: None, rep, stop_ns=start_s * NS)
    if rep.locate is None:
        raise ValueError(f"{ticker} not found in stock directory of {filtered_path}")
    rows = [(ref, side, price, shares) for ref, (side, price, shares) in book.orders.items()]
    return np.array(rows, dtype=np.int64).reshape(-1, 4)


def _replay(data: bytes, sym: bytes, book: OrderBook, emit, rep: Report,
            stop_ns: int | None = None) -> None:
    """Apply every message of `sym` to `book`, calling emit(...) once per LOBSTER row.

    With stop_ns, returns just before the first message of `sym` with timestamp >= stop_ns.
    """
    locate = None
    for raw in iter_messages(data):
        t, loc, ts, f = decode(raw)
        if t == "S":
            rep.system_events.append((f[0].decode(), ts / NS))
            continue
        if t == "R":
            if f[0] == sym:
                locate = loc
                rep.locate = loc
            continue
        if loc != locate or locate is None:
            continue
        if stop_ns is not None and ts >= stop_ns:
            return
        rep.itch_counts[t] += 1

        if t in ("A", "F"):
            ref, side, shares, _stock, price = f[:5]
            d = _SIDE[side]
            book.add(ref, d, price, shares)
            emit(ts, SUBMIT, ref, shares, price, d)
        elif t == "E":
            ref, shares, _match = f
            d, price, _ = book.reduce(ref, shares)
            emit(ts, EXEC, ref, shares, price, d)
        elif t == "C":
            ref, shares, _match, printable, px = f
            d, price, _ = book.reduce(ref, shares)
            rep.exec_with_price += 1
            rep.exec_with_price_nonprintable += printable == b"N"
            rep.exec_with_price_differs += px != price
            emit(ts, EXEC, ref, shares, price, d)
        elif t == "X":
            ref, shares = f
            d, price, _ = book.reduce(ref, shares)
            emit(ts, CANCEL, ref, shares, price, d)
        elif t == "D":
            (ref,) = f
            d, price, shares = book.delete(ref)
            emit(ts, DELETE, ref, shares, price, d)
        elif t == "U":
            old, new, shares, price = f
            d, old_price, old_shares = book.delete(old)
            emit(ts, DELETE, old, old_shares, old_price, d)
            book.add(new, d, price, shares)
            emit(ts, SUBMIT, new, shares, price, d)
        elif t == "P":
            _ref, side, shares, _stock, price, _match = f
            emit(ts, HIDDEN, 0, shares, price, _SIDE[side])
        elif t == "Q":
            shares, _stock, price, _match, _ctype = f
            emit(ts, CROSS, 0, shares, price, -1)
        elif t == "H":
            _stock, state, _res, _reason = f
            emit(ts, HALT, 0, 0, _HALT_PRICE.get(state, -1), -1)


def lobster_stem(ticker: str, date: str, start_s: int, end_s: int) -> str:
    return f"{ticker}_{date}_{start_s * 1000}_{end_s * 1000}"


def format_time(ts_ns: np.ndarray) -> np.ndarray:
    """Exact 'seconds.nanoseconds' strings; no float rounding."""
    sec, ns = np.divmod(ts_ns, NS)
    return np.char.add(np.char.add(sec.astype(str), "."), np.char.zfill(ns.astype(str), 9))


def write_lobster(
    msgs: np.ndarray, book: np.ndarray, out_dir: str | Path, ticker: str, date: str,
    levels: int, start_s: int = 34_200, end_s: int = 57_600, chunk: int = 500_000,
    csv: bool = True,
) -> list[Path]:
    """Always writes a compressed .npz; LOBSTER csv pair only if csv=True (a full day is ~2.4 GB)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = lobster_stem(ticker, date, start_s, end_s)
    npath = out_dir / f"{stem}_{levels}.npz"
    np.savez_compressed(npath, messages=msgs, book=book, levels=levels)
    if not csv:
        return [npath]
    mpath = out_dir / f"{stem}_message_{levels}.csv"
    bpath = out_dir / f"{stem}_orderbook_{levels}.csv"
    write_lobster_csv(msgs, book, mpath, bpath, chunk)
    return [npath, mpath, bpath]


def write_lobster_csv(msgs: np.ndarray, book: np.ndarray, mpath: Path, bpath: Path,
                      chunk: int = 500_000) -> None:
    with open(mpath, "w") as fm, open(bpath, "w") as fb:
        for i in range(0, len(msgs), chunk):
            m = msgs[i:i + chunk]
            df = pd.DataFrame(m[:, 1:], copy=False)
            df.insert(0, "time", format_time(m[:, 0]))
            df.to_csv(fm, header=False, index=False)
            pd.DataFrame(book[i:i + chunk], copy=False).to_csv(fb, header=False, index=False)

In [ ]:
%%writefile /content/lobgen/src/lobgen/validate.py
"""Invariant checks and summary stats for converted LOBSTER data.

Without LOBSTER to diff against, correctness rests on invariants that must hold for any
faithful conversion:
  1. Each orderbook row differs from the previous one exactly by the event on that row
     (+size at its price for a submit, -size for cancel/delete/execution, no change for
     hidden executions, crosses and halts), wherever both rows show that price.
  2. Times never decrease; sizes and prices are positive for order events.
  3. The book is not crossed during continuous trading (it can be before the opening cross).
"""

from __future__ import annotations

import numpy as np

from .book import EMPTY_ASK, EMPTY_BID

_SIGN = {1: 1, 2: -1, 3: -1, 4: -1}
NS = 1_000_000_000


def _side_levels(row: np.ndarray, levels: int):
    asks = {int(row[4 * i]): int(row[4 * i + 1]) for i in range(levels) if row[4 * i + 1] > 0}
    bids = {int(row[4 * i + 2]): int(row[4 * i + 3]) for i in range(levels) if row[4 * i + 3] > 0}
    return asks, bids


def check_book_transitions(msgs: np.ndarray, book: np.ndarray, levels: int, max_examples: int = 5) -> dict:
    bad, examples = 0, []
    prev_a, prev_b = _side_levels(book[0], levels)
    for i in range(1, len(msgs)):
        cur_a, cur_b = _side_levels(book[i], levels)
        _, typ, _, size, price, d = (int(x) for x in msgs[i])
        for side, prev, cur in ((-1, prev_a, cur_a), (1, prev_b, cur_b)):
            # Prices are only comparable inside the range both rows fully show.
            full_prev, full_cur = len(prev) == levels, len(cur) == levels
            if side == -1:
                lim = min(max(prev) if full_prev else EMPTY_ASK, max(cur) if full_cur else EMPTY_ASK)
                inside = lambda p: p <= lim  # noqa: E731
            else:
                lim = max(min(prev) if full_prev else EMPTY_BID, min(cur) if full_cur else EMPTY_BID)
                inside = lambda p: p >= lim  # noqa: E731
            expected = {}
            if typ in _SIGN and d == side and inside(price):
                expected[price] = _SIGN[typ] * size
            diff = {}
            for p in set(prev) | set(cur):
                if inside(p):
                    delta = cur.get(p, 0) - prev.get(p, 0)
                    if delta:
                        diff[p] = delta
            if diff != expected:
                bad += 1
                if len(examples) < max_examples:
                    examples.append({"row": i, "side": side, "msg": msgs[i].tolist(),
                                     "expected": expected, "got": diff})
        prev_a, prev_b = cur_a, cur_b
    return {"rows_checked": len(msgs) - 1, "violations": bad, "examples": examples}


def check_messages(msgs: np.ndarray) -> dict:
    t, typ, size, price, d = msgs[:, 0], msgs[:, 1], msgs[:, 3], msgs[:, 4], msgs[:, 5]
    order_ev = np.isin(typ, [1, 2, 3, 4, 5])
    return {
        "time_decreases": int((np.diff(t) < 0).sum()),
        "nonpositive_size": int((size[order_ev] <= 0).sum()),
        "nonpositive_price": int((price[order_ev] <= 0).sum()),
        "bad_direction": int((~np.isin(d[order_ev], [1, -1])).sum()),
    }


def summary(msgs: np.ndarray, book: np.ndarray, tick: int = 100) -> dict:
    typ = msgs[:, 1]
    n = len(msgs)
    ask, askv, bid, bidv = book[:, 0], book[:, 1], book[:, 2], book[:, 3]
    two_sided = (askv > 0) & (bidv > 0)
    spread_ticks = (ask[two_sided] - bid[two_sided]) // tick
    dt = np.diff(msgs[:, 0])
    # Submissions priced exactly at the touch of their own side (point mass in the post).
    prev_ask, prev_bid = ask[:-1], bid[:-1]
    sub = typ[1:] == 1
    at_touch = np.where(msgs[1:, 5] == 1, msgs[1:, 4] == prev_bid, msgs[1:, 4] == prev_ask)
    vals, counts = np.unique(typ, return_counts=True)
    sp_vals, sp_counts = np.unique(np.clip(spread_ticks, -5, 10), return_counts=True)
    return {
        "events": n,
        "first_time_s": float(msgs[0, 0] / NS) if n else None,
        "last_time_s": float(msgs[-1, 0] / NS) if n else None,
        "event_share": {int(v): round(c / n, 4) for v, c in zip(vals, counts)},
        "trade_share_visible_plus_hidden": round(float(np.isin(typ, [4, 5]).mean()), 4),
        "dt_zero_share": round(float((dt == 0).mean()), 4) if len(dt) else None,
        "dt_median_us": float(np.median(dt) / 1e3) if len(dt) else None,
        "submit_at_touch_share": round(float(at_touch[sub].mean()), 4) if sub.any() else None,
        "spread_ticks_dist (clipped -5..10)": {int(v): round(c / len(spread_ticks), 4)
                                              for v, c in zip(sp_vals, sp_counts)},
        "crossed_or_locked_rows": int((spread_ticks <= 0).sum()),
    }

In [ ]:
%%writefile /content/lobgen/src/lobgen/market.py
"""The market as an input: how news reaches a single stock's order book.

A tariff headline, a rate decision or a sector sell-off does not arrive in GOOG's order book as
text. It arrives as a move in the whole market, and market makers in GOOG re-price off index
futures and ETFs within milliseconds. So the channel this module gives the model is the market
itself: the Nasdaq quotes of QQQ (Nasdaq-100) and SMH (semiconductors, for INTC), taken from
the same ITCH files as the stocks, turned into clock-time features at every stock event.

Per ETF and per time scale tau in CLOCK_TAUS_S (0.1, 1, 10, 100 s):
    dev_tau(t) = log mid(t) - EMA_tau(log mid)(t)        in bps / 10, clipped to +-5
a continuous-time exponential average, so dev_tau ~ the ETF's return over the last ~tau; and
per tau in VOL_TAUS_S (1, 30 s) a turbulence measure:
    vol_tau(t) = log1p( sum_k |d log mid_k| e^{-(t - t_k)/tau} in bps ) / 3
The same clock-time deviations of the stock's *own* mid (CLOCK_TAUS_S) are carried in the
feature stream (features.py), so the model can see the gap between how far the market moved and
how far the stock has followed: lead-lag catch-up is what makes a stock "go down as much as it
should" after market-wide news.

Lookups are exact and causal: features at time t use only ETF quotes at or before t. During a
rollout the generated event times query the same series, so a generated run is conditioned on
the market path: the real one (what the market did), a frozen one (no news after the start),
or a scenario (e.g. QQQ -1% over 30 s: a headline) - see Market.scenario().
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
from numba import njit

ETFS = ("QQQ", "SMH")
CLOCK_TAUS_S = (0.1, 1.0, 10.0, 100.0)
VOL_TAUS_S = (1.0, 30.0)
N_ETF_FEAT = len(CLOCK_TAUS_S) + len(VOL_TAUS_S)
N_OWN = len(CLOCK_TAUS_S)
N_MKT = N_OWN + N_ETF_FEAT * len(ETFS)
NS = 1_000_000_000


# ----------------------------------------------------------------------------- extraction

def bbo_series(filtered_path: str | Path, ticker: str, start_s: int = 32_400,
               end_s: int = 57_600) -> dict[str, np.ndarray]:
    """Best bid / ask after every change, from a filtered ITCH stream (prices x 10000).
    Starts at 09:00 so that the averages are warm at the open."""
    from .book import OrderBook
    from .itch import load_stream
    from .lobster import Report, _replay

    sym = ticker.encode().ljust(8, b" ")
    book = OrderBook()
    t_out, bid_out, ask_out = [], [], []
    last = [None, None]
    lo, hi = start_s * NS, end_s * NS

    def emit(ts, *_):
        if not lo <= ts < hi:
            return
        b, a = book.best_bid(), book.best_ask()
        if (b, a) != (last[0], last[1]):
            last[0], last[1] = b, a
            t_out.append(ts)
            bid_out.append(-1 if b is None else b)
            ask_out.append(-1 if a is None else a)

    rep = Report(ticker=ticker, date="")
    _replay(load_stream(filtered_path), sym, book, emit, rep)
    if rep.locate is None:
        raise ValueError(f"{ticker} not in {filtered_path}")
    return {"t": np.array(t_out, dtype=np.int64), "bid": np.array(bid_out, dtype=np.int64),
            "ask": np.array(ask_out, dtype=np.int64)}


def save_bbo(s: dict, path: str | Path) -> None:
    np.savez_compressed(path, t=s["t"], bid=s["bid"], ask=s["ask"])


# ----------------------------------------------------------------------------- features

@njit(cache=True)
def _precompute(t, x, taus_ns, vtaus_ns, E, R):
    """E[k, j]: EMA_tau_j of the piecewise-constant x at time t[k] (just before x[k] takes
    over). R[k, j]: decayed sum of |jumps| up to and including the jump at t[k], in bps."""
    n = t.shape[0]
    for j in range(taus_ns.shape[0]):
        E[0, j] = x[0]
    for j in range(vtaus_ns.shape[0]):
        R[0, j] = 0.0
    for k in range(1, n):
        dt = float(t[k] - t[k - 1])
        for j in range(taus_ns.shape[0]):
            a = np.exp(-dt / taus_ns[j])
            E[k, j] = a * E[k - 1, j] + (1.0 - a) * x[k - 1]
        jump = abs(x[k] - x[k - 1]) * 1e4
        for j in range(vtaus_ns.shape[0]):
            R[k, j] = R[k - 1, j] * np.exp(-dt / vtaus_ns[j]) + jump


class Series:
    """One ETF's log mid as a step function of time, with what the feature lookups need.

    Scenarios do not copy the (multi-million point) arrays: frozen() stops the path at t0 (the
    averages keep decaying towards the last value, exactly as for a quote that never changes
    again) and shifted() adds a small overlay path. Averages are linear, so the overlay's
    deviations add; jumps at different times add in the turbulence sum."""

    def __init__(self, t: np.ndarray, logmid: np.ndarray) -> None:
        order = np.argsort(t, kind="stable")
        self.t = np.asarray(t, dtype=np.int64)[order]
        self.x = np.asarray(logmid, dtype=np.float64)[order]
        if len(self.t) == 0:
            raise ValueError("empty series")
        self.taus = np.array(CLOCK_TAUS_S) * NS
        self.vtaus = np.array(VOL_TAUS_S) * NS
        self.E = np.empty((len(self.t), len(self.taus)))
        self.R = np.empty((len(self.t), len(self.vtaus)))
        _precompute(self.t, self.x, self.taus, self.vtaus, self.E, self.R)
        self.t_end: int | None = None          # frozen after this time
        self.overlays: list["Series"] = []

    @classmethod
    def from_bbo(cls, s: dict) -> "Series":
        ok = (s["bid"] > 0) & (s["ask"] > 0) & (s["ask"] > s["bid"])
        mid = np.where(ok, (s["bid"] + s["ask"]) / 2.0, np.nan)
        # carry the last good quote over one-sided or crossed moments
        idx = np.where(ok, np.arange(len(mid)), -1)
        np.maximum.accumulate(idx, out=idx)
        keep = idx >= 0
        t, m = s["t"][keep], mid[idx[keep]]
        # one quote per timestamp, the last: a replace ('U') is a delete then an add at the same
        # ns, and the book between the two is not a quote anyone saw
        last = np.append(t[1:] != t[:-1], True)
        t, m = t[last], m[last]
        change = np.concatenate([[True], np.diff(m) != 0])
        return cls(t[change], np.log(m[change]))

    def _copy(self) -> "Series":
        c = object.__new__(Series)
        c.__dict__.update(self.__dict__)
        c.overlays = list(self.overlays)
        return c

    def _index(self, q: np.ndarray) -> np.ndarray:
        qc = q if self.t_end is None else np.minimum(q, self.t_end)
        return np.searchsorted(self.t, qc, side="right") - 1

    def value(self, q) -> np.ndarray:
        q = np.asarray(q, dtype=np.int64)
        k = self._index(q)
        v = np.where(k >= 0, self.x[np.maximum(k, 0)], self.x[0])
        for o in self.overlays:
            ko = o._index(q)
            v = v + np.where(ko >= 0, o.x[np.maximum(ko, 0)], 0.0)
        return v

    def _raw(self, q: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """(log mid - its averages [n, taus], turbulence sum in bps [n, vol taus]); 0 before
        the first point."""
        k = self._index(q)
        before = k < 0
        k = np.maximum(k, 0)
        dt = np.maximum(q - self.t[k], 0).astype(np.float64)
        a = np.exp(-dt[:, None] / self.taus[None, :])
        x = self.x[k][:, None]
        dev = x - (a * self.E[k] + (1 - a) * x)
        R = self.R[k] * np.exp(-dt[:, None] / self.vtaus[None, :])
        dev[before] = 0.0
        R[before] = 0.0
        return dev, R

    def features(self, q: np.ndarray) -> np.ndarray:
        """[len(q), N_ETF_FEAT] at times q (ns since midnight)."""
        q = np.asarray(q, dtype=np.int64)
        dev, R = self._raw(q)
        for o in self.overlays:
            d2, r2 = o._raw(q)
            dev, R = dev + d2, R + r2
        return np.hstack([np.clip(dev * 1e4 / 10, -5, 5), np.log1p(R) / 3])

    def shifted(self, t0: int, bps: float, ramp_s: float, step_ns: int = 10_000_000) -> "Series":
        """This path with an extra move of `bps` that builds up linearly (in 10 ms steps) over
        ramp_s seconds from t0 and then stays: a news shock on top of what the market did."""
        n = max(int(ramp_s * NS) // step_ns, 1)
        j = np.arange(n + 1)
        ov = Series(t0 + j * step_ns, bps / 1e4 * j / n)
        ov.t_end = self.t_end
        c = self._copy()
        c.overlays.append(ov)
        return c

    def frozen(self, t0: int) -> "Series":
        """The path up to t0, then flat: a market with no news after t0."""
        c = self._copy()
        c.t_end = t0 if self.t_end is None else min(t0, self.t_end)
        for i, o in enumerate(c.overlays):
            c.overlays[i] = o._copy()
            c.overlays[i].t_end = c.t_end
        return c


class Market:
    """The ETF series of one day, in ETFS order; features() gives the ETF part of the input."""

    def __init__(self, series: list[Series], date: str = "") -> None:
        if len(series) != len(ETFS):
            raise ValueError(f"need one series per ETF {ETFS}")
        self.series, self.date = series, date

    @classmethod
    def load(cls, date: str, root: str | Path = "data/market") -> "Market":
        root = Path(root)
        out = []
        for e in ETFS:
            p = root / f"{e}_{date}.npz"
            if not p.exists():
                raise FileNotFoundError(f"{p} missing: run notebooks/fetch_itch_colab.ipynb "
                                        "(market part) and download MyDrive/lobgen/market/")
            z = np.load(p)
            out.append(Series.from_bbo({k: z[k] for k in ("t", "bid", "ask")}))
        return cls(out, date)

    def features(self, q: np.ndarray) -> np.ndarray:
        return np.hstack([s.features(q) for s in self.series])

    def scenario(self, t0: int, bps: float, ramp_s: float = 30.0, etfs=ETFS) -> "Market":
        """News scenario: the chosen ETFs move an extra `bps` over ramp_s seconds from t0."""
        return Market([s.shifted(t0, bps, ramp_s) if e in etfs else s
                       for e, s in zip(ETFS, self.series)], self.date)

    def frozen(self, t0: int) -> "Market":
        return Market([s.frozen(t0) for s in self.series], self.date)

    def ret_bps(self, t0: int, t1: int, etf: str = "QQQ") -> float:
        s = self.series[ETFS.index(etf)]
        return float((s.value(np.array([t1]))[0] - s.value(np.array([t0]))[0]) * 1e4)


# ----------------------------------------------------------------------------- own clock EMA

@njit(cache=True)
def own_scan(t, mid, taus_ns, E, state, out):
    """Clock-time deviations of the stock's own log mid, in place.

    state = [last log mid (nan before the first), last time]; E [n_tau]. mid: forward-filled
    mid after each event, ticks (nan until the first). out[i] after event i (bps / 10)."""
    n = t.shape[0]
    T = taus_ns.shape[0]
    for i in range(n):
        x_prev = state[0]
        if not np.isnan(x_prev):
            dt = float(t[i]) - state[1]
            if dt > 0:
                for j in range(T):
                    a = np.exp(-dt / taus_ns[j])
                    E[j] = a * E[j] + (1.0 - a) * x_prev
        if not np.isnan(mid[i]):
            x = np.log(mid[i])
            if np.isnan(x_prev):
                for j in range(T):
                    E[j] = x
            state[0] = x
        state[1] = float(t[i])
        for j in range(T):
            d = 0.0
            if not np.isnan(state[0]):
                d = (state[0] - E[j]) * 1e4 / 10
            out[i, j] = min(max(d, -5.0), 5.0)


OWN_TAUS_NS = np.array(CLOCK_TAUS_S) * NS

In [ ]:
%%writefile /content/lobgen/src/lobgen/sim.py
"""Order-book simulator: the "game engine" that turns generator actions into a valid LOBSTER stream.

A generator never writes LOBSTER rows itself. It emits *actions* expressed relative to the
current book; the simulator resolves each action against an order-level book (one FIFO queue
per price) and returns the LOBSTER message row and orderbook snapshot it produces. Whatever a
generator does, its output is a proper book: never crossed, no cancel of an order that does not
exist, level volumes that add up, one message per row.

Action = 6 int64 fields (columns of an action array, see the DT..SIZE constants):
  DT     ns since the previous event (>= 0).
  TYPE   LOBSTER type: 1 submit | 2 partial cancel | 3 delete | 4 visible execution |
         5 hidden execution. 6 cross / 7 halt pass through unchanged and are never generated.
  SIDE   1 buy | -1 sell. For 2-4 it is the resting order's side, as in LOBSTER.
  PRICE  1:   ticks from the own-side best price, + deeper (more passive), - inside the spread.
         2-4: index of an *existing* price level on the own side, 0 = best. Referring to levels
              (not tick distances) means a cancel can never point at an empty price.
         2-3: a negative value -(r+1) is instead a *pointer*: the r-th newest live order on the
              own side (r = 0 the newest), wherever it rests. encode() uses it whenever the
              target is among the PTR_RANKS newest orders. Real deletes are aimed this way (GOOG
              pulls a ladder of its own 100-share orders a few events after placing them: 44%
              of deletes hit one of the 5 newest orders), which a (level, queue position) action
              can only express indirectly through a moving book.
         5:   half-ticks from the own-side best (midpoint executions sit on half ticks).
         6/7: raw LOBSTER price (pass-through).
  QPOS   2-3: target order's place in its level's queue counted from the BACK (0 = newest);
              0 for pointer actions;
         4:   counted from the FRONT (0 = oldest), i.e. price-time priority. With these
              conventions 0 is the mode for every type: on 2025-12-12 52-90% of cancels and
              deletes hit the newest order and 93% (INTC) / 99% (GOOG) of executions the oldest.
  SIZE   shares. A delete (3) ignores it and removes the whole order. For an execution (4),
         0 means "the whole resting order" (with whole_fill, encode() writes 0 whenever the
         execution consumes the order: 78-79% of executions on 2025-12-12). A generator then
         does not need to know the size of an order it cannot see to produce a full fill.

Own-side best when that side is empty: one tick inside the opposite best; if both sides are
empty, the last own-side reference price used.

Invalid actions are repaired (and counted in Simulator.repairs) rather than rejected:
a submit that would cross is moved to one tick inside the opposite best, an out-of-range level
or queue index (or pointer rank) is clipped, a size larger than the target order is clipped. The one action that
cannot be repaired (2-4 on an empty side) returns None. strict=True raises instead; encode()
runs strict on real data, so real data never needs a repair.

encode() maps real LOBSTER rows to actions using the same rules, and decoding those actions
reproduces the rows exactly (tests + scripts/check_sim_replay.py over every converted day).
"""

from __future__ import annotations

from collections import Counter
from itertools import islice

import numpy as np
from sortedcontainers import SortedDict, SortedList

from .book import BUY, EMPTY_ASK, EMPTY_BID, SELL, BookError

TICK = 100            # $0.01 in LOBSTER price units ($ x 10000)
HALF_TICK = TICK // 2
DT, TYPE, SIDE, PRICE, QPOS, SIZE = range(6)
N_FIELDS = 6
SUBMIT, CANCEL, DELETE, EXEC, HIDDEN, CROSS, HALT = 1, 2, 3, 4, 5, 6, 7
MODELED_TYPES = (SUBMIT, CANCEL, DELETE, EXEC, HIDDEN)
GEN_REF0 = 1 << 40    # refs for generated orders; ITCH refs on these days are < 2**31
PTR_RANKS = 16        # cancels / deletes of the 16 newest orders on a side use a pointer


class SimError(RuntimeError):
    pass


class _Level:
    __slots__ = ("total", "q")

    def __init__(self) -> None:
        self.total = 0
        self.q: dict[int, int] = {}  # ref -> shares; insertion order = time priority


class SimBook:
    """Order-level book with a FIFO queue per price level, plus each side's orders by age."""

    def __init__(self) -> None:
        self.orders: dict[int, tuple[int, int]] = {}  # ref -> (side, price)
        self.side: dict[int, SortedDict] = {BUY: SortedDict(), SELL: SortedDict()}
        self.seq: dict[int, int] = {}                 # ref -> arrival number
        self.age: dict[int, SortedList] = {BUY: SortedList(), SELL: SortedList()}  # live seqs
        self._by_seq: dict[int, int] = {}             # seq -> ref
        self._next_seq = 0

    @classmethod
    def from_orders(cls, rows: np.ndarray) -> "SimBook":
        """rows: [K, 4] (ref, side, price, shares) in arrival order (oldest first), which is
        also queue order within each price."""
        b = cls()
        for ref, side, price, shares in np.asarray(rows).tolist():
            b.add(ref, side, price, shares)
        return b

    def to_orders(self) -> np.ndarray:
        """Live orders in arrival order: from_orders(to_orders()) rebuilds the same queues and
        the same pointer ranks."""
        rows = [(ref, s, p, sh) for s in (BUY, SELL) for p, lv in self.side[s].items()
                for ref, sh in lv.q.items()]
        rows.sort(key=lambda r: self.seq[r[0]])
        return np.array(rows, dtype=np.int64).reshape(-1, 4)

    def add(self, ref: int, side: int, price: int, shares: int) -> None:
        if ref in self.orders:
            raise BookError(f"duplicate order ref {ref}")
        if shares <= 0:
            raise BookError(f"non-positive add size {shares} for ref {ref}")
        levels = self.side[side]
        lv = levels.get(price)
        if lv is None:
            lv = levels[price] = _Level()
        lv.q[ref] = shares
        lv.total += shares
        self.orders[ref] = (side, price)
        s = self._next_seq
        self._next_seq += 1
        self.seq[ref] = s
        self._by_seq[s] = ref
        self.age[side].add(s)

    def reduce(self, ref: int, shares: int) -> int:
        """Remove `shares` from an order; returns what is left of it."""
        try:
            side, price = self.orders[ref]
        except KeyError:
            raise BookError(f"unknown order ref {ref}") from None
        lv = self.side[side][price]
        have = lv.q[ref]
        if not 0 < shares <= have:
            raise BookError(f"reduce {shares} from ref {ref} holding {have}")
        lv.total -= shares
        if shares < have:
            lv.q[ref] = have - shares
            return have - shares
        del lv.q[ref]
        del self.orders[ref]
        if not lv.q:
            del self.side[side][price]
        s = self.seq.pop(ref)
        del self._by_seq[s]
        self.age[side].remove(s)
        return 0

    def apply_message(self, typ: int, ref: int, size: int, price: int, d: int) -> None:
        """Apply a real LOBSTER message by order ref (fast replay, no encoding)."""
        if typ == SUBMIT:
            self.add(ref, d, price, size)
        elif typ in (CANCEL, EXEC):
            self.reduce(ref, size)
        elif typ == DELETE:
            self.reduce(ref, self.shares(ref))

    def shares(self, ref: int) -> int:
        side, price = self.orders[ref]
        return self.side[side][price].q[ref]

    def best(self, side: int) -> int | None:
        levels = self.side[side]
        if not levels:
            return None
        return levels.peekitem(-1)[0] if side == BUY else levels.peekitem(0)[0]

    def n_levels(self, side: int) -> int:
        return len(self.side[side])

    def level_price(self, side: int, k: int) -> int:
        levels = self.side[side]
        return levels.peekitem(-1 - k)[0] if side == BUY else levels.peekitem(k)[0]

    def level_index(self, side: int, price: int) -> int:
        levels = self.side[side]
        i = levels.index(price)
        return len(levels) - 1 - i if side == BUY else i

    def queue_pos(self, ref: int, from_front: bool = False) -> int:
        """Place of `ref` in its level's queue, from the back (0 = newest) or the front."""
        side, price = self.orders[ref]
        q = self.side[side][price].q
        for k, r in enumerate(iter(q) if from_front else reversed(q)):
            if r == ref:
                return k
        raise BookError(f"ref {ref} missing from its queue")

    def n_orders(self, side: int) -> int:
        return len(self.age[side])

    def recency_rank(self, ref: int) -> int:
        """How many live orders on the same side arrived after `ref` (0 = the newest)."""
        side = self.orders[ref][0]
        ages = self.age[side]
        return len(ages) - 1 - ages.index(self.seq[ref])

    def ref_by_rank(self, side: int, r: int) -> int:
        return self._by_seq[self.age[side][-1 - r]]

    def ref_at(self, side: int, price: int, qpos: int, from_front: bool = False) -> int:
        q = self.side[side][price].q
        return next(islice(iter(q) if from_front else reversed(q), qpos, None))

    def queue_len(self, side: int, price: int) -> int:
        return len(self.side[side][price].q)

    def snapshot(self, levels: int) -> list[int]:
        """LOBSTER orderbook row: askP1, askV1, bidP1, bidV1, askP2, ... (4 * levels ints)."""
        asks, bids = self.side[SELL], self.side[BUY]
        na, nb = len(asks), len(bids)
        row: list[int] = []
        for i in range(levels):
            if i < na:
                p, lv = asks.peekitem(i)
                row += (p, lv.total)
            else:
                row += (EMPTY_ASK, 0)
            if i < nb:
                p, lv = bids.peekitem(nb - 1 - i)
                row += (p, lv.total)
            else:
                row += (EMPTY_BID, 0)
        return row


class Simulator:
    """Applies actions to a SimBook and emits LOBSTER rows."""

    def __init__(self, book: SimBook, t_ns: int, levels: int = 10, next_ref: int = GEN_REF0,
                 strict: bool = False) -> None:
        self.book = book
        self.t = int(t_ns)
        self.levels = levels
        self.next_ref = next_ref
        self.strict = strict
        self.repairs: Counter = Counter()
        self.skipped = 0
        self.last_action: tuple | None = None  # the action as applied, after repairs
        self._anchor = {BUY: book.best(BUY), SELL: book.best(SELL)}

    def _repair(self, why: str) -> None:
        if self.strict:
            raise SimError(why)
        self.repairs[why] += 1

    def ref_price(self, side: int) -> int:
        """Own-side best, else one tick inside the opposite best, else the last value returned."""
        b = self.book.best(side)
        if b is None:
            o = self.book.best(-side)
            b = o - side * TICK if o is not None else self._anchor[side]
            if b is None:
                raise SimError("both sides empty and no anchor price")
        self._anchor[side] = b
        return b

    def step(self, dt: int, typ: int, side: int, p: int, qpos: int, size: int,
             ref: int | None = None, snapshot: bool = True):
        """Apply one action. Returns (message tuple, snapshot list | None), or None if the
        action targets an empty side. `ref` forces the order id of a submit (replay)."""
        bk = self.book
        if dt < 0:
            self._repair("dt")
            dt = 0
        if side not in (BUY, SELL):
            self._repair("side")
            side = BUY if side > 0 else SELL
        t = self.t + dt

        if typ == SUBMIT:
            if size <= 0:
                self._repair("size")
                size = 1
            ref_px = self.ref_price(side)
            price = ref_px - side * p * TICK
            opp = bk.best(-side)
            if opp is not None and side * (price - opp) >= 0:
                self._repair("cross")
                price = opp - side * TICK
            if price <= 0:
                self._repair("price")
                price = TICK
            p = side * (ref_px - price) // TICK
            if ref is None:
                ref = self.next_ref
                self.next_ref += 1
            bk.add(ref, side, price, size)
            msg = (t, SUBMIT, ref, size, price, side)

        elif typ in (CANCEL, DELETE, EXEC):
            n = bk.n_levels(side)
            if n == 0:
                self._repair("empty_side")
                self.skipped += 1
                return None
            if p < 0 and typ != EXEC:                      # pointer: r-th newest order
                r, n_live = -p - 1, bk.n_orders(side)
                if r >= n_live:
                    self._repair("rank")
                    r = n_live - 1
                    p = -r - 1
                qpos = 0
                target = bk.ref_by_rank(side, r)
                price = bk.orders[target][1]
            else:
                if not 0 <= p < n:
                    self._repair("level")
                    p = min(max(p, 0), n - 1)
                price = bk.level_price(side, p)
                qlen = bk.queue_len(side, price)
                if not 0 <= qpos < qlen:
                    self._repair("qpos")
                    qpos = min(max(qpos, 0), qlen - 1)
                target = bk.ref_at(side, price, qpos, from_front=typ == EXEC)
            have = bk.side[side][price].q[target]
            if typ == DELETE or (typ == EXEC and size == 0):   # 0 = whole order
                size = have
            elif not 0 < size <= have:
                self._repair("size")
                size = min(max(size, 1), have)
            whole = typ == EXEC and size == have
            bk.reduce(target, size)
            msg = (t, typ, target, size, price, side)

        elif typ == HIDDEN:
            if size <= 0:
                self._repair("size")
                size = 1
            ref_px = self.ref_price(side)
            price = ref_px - side * p * HALF_TICK
            if price <= 0:
                self._repair("price")
                price = HALF_TICK
                p = side * (ref_px - price) // HALF_TICK
            msg = (t, HIDDEN, 0, size, price, side)

        elif typ in (CROSS, HALT):
            msg = (t, typ, 0, size, p, side)

        else:
            self._repair("type")
            self.skipped += 1
            return None

        self.t = t
        a_size = 0 if typ == EXEC and whole else size
        self.last_action = (dt, typ, side, p, qpos if typ in (CANCEL, DELETE, EXEC) else 0, a_size)
        return msg, (bk.snapshot(self.levels) if snapshot else None)

    def run(self, actions: np.ndarray, snapshot: bool = True):
        """Apply an [N, 6] action array, skipping inapplicable ones.

        Returns (messages [M, 6], book [M, 4*levels] or None, applied actions [M, 6]), M <= N.
        """
        msgs, books, applied = [], [], []
        for a in np.asarray(actions).tolist():
            out = self.step(*a, snapshot=snapshot)
            if out is not None:
                msgs.append(out[0])
                applied.append(self.last_action)
                if snapshot:
                    books.append(out[1])
        m = np.array(msgs, dtype=np.int64).reshape(-1, 6)
        b = np.array(books, dtype=np.int64).reshape(-1, 4 * self.levels) if snapshot else None
        return m, b, np.array(applied, dtype=np.int64).reshape(-1, N_FIELDS)


def encode(msgs: np.ndarray, init_orders: np.ndarray, t0: int | None = None,
           levels: int = 10, pointer: int = PTR_RANKS,
           whole_fill: bool = True) -> tuple[np.ndarray, np.ndarray]:
    """Real LOBSTER messages -> actions, replaying them through a strict Simulator.

    init_orders is the order-level book before msgs[0] (lobster.preopen_orders, or
    states_at for a later start), oldest order first; t0 the time of the event before msgs[0]
    (default msgs[0]). A cancel / delete whose target is among the `pointer` newest orders on
    its side is encoded as a pointer (pointer=0: always level + queue position). whole_fill:
    an execution that consumes the whole order gets size 0. (pointer=0, whole_fill=False) is
    the v1 encoding.
    Returns (actions [N, 6] int64, inexact [N] bool). A row is inexact when its price is off
    the tick grid the action can express (1-6 sub-dollar stub quotes per day, e.g. a $0.0001
    bid, which may be legally sub-penny) or it touches an order whose submit was inexact; decoding reproduces every other
    row exactly, which this function checks as it goes.
    """
    m = np.asarray(msgs)
    sim = Simulator(SimBook.from_orders(init_orders), int(m[0, 0]) if t0 is None else t0,
                    levels, strict=True)
    bk = sim.book
    actions = np.empty((len(m), N_FIELDS), dtype=np.int64)
    inexact = np.zeros(len(m), dtype=bool)
    off_grid_refs: set[int] = set()
    for i, (t, typ, ref, size, price, d) in enumerate(m.tolist()):
        qpos = 0
        if typ in (SUBMIT, HIDDEN):
            unit = TICK if typ == SUBMIT else HALF_TICK
            num = d * (sim.ref_price(d) - price)
            p, r = divmod(num, unit)
            if r:
                p = int(round(num / unit))
                inexact[i] = True
                if typ == SUBMIT:
                    off_grid_refs.add(ref)
        elif typ in (CANCEL, DELETE, EXEC):
            side_b, price_b = bk.orders[ref]
            if side_b != d:
                raise SimError(f"row {i}: side {d} but ref {ref} rests on side {side_b}")
            r = bk.recency_rank(ref) if typ != EXEC and pointer else pointer
            if r < pointer:
                p = -r - 1
            else:
                p = bk.level_index(side_b, price_b)
                qpos = bk.queue_pos(ref, from_front=typ == EXEC)
            inexact[i] = ref in off_grid_refs
        else:
            p = price
        a_size = 0 if typ == EXEC and whole_fill and size == bk.shares(ref) else size
        a = (t - sim.t, typ, d, p, qpos, a_size)
        actions[i] = a
        sim.strict = not inexact[i]  # a rounded stub price (e.g. $0.0001 -> $0) may need a repair
        out = sim.step(*a, ref=ref if typ == SUBMIT else None, snapshot=False)
        if not inexact[i] and out[0] != (t, typ, ref, size, price, d):
            raise SimError(f"row {i}: decode gave {out[0]}, real {(t, typ, ref, size, price, d)}")
    return actions, inexact


def states_at(msgs: np.ndarray, init_orders: np.ndarray, starts) -> list[np.ndarray]:
    """Order-level book just before each row index in `starts` (ascending), as [K, 4] arrays."""
    starts = list(starts)
    if starts != sorted(starts):
        raise ValueError("starts must be ascending")
    bk = SimBook.from_orders(init_orders)
    out, j = [], 0
    m = np.asarray(msgs)
    for s in starts:
        for _t, typ, ref, size, price, d in m[j:s].tolist():
            bk.apply_message(typ, ref, size, price, d)
        j = s
        out.append(bk.to_orders())
    return out


def verify_decode(msgs: np.ndarray, book: np.ndarray, init_orders: np.ndarray,
                  actions: np.ndarray, inexact: np.ndarray, levels: int = 10) -> dict:
    """Decode `actions` with fresh order ids and compare every row with the real data.

    Messages are compared on all columns but the order id (generated ids are new); ids are
    checked to map one-to-one onto real ids. Book snapshots are compared on every row.
    """
    m = np.asarray(msgs)
    sim = Simulator(SimBook.from_orders(init_orders), int(m[0, 0] - actions[0, DT]), levels,
                    strict=True)
    real_to_gen: dict[int, int] = {}
    gen_to_real: dict[int, int] = {}
    bad_msg = bad_ref = bad_book = 0
    first_bad = None
    for i, a in enumerate(np.asarray(actions).tolist()):
        sim.strict = not inexact[i]
        msg, snap = sim.step(*a)
        t, typ, ref, size, price, d = m[i].tolist()
        if not inexact[i]:
            if (msg[0], msg[1], msg[3], msg[4], msg[5]) != (t, typ, size, price, d):
                bad_msg += 1
            if ref and (real_to_gen.setdefault(ref, msg[2]) != msg[2]
                        or gen_to_real.setdefault(msg[2], ref) != ref):
                bad_ref += 1
        if snap != book[i].tolist():
            bad_book += 1
        if first_bad is None and (bad_msg or bad_ref or bad_book):
            first_bad = i
    return {"rows": len(m), "inexact_rows": int(np.sum(inexact)), "message_mismatches": bad_msg,
            "id_mismatches": bad_ref, "book_mismatches": bad_book, "first_bad_row": first_bad,
            "repairs": dict(sim.repairs)}

In [ ]:
%%writefile /content/lobgen/src/lobgen/features.py
"""Model input features for each event (numpy + numba only, so windows can carry state).

Token for event i = the action as applied (type/side, price field, queue position) + what the
event did (size, gap) + the book right after it + time of day + a *day-so-far* summary.

The day-so-far summary is the reason this module keeps state. GOOG prints ~400 events a second,
so a 256-event context is ~0.6 s of history: far too little to tell what kind of day it is
(on Fri 2025-12-12 79% of GOOG submits were exactly 100 shares, on Mon 34%). Each token
therefore also carries exponentially weighted means, over the day so far, of 14 per-event
quantities (event mix, round lots, sizes, gaps, pointer cancels, spread, mid moves) at two
half-lives: 2,000 events (GOOG ~5 s, INTC ~20 s) and 100,000 events (GOOG ~4 min, INTC ~18 min).
They are causal and are carried through generated events exactly as through real ones.

Order-flow memory (v3, ModelConfig.flow). A 128-event context holds only ~3 visible trades, but
real trade signs stay correlated for hundreds of trades (large orders are split into many small
ones). So each token also carries, in *trade time*, exponentially weighted means of the aggressor
sign and of the signed log size of visible trades at half-lives of 8, 64 and 512 trades; the log
number of events since the last trade; and how far the mid is from its own moving average at
half-lives of 256 and 4,096 events (in ticks / 10, clipped to +-3), so that price moves can
revert as they do after real trades.

Market inputs (v4, ModelConfig.market; see market.py). The last N_MKT columns: clock-time
deviations of the stock's own log mid from its moving averages at 0.1 / 1 / 10 / 100 s, then the
same for the QQQ and SMH quotes plus their turbulence. The ETF columns need a market.Market for
the day (token_features(..., market=)); without one they are 0, which only models trained
without them may see.

FeatState holds what a stream needs to continue: the last valid mid and the running sums.
"""

from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
from numba import njit

from .market import N_MKT, N_OWN, OWN_TAUS_NS, own_scan
from .sim import CANCEL, DELETE, DT, EXEC, MODELED_TYPES, PRICE, QPOS, SIDE, SUBMIT, TICK, TYPE

N_TS = 10                    # (type 1..5) x (side)
P_LO, P_HI = 20, 60          # input embedding range of the price field
Q_MAX, SPREAD_MAX, SIZE_BUCKETS = 31, 31, 42
N_CAT = 4
N_BASE = 47                  # per-event continuous features
HALF_LIVES = (2_000, 100_000)
N_REG = 14                   # per-event quantities averaged over the day so far
N_CONT = N_BASE + N_REG * len(HALF_LIVES)  # v2 inputs (regime)
REGIME_COLS = list(range(N_BASE, N_CONT))
TRADE_HALF_LIVES = (8, 64, 512)             # in visible trades
MID_HALF_LIVES = (256, 4_096)               # in events
N_FLOW = 2 * len(TRADE_HALF_LIVES) + 1 + len(MID_HALF_LIVES)
N_ALL = N_CONT + N_FLOW                     # v3 inputs (regime + order-flow memory)
FLOW_COLS = list(range(N_CONT, N_ALL))
FLOW_BOOK = list(range(N_ALL - len(MID_HALF_LIVES), N_ALL))  # mid vs its average: book-derived
N_FULL = N_ALL + N_MKT                      # v4 inputs (+ own clock-time moves + market)
OWN_COLS = list(range(N_ALL, N_ALL + N_OWN))  # the stock's own clock-time moves: book-derived
ETF_COLS = list(range(N_ALL + N_OWN, N_FULL))
BOOK_COLS = list(range(5, N_BASE))          # per-event columns that describe the book
REGIME_BOOK = (12, 13)                      # regime quantities derived from the book
_DECAY = np.array([0.5 ** (1.0 / h) for h in HALF_LIVES])
_TR_DECAY = np.array([0.5 ** (1.0 / h) for h in TRADE_HALF_LIVES])
_MID_DECAY = np.array([0.5 ** (1.0 / h) for h in MID_HALF_LIVES])


def ts_index(typ: np.ndarray, side: np.ndarray) -> np.ndarray:
    return (np.clip(typ, 1, 5) - 1) * 2 + (side > 0)


def ts_decode(ts: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    return ts // 2 + 1, np.where(ts % 2 == 1, 1, -1)


def size_bucket(size):
    """Half-octave buckets of the size field, conditioning later heads on the chosen size.
    0 (an execution of the whole order) has its own bucket."""
    try:
        import torch
        if isinstance(size, torch.Tensor):
            b = torch.round(2 * torch.log2(size.clamp(min=1).float())).long() + 1
            return torch.where(size <= 0, 0, b.clamp(1, SIZE_BUCKETS - 1))
    except ImportError:
        pass
    s = np.asarray(size)
    b = np.clip(np.round(2 * np.log2(np.maximum(s, 1))).astype(np.int64) + 1, 1, SIZE_BUCKETS - 1)
    return np.where(s <= 0, 0, b)


@dataclass
class FeatState:
    """Everything a feature stream carries from one event to the next."""
    prev_mid: float = np.nan     # last valid mid, ticks
    num: np.ndarray = field(default_factory=lambda: np.zeros((len(HALF_LIVES), N_REG)))
    den: np.ndarray = field(default_factory=lambda: np.zeros(len(HALF_LIVES)))
    # order-flow memory: [sign, signed log size] sums per trade half-life, events since the
    # last trade, and the mid's moving-average sums per mid half-life
    tr_num: np.ndarray = field(default_factory=lambda: np.zeros((len(TRADE_HALF_LIVES), 2)))
    tr_den: np.ndarray = field(default_factory=lambda: np.zeros(len(TRADE_HALF_LIVES)))
    since: np.ndarray = field(default_factory=lambda: np.full(1, 1e4))
    m_num: np.ndarray = field(default_factory=lambda: np.zeros(len(MID_HALF_LIVES)))
    m_den: np.ndarray = field(default_factory=lambda: np.zeros(len(MID_HALF_LIVES)))
    # own clock-time averages of the log mid, and [last log mid, last event time ns]
    c_E: np.ndarray = field(default_factory=lambda: np.zeros(N_OWN))
    c_state: np.ndarray = field(default_factory=lambda: np.array([np.nan, 0.0]))

    def copy(self) -> "FeatState":
        return FeatState(self.prev_mid, self.num.copy(), self.den.copy(), self.tr_num.copy(),
                         self.tr_den.copy(), self.since.copy(), self.m_num.copy(),
                         self.m_den.copy(), self.c_E.copy(), self.c_state.copy())


def _mids(book: np.ndarray) -> np.ndarray:
    b = np.asarray(book)
    two = (b[:, 1] > 0) & (b[:, 3] > 0)
    return np.where(two, (b[:, 0] + b[:, 2]) / (2 * TICK), np.nan)


def _mid_moves(mid: np.ndarray, prev_mid: float) -> tuple[np.ndarray, np.ndarray, float]:
    """Mid change per event (0 when either side is empty; forward-filled), the forward-filled
    mid after each event (nan until the first valid one), and the last mid."""
    full = np.concatenate([[prev_mid], mid])
    idx = np.where(~np.isnan(full), np.arange(len(full)), 0)
    np.maximum.accumulate(idx, out=idx)
    filled = full[idx]
    return np.nan_to_num(np.diff(filled), nan=0.0), filled[1:], float(filled[-1])


def _base(actions, msgs, book, dmid) -> tuple[np.ndarray, np.ndarray]:
    a, m, b = np.asarray(actions), np.asarray(msgs), np.asarray(book)
    n = len(a)
    typ, side, p, q, dt = a[:, TYPE], a[:, SIDE], a[:, PRICE], a[:, QPOS], a[:, DT]
    size, t = m[:, 3], m[:, 0]
    ask, askv, bid, bidv = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    two = (askv > 0) & (bidv > 0)
    cat = np.empty((n, N_CAT), dtype=np.int64)
    cat[:, 0] = ts_index(typ, side)
    cat[:, 1] = np.clip(p, -P_LO, P_HI) + P_LO
    cat[:, 2] = np.clip(q, 0, Q_MAX)
    cat[:, 3] = np.clip(np.where(two, (ask - bid) // TICK, 0), 0, SPREAD_MAX)

    cont = np.zeros((n, N_FULL), dtype=np.float32)
    cont[:, 0] = np.log1p(np.maximum(dt, 0)) / 10
    cont[:, 1] = dt == 0
    cont[:, 2] = np.log1p(np.maximum(size, 0)) / 5      # shares the event moved (message)
    cont[:, 3] = (size % 100) == 0
    cont[:, 4] = (t / 1e9 - 34_200) / 23_400
    levels = b.shape[1] // 4
    av, bv = b[:, 1::4][:, :10], b[:, 3::4][:, :10]
    ap, bp = b[:, 0::4][:, :10], b[:, 2::4][:, :10]
    cont[:, 5:5 + av.shape[1]] = np.log1p(av) / 5
    cont[:, 15:15 + bv.shape[1]] = np.log1p(bv) / 5
    if levels > 1:
        k = min(levels, 10) - 1
        ga = np.where((av[:, 1:] > 0) & (av[:, :1] > 0), (ap[:, 1:] - ap[:, :1]) / TICK / 10, 0)
        gb = np.where((bv[:, 1:] > 0) & (bv[:, :1] > 0), (bp[:, :1] - bp[:, 1:]) / TICK / 10, 0)
        cont[:, 25:25 + k] = np.clip(ga, 0, 10)
        cont[:, 34:34 + k] = np.clip(gb, 0, 10)
    tot = askv + bidv
    cont[:, 43] = np.where(tot > 0, (bidv - askv) / np.where(tot > 0, tot, 1), 0)
    cont[:, 44] = np.clip(dmid, -10, 10) / 10
    cont[:, 45] = np.log1p(av.sum(1)) / 8
    cont[:, 46] = np.log1p(bv.sum(1)) / 8
    return cat, cont


def regime_inputs(actions, msgs, book, dmid) -> np.ndarray:
    """The 14 per-event quantities averaged by the day-so-far features, float64 [N, 14]."""
    a, m, b = np.asarray(actions), np.asarray(msgs), np.asarray(book)
    typ, side, p, dt = a[:, TYPE], a[:, SIDE], a[:, PRICE], a[:, DT]
    size = m[:, 3]
    sub = typ == SUBMIT
    two = (b[:, 1] > 0) & (b[:, 3] > 0)
    x = np.empty((len(a), N_REG))
    for k in range(5):
        x[:, k] = typ == k + 1
    x[:, 5] = side > 0
    x[:, 6] = sub & (size == 100)
    x[:, 7] = sub & (size % 100 == 0)
    x[:, 8] = np.log1p(np.maximum(size, 0)) / 5
    x[:, 9] = np.log1p(np.maximum(dt, 0)) / 10
    x[:, 10] = dt == 0
    x[:, 11] = ((typ == CANCEL) | (typ == DELETE)) & (p < 0)
    x[:, 12] = np.where(two, np.clip((b[:, 0] - b[:, 2]) / TICK, 0, 10) / 10, 0)
    x[:, 13] = np.clip(np.abs(dmid), 0, 5) / 5
    return x


@njit(cache=True)
def _ema_scan(x, valid, decay, num, den, out):
    """Day-so-far means, in place: num/den updated by each valid event; out[i] after event i."""
    n, r = x.shape
    h = decay.shape[0]
    for i in range(n):
        if valid[i]:
            for k in range(h):
                den[k] = decay[k] * den[k] + 1.0
                for j in range(r):
                    num[k, j] = decay[k] * num[k, j] + x[i, j]
        for k in range(h):
            inv = 1.0 / den[k] if den[k] > 0 else 0.0
            for j in range(r):
                out[i, k * r + j] = num[k, j] * inv


def flow_inputs(msgs) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Visible trades (LOBSTER type 4), aggressor sign (-direction of the resting order, as in
    facts.py) and log size / 5."""
    m = np.asarray(msgs)
    trade = m[:, 1] == EXEC
    return trade, -m[:, 5].astype(np.float64), np.log1p(np.maximum(m[:, 3], 0)) / 5


@njit(cache=True)
def _flow_scan(trade, eps, lsize, valid, mid, tr_decay, m_decay, tr_num, tr_den, since,
               m_num, m_den, out):
    """Order-flow memory, in place (see the module docstring); out[i] after event i."""
    n = trade.shape[0]
    T, M = tr_decay.shape[0], m_decay.shape[0]
    for i in range(n):
        if valid[i]:
            since[0] += 1.0
            if trade[i]:
                since[0] = 0.0
                for k in range(T):
                    tr_den[k] = tr_decay[k] * tr_den[k] + 1.0
                    tr_num[k, 0] = tr_decay[k] * tr_num[k, 0] + eps[i]
                    tr_num[k, 1] = tr_decay[k] * tr_num[k, 1] + eps[i] * lsize[i]
            if not np.isnan(mid[i]):
                for k in range(M):
                    m_den[k] = m_decay[k] * m_den[k] + 1.0
                    m_num[k] = m_decay[k] * m_num[k] + mid[i]
        for k in range(T):
            inv = 1.0 / tr_den[k] if tr_den[k] > 0 else 0.0
            out[i, k] = tr_num[k, 0] * inv
            out[i, T + k] = tr_num[k, 1] * inv
        out[i, 2 * T] = np.log1p(since[0]) / 10
        for k in range(M):
            d = 0.0
            if m_den[k] > 0 and not np.isnan(mid[i]):
                d = (mid[i] - m_num[k] / m_den[k]) / 10
            out[i, 2 * T + 1 + k] = min(max(d, -3.0), 3.0)


def _scan(st: FeatState, a: np.ndarray, msgs, book, want_features: bool):
    """Advance `st` in place over a stream; return (dmid, day-so-far + flow + own clock-time
    columns or None)."""
    dmid, filled, st.prev_mid = _mid_moves(_mids(book), st.prev_mid)
    own = np.empty((len(a), N_OWN))
    own_scan(np.asarray(msgs)[:, 0], filled, OWN_TAUS_NS, st.c_E, st.c_state, own)
    x = regime_inputs(a, msgs, book, dmid)
    valid = np.isin(a[:, TYPE], MODELED_TYPES)
    reg = np.empty((len(a), N_REG * len(HALF_LIVES)))
    _ema_scan(x, valid, _DECAY, st.num, st.den, reg)
    flow = np.empty((len(a), N_FLOW))
    trade, eps, lsize = flow_inputs(msgs)
    _flow_scan(trade, eps, lsize, valid, filled, _TR_DECAY, _MID_DECAY, st.tr_num, st.tr_den,
               st.since, st.m_num, st.m_den, flow)
    return dmid, (np.hstack([reg, flow, own]) if want_features else None)


def token_features(actions: np.ndarray, msgs: np.ndarray, book: np.ndarray,
                   state: FeatState | None = None,
                   market=None) -> tuple[np.ndarray, np.ndarray, FeatState]:
    """One stream of consecutive events: actions [N, 6] as applied, LOBSTER messages [N, 6],
    book [N, 4L] after each event. Returns (cat int64 [N, 4], cont float32 [N, N_FULL], state
    after the last event). Chunking a stream and passing the state on gives identical output.
    A model reads the first ModelConfig.n_cont columns (v1: N_BASE, v2: N_CONT, v3: N_ALL,
    v4: N_FULL). market: market.Market of the day for the ETF columns (else they are 0)."""
    st = FeatState() if state is None else state.copy()
    a = np.asarray(actions)
    dmid, extra = _scan(st, a, msgs, book, True)
    cat, cont = _base(a, msgs, book, dmid)
    cont[:, N_BASE:N_ALL + N_OWN] = extra
    if market is not None:
        cont[:, N_ALL + N_OWN:] = market.features(np.asarray(msgs)[:, 0])
    return cat, cont, st


def advance(state: FeatState, actions, msgs, book) -> FeatState:
    """State after a stream of events, without building the per-event features."""
    st = state.copy()
    _scan(st, np.asarray(actions), msgs, book, False)
    return st


def states_before(actions, msgs, book, rows, chunk: int = 1_000_000) -> list[FeatState]:
    """Feature state just before each row index in `rows` (ascending) of a day's stream."""
    rows = list(rows)
    if rows != sorted(rows):
        raise ValueError("rows must be ascending")
    st, j, out = FeatState(), 0, []
    for r in rows:
        while j < r:
            e = min(r, j + chunk)
            st = advance(st, actions[j:e], msgs[j:e], book[j:e])
            j = e
        out.append(st.copy())
    return out


class BatchState:
    """FeatStates of B independent streams, stepped one event at a time (rollouts).
    markets: one market.Market (or None) per stream, for the ETF columns."""

    def __init__(self, states: list[FeatState], markets: list | None = None) -> None:
        self.markets = markets if markets is not None else [None] * len(states)
        self.c_E = np.stack([s.c_E for s in states])          # [B, N_OWN]
        self.c_x = np.array([s.c_state[0] for s in states], dtype=np.float64)
        self.c_t = np.array([s.c_state[1] for s in states], dtype=np.float64)
        self.prev_mid = np.array([s.prev_mid for s in states], dtype=np.float64)
        self.num = np.stack([s.num for s in states])          # [B, H, R]
        self.den = np.stack([s.den for s in states])          # [B, H]
        self.tr_num = np.stack([s.tr_num for s in states])    # [B, T, 2]
        self.tr_den = np.stack([s.tr_den for s in states])    # [B, T]
        self.since = np.array([s.since[0] for s in states], dtype=np.float64)
        self.m_num = np.stack([s.m_num for s in states])      # [B, M]
        self.m_den = np.stack([s.m_den for s in states])      # [B, M]

    def step(self, actions, msgs, book) -> tuple[np.ndarray, np.ndarray]:
        """One new event per stream (row b = stream b). Same values token_features gives."""
        mid = _mids(book)
        dmid = np.nan_to_num(mid - self.prev_mid, nan=0.0)
        self.prev_mid = np.where(np.isnan(mid), self.prev_mid, mid)
        cat, cont = _base(actions, msgs, book, dmid)
        x = regime_inputs(actions, msgs, book, dmid)
        valid = np.isin(np.asarray(actions)[:, TYPE], MODELED_TYPES)
        d = _DECAY[None, :]
        self.den = np.where(valid[:, None], d * self.den + 1.0, self.den)
        self.num = np.where(valid[:, None, None], d[..., None] * self.num + x[:, None, :], self.num)
        inv = np.where(self.den > 0, 1.0 / np.where(self.den > 0, self.den, 1), 0.0)
        cont[:, N_BASE:N_CONT] = (self.num * inv[..., None]).reshape(len(x), -1)
        cont[:, N_CONT:N_ALL] = self._flow_step(msgs, valid, self.prev_mid)
        t = np.asarray(msgs)[:, 0]
        cont[:, N_ALL:N_ALL + N_OWN] = self._own_step(t, self.prev_mid)
        groups: dict[int, list[int]] = {}
        for b, mk in enumerate(self.markets):
            if mk is not None:
                groups.setdefault(id(mk), []).append(b)
        for rows in groups.values():
            cont[rows, N_ALL + N_OWN:] = self.markets[rows[0]].features(t[rows])
        return cat, cont

    def _own_step(self, t, mid) -> np.ndarray:
        """market.own_scan for one event of each stream (mid = forward-filled mid after it)."""
        t = t.astype(np.float64)
        dt = t - self.c_t
        upd = ~np.isnan(self.c_x) & (dt > 0)
        a = np.exp(-np.where(upd, dt, 0.0)[:, None] / OWN_TAUS_NS[None, :])
        self.c_E = np.where(upd[:, None], a * self.c_E + (1 - a) * np.nan_to_num(self.c_x)[:, None],
                            self.c_E)
        with np.errstate(invalid="ignore", divide="ignore"):
            x = np.log(mid)
        has = ~np.isnan(x)
        first = np.isnan(self.c_x) & has
        self.c_E = np.where(first[:, None], np.nan_to_num(x)[:, None], self.c_E)
        self.c_x = np.where(has, x, self.c_x)
        self.c_t = t
        d = np.where(np.isnan(self.c_x)[:, None], 0.0,
                     (np.nan_to_num(self.c_x)[:, None] - self.c_E) * 1e4 / 10)
        return np.clip(d, -5, 5)

    def _flow_step(self, msgs, valid, mid) -> np.ndarray:
        """_flow_scan for one event of each stream (mid = forward-filled mid after it)."""
        trade, eps, lsize = flow_inputs(msgs)
        tr = valid & trade
        self.since = np.where(valid, np.where(trade, 0.0, self.since + 1.0), self.since)
        d = _TR_DECAY[None, :]
        self.tr_den = np.where(tr[:, None], d * self.tr_den + 1.0, self.tr_den)
        new = d[..., None] * self.tr_num + np.stack([eps, eps * lsize], -1)[:, None, :]
        self.tr_num = np.where(tr[:, None, None], new, self.tr_num)
        has_mid = valid & ~np.isnan(mid)
        dm = _MID_DECAY[None, :]
        self.m_den = np.where(has_mid[:, None], dm * self.m_den + 1.0, self.m_den)
        self.m_num = np.where(has_mid[:, None], dm * self.m_num + np.nan_to_num(mid)[:, None],
                              self.m_num)
        inv = np.where(self.tr_den > 0, 1.0 / np.where(self.tr_den > 0, self.tr_den, 1), 0.0)
        mavg = self.m_num / np.where(self.m_den > 0, self.m_den, 1)
        dev = np.where((self.m_den > 0) & ~np.isnan(mid)[:, None], (mid[:, None] - mavg) / 10, 0.0)
        return np.hstack([self.tr_num[..., 0] * inv, self.tr_num[..., 1] * inv,
                          np.log1p(self.since)[:, None] / 10, np.clip(np.nan_to_num(dev), -3, 3)])

In [ ]:
%%writefile /content/lobgen/src/lobgen/data.py
"""Converted days, evaluation windows, and the generator interface every generator implements.

A *window* is a point in a real day: the `cond_len` real events before it (what a generator
may condition on), the order-level book at that moment, and the `gen_len` real events after it
(what the generator's output is compared with). Generators get the context and the book state,
never the real continuation.
"""

from __future__ import annotations

from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np

from .features import FeatState, states_before
from .sim import MODELED_TYPES, TYPE, SimBook, Simulator, states_at

NS = 1_000_000_000
OPEN_S, CLOSE_S = 34_200, 57_600


@dataclass
class Day:
    ticker: str
    date: str
    msgs: np.ndarray
    book: np.ndarray | None                # None when loaded with book=False
    actions: np.ndarray | None = None      # [N, 6] simulator actions (scripts/build_actions.py)
    init_orders: np.ndarray | None = None  # order-level book before msgs[0], oldest first
    encoding: tuple = (0, False)           # encoding of `actions`: sim.encode (pointer, whole_fill)

    @classmethod
    def load(cls, npz: str | Path, actions: bool = True, book: bool = True) -> "Day":
        """book=False skips the snapshot array (~3 GB for a GOOG day) when only actions are needed."""
        p = Path(npz)  # GOOG_2025-12-12_34200000_57600000_10.npz
        ticker, date = p.name.split("_")[:2]
        z = np.load(p)
        day = cls(ticker, date, z["messages"], z["book"] if book else None)
        a = p.with_name(p.stem + "_actions.npz")
        if actions:
            if not a.exists():
                raise FileNotFoundError(f"{a} missing: run scripts/build_actions.py")
            za = np.load(a)
            day.actions, day.init_orders = za["actions"], za["init_orders"]
            day.encoding = (int(za["pointer"]) if "pointer" in za.files else 0,
                            bool(za["whole_fill"]) if "whole_fill" in za.files else False)
        return day

    def time_of_day_ns(self) -> np.ndarray:
        return self.msgs[:, 0]

    def model_rows(self, margin_s: int = 300) -> np.ndarray:
        """Row mask: modeled event types, away from the open/close crosses."""
        t = self.msgs[:, 0]
        return (np.isin(self.msgs[:, 1], MODELED_TYPES)
                & (t >= (OPEN_S + margin_s) * NS) & (t < (CLOSE_S - margin_s) * NS))


@dataclass
class Window:
    i: int
    start: int              # day row index of the first continuation event
    t_prev: int             # time of the last context event (ns)
    state: np.ndarray       # order-level book just before `start`, [K, 4]
    cond_msgs: np.ndarray
    cond_book: np.ndarray
    cond_actions: np.ndarray
    real_msgs: np.ndarray
    real_book: np.ndarray
    real_actions: np.ndarray
    encoding: tuple = (0, False)        # action encoding of cond_actions / real_actions
    feat_state: FeatState | None = None  # model feature stream just before cond_msgs[0]
    market: object = None                # market.Market the run is conditioned on (v4 models)


def pick_windows(day: Day, n: int, cond_len: int, gen_len: int, rng: np.random.Generator,
                 margin_s: int = 300) -> np.ndarray:
    """n non-overlapping window starts (index of first continuation msg), away from open/close."""
    t = day.time_of_day_ns()
    lo = max(cond_len, int(np.searchsorted(t, (OPEN_S + margin_s) * NS)))
    hi = min(len(t) - gen_len, int(np.searchsorted(t, (CLOSE_S - margin_s) * NS)))
    span = cond_len + gen_len
    if (hi - lo) < n * span:
        raise ValueError(f"{day.ticker} {day.date}: not enough events for {n} windows")
    # Stratified: one start per equal slice of the day keeps time-of-day coverage even.
    edges = np.linspace(lo, hi - span, n + 1).astype(np.int64)
    return np.array([rng.integers(edges[k], max(edges[k] + 1, edges[k + 1] - span))
                     for k in range(n)]) + cond_len


def make_windows(day: Day, n: int, cond_len: int = 500, gen_len: int = 1000, seed: int = 0,
                 margin_s: int = 300, feat_states: bool = True) -> list[Window]:
    """feat_states: also run the model's feature stream from the open to each context start
    (what a model generator needs; a few seconds per day)."""
    starts = pick_windows(day, n, cond_len, gen_len, np.random.default_rng(seed), margin_s)
    return windows_at(day, starts, cond_len, gen_len, feat_states)


def windows_at(day: Day, starts, cond_len: int = 500, gen_len: int = 1000,
               feat_states: bool = True) -> list[Window]:
    """Windows whose continuation starts at the given day rows (ascending; e.g. the row right
    after a large market order, for the shock test)."""
    starts = np.asarray(starts, dtype=np.int64)
    if len(starts) and (starts[0] < cond_len or starts[-1] + gen_len > len(day.msgs)):
        raise ValueError("a window runs past the start or end of the day")
    states = states_at(day.msgs, day.init_orders, starts)
    fs = (states_before(day.actions, day.msgs, day.book, (starts - cond_len).tolist())
          if feat_states else [None] * len(starts))
    out = []
    for i, (s, st, f) in enumerate(zip(starts.tolist(), states, fs)):
        c, e = slice(s - cond_len, s), slice(s, s + gen_len)
        out.append(Window(i, s, int(day.msgs[s - 1, 0]), st,
                          day.msgs[c], day.book[c], day.actions[c],
                          day.msgs[e], day.book[e], day.actions[e], day.encoding, f))
    return out


@dataclass
class Generated:
    msgs: np.ndarray
    book: np.ndarray
    repairs: Counter = field(default_factory=Counter)
    skipped: int = 0
    actions: np.ndarray | None = None  # the actions as applied (after repairs), [N, 6]


class Generator:
    """Produces a continuation for each window. Subclasses set `name` and override generate()
    (or, for per-event samplers, subclass ActionGenerator)."""

    name = "generator"

    def generate(self, windows: list[Window], n_events: int, seed: int = 0) -> list[Generated]:
        raise NotImplementedError


class ActionGenerator(Generator):
    """A generator that emits one action at a time and lets the simulator build the book.

    Subclasses implement reset(window, rng) and sample(sim, rng) -> action tuple, and may use
    observe(applied_action) to track their own history.
    """

    max_skips = 1000  # inapplicable actions (2-4 on an empty side) are redrawn

    def reset(self, w: Window, rng: np.random.Generator) -> None:
        pass

    def sample(self, sim: Simulator, rng: np.random.Generator) -> tuple:
        raise NotImplementedError

    def observe(self, action: tuple) -> None:
        pass

    def rollout(self, w: Window, n_events: int, rng: np.random.Generator,
                levels: int = 10) -> Generated:
        sim = Simulator(SimBook.from_orders(w.state), w.t_prev, levels)
        self.reset(w, rng)
        msgs, books, acts = [], [], []
        while len(msgs) < n_events:
            out = sim.step(*self.sample(sim, rng))
            if out is None:
                if sim.skipped > self.max_skips:
                    raise RuntimeError(f"{self.name}: window {w.i} stuck after {sim.skipped} skips")
                continue
            msgs.append(out[0])
            books.append(out[1])
            acts.append(sim.last_action)
            self.observe(sim.last_action)
        return Generated(np.array(msgs, dtype=np.int64), np.array(books, dtype=np.int64),
                         sim.repairs, sim.skipped, np.array(acts, dtype=np.int64))

    def generate(self, windows: list[Window], n_events: int, seed: int = 0) -> list[Generated]:
        return [self.rollout(w, n_events, np.random.default_rng([seed, w.i])) for w in windows]


def modeled(actions: np.ndarray) -> np.ndarray:
    return actions[np.isin(actions[:, TYPE], MODELED_TYPES)]

In [ ]:
%%writefile /content/lobgen/src/lobgen/baselines.py
"""Reference generators (the "perfect" end of the scale) and dumb baselines (the "clearly bad" end).

References, built from real data:
  Replay        the window's real continuation re-played through the simulator. Output equals
                the real data except for order ids, so a detective must score exactly chance;
                a check on the simulator and the detective, not a floor.
  RealOtherDay  the real continuation of another day at the same time of day. The noise floor:
                a perfect model of "a day like this one" cannot beat it.

Baselines, all driving the simulator so their output is a valid book:
  HistoryBootstrap  each event is a random draw from the window's own conditioning events.
  PooledIID         each event is a random draw from all events of the training days.
  Markov            first-order Markov chain over a 40-class summary of the previous event
                    (type x side x price bucket); the next event is a random training event that
                    followed the same class, so all its fields (dt, price, size...) come jointly.
"""

from __future__ import annotations

import numpy as np

from .data import ActionGenerator, Day, Generated, Generator, Window, modeled
from .sim import CANCEL, DELETE, EXEC, PRICE, SIDE, SUBMIT, TYPE, Simulator

N_CLASSES = 40


class Replay(ActionGenerator):
    name = "replay"

    def reset(self, w: Window, rng) -> None:
        self._it = iter(w.real_actions.tolist())

    def sample(self, sim: Simulator, rng) -> tuple:
        return next(self._it)


class RealOtherDay(Generator):
    def __init__(self, other: Day) -> None:
        self.other = other
        self.name = f"real_{other.date}"

    def generate(self, windows: list[Window], n_events: int, seed: int = 0) -> list[Generated]:
        t = self.other.time_of_day_ns()
        out = []
        for w in windows:
            j = min(int(np.searchsorted(t, w.real_msgs[0, 0])), len(t) - n_events)
            out.append(Generated(self.other.msgs[j:j + n_events].copy(),
                                 self.other.book[j:j + n_events].copy()))
        return out


class _PoolSampler(ActionGenerator):
    """Draws whole action rows from a pool; draws are made in blocks for speed."""

    _block = 4096

    def _draw(self, pool: np.ndarray, rng) -> tuple:
        if self._k >= len(self._idx):
            self._idx = rng.integers(len(pool), size=self._block).tolist()
            self._k = 0
        j = self._idx[self._k]
        self._k += 1
        return tuple(pool[j].tolist())


class HistoryBootstrap(_PoolSampler):
    name = "history"

    def reset(self, w: Window, rng) -> None:
        self.pool = modeled(w.cond_actions)
        self._idx, self._k = [], 0

    def sample(self, sim: Simulator, rng) -> tuple:
        return self._draw(self.pool, rng)


def train_pool(days: list[Day], max_events: int = 3_000_000, seed: int = 0) -> list[np.ndarray]:
    """Modeled actions of each training day (open/close margins dropped), subsampled in
    contiguous blocks to at most max_events in total so consecutive pairs survive."""
    per_day = max_events // max(len(days), 1)
    rng = np.random.default_rng(seed)
    out = []
    for d in days:
        a = d.actions[d.model_rows()]
        if len(a) > per_day:
            s = int(rng.integers(0, len(a) - per_day))
            a = a[s:s + per_day]
        out.append(a)
    return out


class PooledIID(_PoolSampler):
    name = "pooled"

    def __init__(self, train: list[np.ndarray]) -> None:
        self.pool = np.concatenate(train)

    def reset(self, w: Window, rng) -> None:
        self._idx, self._k = [], 0

    def sample(self, sim: Simulator, rng) -> tuple:
        return self._draw(self.pool, rng)


def event_class(a: np.ndarray) -> np.ndarray:
    """40 classes: type (5) x side (2) x price bucket (4).

    Submit bucket: inside the spread / at the own best / 1 tick deeper / deeper.
    Cancel, delete, execution bucket: level 0 / 1 / 2+, or (cancel, delete) a pointer to one
    of the newest orders. Hidden executions: one bucket.
    """
    a = np.atleast_2d(a)
    typ, side, p = a[:, TYPE], a[:, SIDE], a[:, PRICE]
    bucket = np.where(typ == SUBMIT, np.clip(p, -1, 2) + 1,
                      np.where(np.isin(typ, [CANCEL, DELETE, EXEC]),
                               np.where(p < 0, 3, np.clip(p, 0, 2)), 0))
    return ((np.clip(typ, 1, 5) - 1) * 2 + (side > 0)) * 4 + bucket


class Markov(ActionGenerator):
    name = "markov"

    def __init__(self, train: list[np.ndarray]) -> None:
        prev = np.concatenate([event_class(a[:-1]) for a in train])
        nxt = np.concatenate([a[1:] for a in train])
        order = np.argsort(prev, kind="stable")
        self.next = nxt[order]
        self.bounds = np.searchsorted(prev[order], np.arange(N_CLASSES + 1))
        self.pool = nxt

    def reset(self, w: Window, rng) -> None:
        self.c = int(event_class(w.cond_actions[-1])[0])

    def sample(self, sim: Simulator, rng) -> tuple:
        lo, hi = self.bounds[self.c], self.bounds[self.c + 1]
        if hi == lo:  # class never seen in training: fall back to the marginal
            return tuple(self.pool[rng.integers(len(self.pool))].tolist())
        return tuple(self.next[rng.integers(lo, hi)].tolist())

    def observe(self, action: tuple) -> None:
        self.c = int(event_class(np.array(action))[0])

In [ ]:
%%writefile /content/lobgen/src/lobgen/detective.py
"""The detective: a classifier that tells real from generated runs, at each run length.

For a run length L, every sequence (real continuation or generated one) is cut to its first
L events and summarised by ~70 features computed from the LOBSTER rows alone (messages + book
snapshots), so any generator's output can be judged, including files written by other code:

  event mix   share of each type x side
  timing      share of zero gaps, quantiles of log inter-arrival time, burstiness, duration
  sizes       log size by event type, share of round lots / exactly 100 shares
  prices      submit / cancel / execution price relative to the book just before the event
  book        spread, touch depth, imbalance, total depth, book span, mid-price path
  sequence    replace pairs (delete then submit, same side, same ns), side / type persistence,
              execution sweeps
  lifecycle   cancels and executions of orders submitted inside the run, time to delete

All features are relative (ticks from the book, logs of sizes and gaps): no absolute price,
no order id value. The detective sees only the continuation, never the conditioning context;
otherwise "real data from another day" would be caught by the jump at the window start and
could not serve as a noise floor.

Scoring: gradient-boosted trees, 5-fold cross-validation grouped by window (a window's real
and generated runs are always in the same fold), out-of-fold AUC and accuracy, with 95%
bootstrap intervals over windows. AUC 0.5 = indistinguishable, 1.0 = always caught.
"""

from __future__ import annotations

import warnings

import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

TICK = 100
Seq = tuple[np.ndarray, np.ndarray]  # (messages [L, 6], book [L, 4*levels])


def _mean(x) -> float:
    return float(np.mean(x)) if len(x) else np.nan


def _std(x) -> float:
    return float(np.std(x)) if len(x) > 1 else np.nan


def _q(x, qs) -> list[float]:
    return np.quantile(x, qs).tolist() if len(x) else [np.nan] * len(qs)


def _corr(a, b) -> float:
    if len(a) < 3 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def features(m: np.ndarray, b: np.ndarray, tick: int = TICK) -> dict[str, float]:
    t, typ, ref, size, price, d = (m[:, k] for k in range(6))
    f: dict[str, float] = {}
    for k in range(1, 6):
        for s, sname in ((1, "b"), (-1, "s")):
            f[f"share_t{k}{sname}"] = _mean((typ == k) & (d == s))

    dt = np.diff(t)
    ldt = np.log10(dt + 1.0)
    pos = ldt[dt > 0]
    f["dt_zero"] = _mean(dt == 0)
    for q, v in zip((10, 25, 50, 75, 90), _q(pos, [0.1, 0.25, 0.5, 0.75, 0.9])):
        f[f"ldt_q{q}"] = v
    f["ldt_mean"], f["ldt_std"] = _mean(ldt), _std(ldt)
    f["ldt_ac1"] = _corr(ldt[:-1], ldt[1:])
    f["log_duration"] = float(np.log10(t[-1] - t[0] + 1.0))

    sub, can, dele, exe, hid = (typ == k for k in (1, 2, 3, 4, 5))
    lsize = np.log10(np.maximum(size, 1))
    for name, mask in (("sub", sub), ("can", can), ("del", dele), ("exe", exe), ("hid", hid)):
        f[f"{name}_lsize"] = _mean(lsize[mask])
    f["sub_lsize_std"] = _std(lsize[sub])
    f["sub_size100"] = _mean(size[sub] == 100)
    f["sub_round"] = _mean(size[sub] % 100 == 0)
    f["exe_round"] = _mean(size[exe] % 100 == 0)

    # Prices relative to the own-side best just before the event (+ = deeper in the book).
    own = np.where(d[1:] > 0, b[:-1, 2], b[:-1, 0])
    own_ok = np.where(d[1:] > 0, b[:-1, 3], b[:-1, 1]) > 0
    rel = d[1:] * (own - price[1:]) / tick
    s1, c1, e1, h1 = (x[1:] & own_ok for x in (sub, can | dele, exe, hid))
    rs = rel[s1]
    f["sub_inside"], f["sub_at"] = _mean(rs < 0), _mean(rs == 0)
    f["sub_1"], f["sub_2_5"], f["sub_deep"] = _mean(rs == 1), _mean((rs >= 2) & (rs <= 5)), _mean(rs > 5)
    f["sub_rel"] = _mean(np.clip(rs, -5, 20))
    f["can_at"], f["can_rel"] = _mean(rel[c1] == 0), _mean(np.clip(rel[c1], 0, 20))
    f["exe_at"] = _mean(rel[e1] == 0)
    f["hid_rel"] = _mean(np.clip(rel[h1], -10, 10))

    ask, askv, bid, bidv = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    two = (askv > 0) & (bidv > 0)
    spread = (ask[two] - bid[two]) / tick
    f["spread_mean"], f["spread_std"] = _mean(spread), _std(spread)
    f["spread_min"] = float(spread.min()) if len(spread) else np.nan
    f["spread_max"] = float(spread.max()) if len(spread) else np.nan
    f["spread_1"], f["spread_2"] = _mean(spread == 1), _mean(spread == 2)
    f["spread_changes"] = _mean(np.diff(spread) != 0)
    la, lb = np.log1p(askv), np.log1p(bidv)
    f["l1_ask_mean"], f["l1_ask_std"] = _mean(la), _std(la)
    f["l1_bid_mean"], f["l1_bid_std"] = _mean(lb), _std(lb)
    tot = askv + bidv
    imb = ((bidv - askv) / np.where(tot > 0, tot, 1))[tot > 0]
    f["imb_mean"], f["imb_std"], f["imb_abs"] = _mean(imb), _std(imb), _mean(np.abs(imb))
    av, bv = b[:, 1::4], b[:, 3::4]
    f["depth_ask"], f["depth_bid"] = _mean(np.log1p(av.sum(1))), _mean(np.log1p(bv.sum(1)))
    f["levels_ask"], f["levels_bid"] = _mean((av > 0).sum(1)), _mean((bv > 0).sum(1))
    last = b.shape[1] // 4 - 1
    full_a, full_b = av[:, last] > 0, bv[:, last] > 0
    f["span_ask"] = _mean((b[full_a, 4 * last] - ask[full_a]) / tick)
    f["span_bid"] = _mean((bid[full_b] - b[full_b, 4 * last + 2]) / tick)
    mid = (ask[two] + bid[two]) / 2 / tick
    dmid = np.diff(mid)
    f["mid_changes"] = _mean(dmid != 0)
    f["mid_move"] = float(abs(mid[-1] - mid[0])) if len(mid) else np.nan
    f["mid_path"] = float(np.abs(dmid).sum()) if len(dmid) else np.nan
    f["mid_rv"] = float(np.sqrt((dmid ** 2).sum())) if len(dmid) else np.nan

    f["replace_pairs"] = _mean(dele[:-1] & sub[1:] & (dt == 0) & (d[:-1] == d[1:]))
    f["same_side"] = _mean(d[1:] == d[:-1])
    f["same_type"] = _mean(typ[1:] == typ[:-1])
    f["exec_sweep"] = _mean(exe[:-1] & exe[1:] & (dt == 0))

    sub_time = dict(zip(ref[sub].tolist(), t[sub].tolist()))
    cd = can | dele
    young_cd = np.array([r in sub_time for r in ref[cd].tolist()], dtype=bool)
    f["young_cancel"] = _mean(young_cd)
    f["young_exec"] = _mean(np.array([r in sub_time for r in ref[exe].tolist()], dtype=bool))
    ttd = [tt - sub_time[r] for r, tt in zip(ref[dele].tolist(), t[dele].tolist()) if r in sub_time]
    lttd = np.log10(np.array(ttd, dtype=float) + 1.0)
    f["ttd_q25"], f["ttd_q50"], f["ttd_q75"] = _q(lttd, [0.25, 0.5, 0.75])
    gone = set(ref[dele].tolist())
    f["sub_deleted"] = _mean(np.array([r in gone for r in ref[sub].tolist()], dtype=bool))
    return f


def feature_matrix(seqs: list[Seq], length: int) -> tuple[np.ndarray, list[str]]:
    rows = []
    for m, b in seqs:
        if len(m) < length:
            raise ValueError(f"sequence of {len(m)} events is shorter than run length {length}")
        rows.append(features(m[:length], b[:length]))
    names = list(rows[0])
    return np.array([[r[k] for k in names] for r in rows], dtype=np.float64), names


def _classifier(seed: int) -> HistGradientBoostingClassifier:
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=20, l2_regularization=1.0,
                                          random_state=seed)


def _boot_ci(y, p, groups, n_boot: int, rng) -> tuple[list[float], list[float]]:
    ug, inv = np.unique(groups, return_inverse=True)
    members = [np.flatnonzero(inv == k) for k in range(len(ug))]
    aucs, accs = [], []
    for _ in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(len(ug), size=len(ug))])
        if len(np.unique(y[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y[idx], p[idx]))
        accs.append(np.mean((p[idx] > 0.5) == y[idx]))
    return (np.quantile(aucs, [0.025, 0.975]).tolist(), np.quantile(accs, [0.025, 0.975]).tolist())


def detect(real: list[Seq], fake: list[Seq], lengths, real_groups=None, fake_groups=None,
           folds: int = 5, seed: int = 0, n_boot: int = 500, importance: bool = True) -> dict:
    """Out-of-fold real-vs-fake AUC / accuracy at each run length.

    real[i] and fake[i] belong to window i unless groups are given; a group never spans folds.
    """
    real_groups = np.arange(len(real)) if real_groups is None else np.asarray(real_groups)
    fake_groups = np.arange(len(fake)) if fake_groups is None else np.asarray(fake_groups)
    y = np.r_[np.zeros(len(real)), np.ones(len(fake))].astype(int)
    groups = np.r_[real_groups, fake_groups]
    rng = np.random.default_rng(seed)
    out = {}
    lengths = sorted(lengths)
    for L in lengths:
        Xr, names = feature_matrix(real, L)
        Xf, _ = feature_matrix(fake, L)
        X = np.vstack([Xr, Xf])
        p = np.empty(len(y))
        cv = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            for tr, te in cv.split(X, y, groups):
                p[te] = _classifier(seed).fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
        auc_ci, acc_ci = _boot_ci(y, p, groups, n_boot, rng)
        res = {"auc": float(roc_auc_score(y, p)), "auc_ci": auc_ci,
               "acc": float(np.mean((p > 0.5) == y)), "acc_ci": acc_ci,
               "n_real": len(real), "n_fake": len(fake)}
        if importance and L in (lengths[0], lengths[-1]):
            res["top_features"] = _importance(X, y, groups, names, seed)
        out[L] = res
    return out


def _importance(X, y, groups, names, seed, top: int = 10) -> list[tuple[str, float]]:
    """Permutation importance (drop in held-out AUC) on one 80/20 grouped split."""
    tr, te = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
                  .split(X, y, groups))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        clf = _classifier(seed).fit(X[tr], y[tr])
        r = permutation_importance(clf, X[te], y[te], scoring="roc_auc", n_repeats=5,
                                   random_state=seed)
    order = np.argsort(-r.importances_mean)[:top]
    return [(names[k], round(float(r.importances_mean[k]), 4)) for k in order]

In [ ]:
%%writefile /content/lobgen/src/lobgen/plots.py
"""Charts for results/. Needs the `bench` dependency group (matplotlib)."""

from __future__ import annotations

from pathlib import Path

import numpy as np

# Fixed colour per entity (validated categorical order, light mode); floors are neutral grey.
COLORS = {"markov": "#eb6834", "pooled": "#1baf7a", "history": "#eda100"}
# Trained models: the named runs keep one colour across every chart; any other model name takes
# the remaining slots in name order.
MODEL_COLORS = ["#2a78d6", "#6250d6", "#e87ba4", "#008300", "#e34948"]
NAMED_MODELS = {"full": MODEL_COLORS[0], "v1": MODEL_COLORS[1], "v2": MODEL_COLORS[2]}


def model_colors(names) -> dict:
    """Colour per trained-model name, stable across charts (colour follows the model)."""
    free = [c for c in MODEL_COLORS if c not in NAMED_MODELS.values()]
    others = sorted(n for n in names if n not in NAMED_MODELS)
    return {n: NAMED_MODELS[n] for n in sorted(names) if n in NAMED_MODELS} | dict(zip(others, free))
REFERENCES = {"same_day", "replay"}
OTHER_DAY_DASHES = [":", "-."]
TEXT, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"


def plot_detective(res: dict, ticker: str, path: str | Path, metric: str = "auc") -> Path:
    """Detective AUC vs run length: one line per generator, grey floors, 95% CI bands.

    res: {generator name: {run length (int or str): {"auc": .., "auc_ci": [lo, hi], ...}}}
    Generators whose name starts with "real_" are drawn as the other-day floor; "replay"
    (a sanity check that is 0.5 by construction) is left off.
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8.6, 4.6), dpi=150)
    lows = [0.45]

    def series(name):
        r = res[name]
        L = np.array(sorted(int(k) for k in r))
        get = lambda k: r[str(k)] if str(k) in r else r[k]  # noqa: E731
        y = np.array([get(k)[metric] for k in L])
        ci = np.array([get(k)[f"{metric}_ci"] for k in L])
        return L, y, ci

    others = sorted(n for n in res if n.startswith("real_"))
    for name in (["same_day"] if "same_day" in res else []) + others:
        L, y, ci = series(name)
        if name == "same_day":
            style, label = "--", "real, neighbouring window (floor)"
        else:
            style, label = OTHER_DAY_DASHES[others.index(name) % 2], f"real, other day ({name[5:]})"
        ax.fill_between(L, ci[:, 0], ci[:, 1], color="#8a8984", alpha=0.10, lw=0)
        ax.plot(L, y, style, color="#6f6e69", lw=1.6, label=label)
        lows.append(float(ci[:, 0].min()))

    models = [n for n in res if n not in COLORS and n not in REFERENCES and not n.startswith("real_")]
    colors = COLORS | model_colors(models)
    for name, color in colors.items():
        if name not in res:
            continue
        L, y, ci = series(name)
        ax.fill_between(L, ci[:, 0], ci[:, 1], color=color, alpha=0.15, lw=0)
        ax.plot(L, y, "-", color=color, lw=2, marker="o", ms=4, label=name)
        lows.append(float(ci[:, 0].min()))

    ax.set_xscale("log")
    ax.set_ylim(min(lows) - 0.02, 1.02)
    ax.axhline(0.5, color=MUTED, lw=0.8)
    ax.set_xlabel("run length (events the detective sees)", color=MUTED)
    ax.set_ylabel("detective AUC  (0.5 = can't tell, 1 = always caught)", color=MUTED)
    ax.set_title(f"{ticker}: how fast generated order flow gets caught", color=TEXT, loc="left",
                 fontsize=11)
    ax.grid(True, which="major", color=GRID, lw=0.6)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
    fig.tight_layout()
    path = Path(path)
    fig.savefig(path)
    plt.close(fig)
    return path


def plot_facts(facts: dict, ticker: str, path: str | Path) -> Path:
    """Six stylized facts, real continuations vs references vs generators (small multiples).

    facts: {name: stylized_facts(...) dict}; "real" is the reference, "real_<date>" another day.
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    from .facts import IMPACT_LAGS, RET_H, SIGN_LAGS, VOL_LAGS

    names = [n for n in facts if n != "real"]
    others = sorted(n for n in names if n.startswith("real_"))
    models = [n for n in names if n not in COLORS and not n.startswith("real_")]
    colors = COLORS | model_colors(models)

    def style(name):
        if name == "real":
            return dict(color=TEXT, lw=2.4, ls="-", label="real (same windows)", zorder=5)
        if name.startswith("real_"):
            return dict(color="#6f6e69", lw=1.6, ls=OTHER_DAY_DASHES[others.index(name) % 2],
                        label=f"real, other day ({name[5:]})")
        return dict(color=colors[name], lw=1.8, ls="-", label=name)

    order = ["real"] + others + [n for n in colors if n in facts]
    fig, axes = plt.subplots(2, 3, figsize=(12.5, 6.6), dpi=150)
    panels = [
        ("spread (share of events)", np.arange(1, 6), "spread", "spread, ticks (5 = 5+)", False),
        ("mid-price move std", np.array(RET_H), "ret_std", "horizon, events", True),
        ("volatility clustering: acf of |mid move|", VOL_LAGS, "vol_acf", "lag, events", False),
        ("trade-sign memory: acf of trade signs", SIGN_LAGS, "sign_acf", "lag, trades", False),
        ("price impact R(l), ticks", IMPACT_LAGS, "impact", "lag l, trades", False),
        ("depth profile (level 1 = 1)", np.arange(1, 11), "depth", "level", False),
    ]
    for ax, (title, x, key, xlabel, logy) in zip(axes.flat, panels):
        for name in order:
            f = facts[name]
            y = (np.add(f["depth_ask"], f["depth_bid"]) / 2) if key == "depth" else np.asarray(f[key])
            st = style(name)
            ax.plot(x[:len(y)], y, marker="o" if len(x) <= 10 else None, ms=3.5, **st)
        if key in ("vol_acf", "sign_acf", "impact", "ret_std"):
            ax.set_xscale("log")
        if logy:
            ax.set_yscale("log")
        if key in ("vol_acf", "sign_acf"):
            ax.axhline(0, color=MUTED, lw=0.8)
        ax.set_title(title, color=TEXT, loc="left", fontsize=9.5)
        ax.set_xlabel(xlabel, color=MUTED, fontsize=8)
        ax.grid(True, which="major", color=GRID, lw=0.6)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        for s in ("left", "bottom"):
            ax.spines[s].set_color(GRID)
        ax.tick_params(colors=MUTED, labelsize=7.5)
    h, lab = axes.flat[0].get_legend_handles_labels()
    fig.legend(h, lab, frameon=False, fontsize=8.5, loc="center left", bbox_to_anchor=(1.0, 0.5))
    fig.suptitle(f"{ticker}: stylized facts of 1000-event runs, real vs generated", color=TEXT,
                 x=0.01, ha="left", fontsize=11.5)
    fig.tight_layout()
    path = Path(path)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_shock(res: dict, ticker: str, path: str | Path, impact_ms: float | None = None) -> Path:
    """Shock test (scripts/shock_test.py): the reaction to a large market order, real vs
    generated. res: results/shock.json[ticker]."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    meta = res["_meta"]
    ev, ms = np.array(meta["events"]), np.array(meta["times_ms"])
    gens = [n for n in res if not n.startswith("_") and n not in ("real", "real_by_size")]
    models = [n for n in gens if n not in COLORS]
    colors = COLORS | model_colors(models)
    arr = lambda v: np.array([np.nan if x is None else x for x in v], dtype=float)  # noqa: E731

    def line(ax, x, s, color, label, real=False) -> bool:
        y, lo, hi = arr(s["mean"]), arr(s["lo"]), arr(s["hi"])
        ax.fill_between(x, lo, hi, color=color, alpha=0.18 if real else 0.12, lw=0)
        ax.plot(x, y, "-", color=color, lw=2.4 if real else 1.8, marker="o", ms=3.5,
                label=label, zorder=5 if real else 3)
        return bool(np.isfinite(y).any())

    fig, axes = plt.subplots(2, 3, figsize=(13, 7), dpi=150)
    panels = [
        ("mid move after the order, ticks (+ = order's direction)", "move_t", ms, "time after the order, ms"),
        ("mid move, ticks, by events after the order", "move_ev", ev, "events after the order"),
        ("bid-ask spread, ticks", "spread_ev", ev, "events after the order"),
        ("follow-on aggressive volume in the order's direction, shares", "flow_ev", ev,
         "events after the order"),
        ("time to the k-th event, ms", "ms_ev", ev, "k (events after the order)"),
    ]
    for ax, (title, key, x, xlabel) in zip(axes.flat, panels):
        drawn = line(ax, x, res["real"][key], TEXT, "real (what followed)", real=True)
        for n in gens:
            if key in res[n]:
                drawn |= line(ax, x, res[n][key], colors.get(n, MUTED), n)
        if not drawn:   # e.g. runs too short to reach any clock horizon
            ax.text(0.5, 0.5, "no horizon covered", transform=ax.transAxes, ha="center",
                    color=MUTED, fontsize=8)
            continue
        ax.set_xscale("log")
        if key == "ms_ev":   # symlog: a generator whose events all share one ns still plots
            ax.set_yscale("symlog", linthresh=0.01)
        if key in ("move_t", "move_ev", "flow_ev"):
            ax.axhline(0, color=MUTED, lw=0.8)
        ax.set_title(title, color=TEXT, loc="left", fontsize=9)
        ax.set_xlabel(xlabel, color=MUTED, fontsize=8)

    # Impact vs order size: real orders binned by size vs injected orders (net of no-order runs).
    ax = axes.flat[5]
    rb = res["real_by_size"]
    if impact_ms is None:
        cover = [k for k, c in enumerate(res["real"]["move_t"]["cover"]) if c >= 0.95]
        j = min(cover[-1] if cover else 2, 3)
    else:
        j = int(np.argmin(np.abs(ms - impact_ms)))
    xs, ys, los, his = [], [], [], []
    for b in rb["bins"]:
        if "mean" in b and b["mean"][j] is not None:
            xs.append(b["median_size"])
            ys.append(b["mean"][j])
            los.append(b["lo"][j])
            his.append(b["hi"][j])
    if xs:
        ax.errorbar(xs, ys, yerr=[np.subtract(ys, los), np.subtract(his, ys)], color=TEXT, fmt="o-",
                    lw=2.2, ms=5, capsize=3, label="real orders of that size", zorder=5)
    for n in gens:
        inj = res[n].get("inject")
        if not inj:
            continue
        xs = [b["size"] for b in inj["by_size"]]
        s = [b["impact_t"] for b in inj["by_size"]]
        y = [np.nan if v["mean"][j] is None else v["mean"][j] for v in s]
        lo = [np.nan if v["lo"][j] is None else v["lo"][j] for v in s]
        hi = [np.nan if v["hi"][j] is None else v["hi"][j] for v in s]
        c = colors.get(n, MUTED)
        ax.fill_between(xs, lo, hi, color=c, alpha=0.12, lw=0)
        ax.plot(xs, y, "s--", color=c, lw=1.6, ms=4, label=f"{n}, injected")
    if ax.has_data():
        ax.set_xscale("log")
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_title(f"price impact {ms[j]:g} ms after an order, ticks", color=TEXT, loc="left", fontsize=9)
    ax.set_xlabel("order size, shares", color=MUTED, fontsize=8)

    for ax in axes.flat:
        ax.grid(True, which="major", color=GRID, lw=0.6)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        for s in ("left", "bottom"):
            ax.spines[s].set_color(GRID)
        ax.tick_params(colors=MUTED, labelsize=7.5)
    h, lab = axes.flat[0].get_legend_handles_labels()
    h2, lab2 = axes.flat[5].get_legend_handles_labels()
    fig.legend(h + h2, lab + lab2, frameon=False, fontsize=8.5, loc="center left",
               bbox_to_anchor=(1.0, 0.5))
    fig.suptitle(f"{ticker}: reaction to {meta['n_shocks']} large market orders "
                 f"(>= {meta['min_size']:,} shares), test day {meta['test']}",
                 color=TEXT, x=0.01, ha="left", fontsize=11.5)
    fig.tight_layout()
    path = Path(path)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_track_case(case: dict, ticker: str, path: str | Path) -> Path:
    """Tracking case study (scripts/track_test.py): the real mid path around the test day's
    sharpest market drop, against generated runs (median and 10-90% band of the runs)."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    g = np.array(case["grid_s"])
    fig, ax = plt.subplots(figsize=(9, 4.8), dpi=150)
    names = list(case["runs"])
    base = sorted({n.split("@")[0] for n in names})
    colors = model_colors(base)
    for n in names:
        runs = np.array(case["runs"][n], dtype=float)
        if not len(runs):
            continue
        c = colors.get(n.split("@")[0], MUTED)
        frozen = n.endswith("@frozen")
        with np.errstate(all="ignore"):
            lo, med, hi = np.nanpercentile(runs, [10, 50, 90], 0)
        ax.fill_between(g, lo, hi, color=c, alpha=0.08 if frozen else 0.16, lw=0)
        ax.plot(g, med, ":" if frozen else "-", color=c, lw=1.8,
                label=f"{n.split('@')[0]}, market frozen at start (no news)" if frozen
                else f"{n} (median, 10-90% of runs)")
    q = np.array(case["qqq"], dtype=float)
    ax.plot(g, q * case["beta_qqq"], "--", color="#8a8984", lw=1.4,
            label=f"QQQ x beta {case['beta_qqq']:.2f} (training days)")
    ax.plot(g, np.array(case["real"], dtype=float), "-", color=TEXT, lw=2.4, label=f"real {ticker}",
            zorder=5)
    ax.axhline(0, color=MUTED, lw=0.8)
    hh, mm = divmod(int(case["t0_s"]) // 60, 60)
    ax.set_title(f"{ticker}: price path from {hh:02d}:{mm:02d} ET on the test day "
                 "(the day's sharpest market drop), real vs generated", color=TEXT, loc="left",
                 fontsize=10)
    ax.set_xlabel("seconds from the start of the run", color=MUTED, fontsize=8)
    ax.set_ylabel("mid move, bps", color=MUTED, fontsize=8)
    ax.grid(True, which="major", color=GRID, lw=0.6)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=7.5)
    ax.legend(frameon=False, fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
    fig.tight_layout()
    path = Path(path)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path

In [ ]:
%%writefile /content/lobgen/src/lobgen/model.py
"""Autoregressive transformer over simulator actions, with spike-aware output heads.

Input token for event i = the action as applied (type, side, price field, queue position,
size, dt) + the book right after it (spread, 10 levels of volume and price gaps, imbalance,
mid move) + time of day. Everything is relative to the book, so one model serves any stock.

Output for event i+1, factorised so that every field sees the fields chosen before it:
    (type, side)  ->  price field  ->  queue position  ->  size  ->  dt
(type, side) is a 10-way softmax. The four integer fields share one generic head, SpikeHead:

    P(x) = sum_j w_j [x == a_j]  +  w_c * sum_m pi_m * Logistic(T(x) bin | mu_m, s_m)

  * atoms a_j: every value holding >= `atom_share` of the training data for that field (up to
    `max_atoms`), picked automatically - e.g. dt = 0 ns (17-32% of events), size = 100 shares
    (40-78% of submits), price field = 0 (at the touch / level 0). Nothing is hand-chosen per
    field; the same rule runs on every field.
  * continuous part: mixture of logistics on T(x) = sign(x) log(1 + |x|), integrated over each
    integer's bin, so the likelihood is exact for integer data spanning nine orders of
    magnitude (dt from 1 ns to minutes).
A continuous density alone smears a point mass over its neighbours (the problem "atom
smoothing" works around in Jane Street's post); atoms make each spike exact. use_atoms=False
removes them (ablation).

Masks: queue position is only scored for cancels / deletes / executions; size is not scored
for deletes (the simulator sets it to the order's size).
"""

from __future__ import annotations

import math
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .data import Day, Generated, Generator, Window
from .features import (BOOK_COLS, FLOW_BOOK, N_ALL, N_BASE, N_CAT, N_CONT, N_FULL,  # noqa: F401
                       N_TS, OWN_COLS, P_HI, P_LO, Q_MAX, REGIME_BOOK, SIZE_BUCKETS, SPREAD_MAX,
                       BatchState,
                       FeatState, N_REG, HALF_LIVES, size_bucket, token_features, ts_decode,
                       ts_index)
from .sim import (CANCEL, DELETE, DT, EXEC, PRICE, QPOS, SIDE, SIZE, SUBMIT, TICK, TYPE,
                  SimBook, Simulator)

FIELDS = ("ts", "p", "q", "size", "dt")


# ----------------------------------------------------------------------------- atoms

def choose_atoms(values: np.ndarray, min_share: float, max_atoms: int) -> list[int]:
    v, c = np.unique(np.asarray(values), return_counts=True)
    share = c / c.sum()
    order = np.argsort(-share)
    return [int(v[k]) for k in order[:max_atoms] if share[k] >= min_share]


# ----------------------------------------------------------------------------- heads

def _T(x: torch.Tensor) -> torch.Tensor:
    return torch.sign(x) * torch.log1p(torch.abs(x))


def _T_inv(u: torch.Tensor) -> torch.Tensor:
    return torch.sign(u) * torch.expm1(torch.abs(u))


class SpikeHead(nn.Module):
    """Atoms + discretised mixture of logistics on sign(x)log(1+|x|), for one integer field."""

    def __init__(self, d: int, atoms: list[int], n_mix: int = 8, lo: int | None = None,
                 hidden: int = 256) -> None:
        super().__init__()
        self.register_buffer("atoms", torch.tensor(atoms, dtype=torch.long), persistent=False)
        self.K, self.M, self.lo = len(atoms), n_mix, lo
        self.net = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, hidden), nn.GELU(),
                                 nn.Linear(hidden, self.K + 1 + 3 * n_mix))

    def _params(self, x: torch.Tensor):
        o = self.net(x).float()
        K, M = self.K, self.M
        comp = F.log_softmax(o[..., :K + 1], -1)
        mix = F.log_softmax(o[..., K + 1:K + 1 + M], -1)
        mu = o[..., K + 1 + M:K + 1 + 2 * M]
        log_s = o[..., K + 1 + 2 * M:].clamp(-7.0, 3.0)
        return comp, mix, mu, log_s

    def log_prob(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        comp, mix, mu, log_s = self._params(x)
        yf = y.float()
        s = log_s.exp()
        hi = (_T(yf + 0.5).unsqueeze(-1) - mu) / s
        lo = (_T(yf - 0.5).unsqueeze(-1) - mu) / s
        at_lo = torch.zeros_like(y, dtype=torch.bool) if self.lo is None else (y <= self.lo)
        log_hi = -F.softplus(-hi)                                # log sigmoid(hi)
        delta = torch.sigmoid(hi) - torch.sigmoid(lo)
        mid = (_T(yf).unsqueeze(-1) - mu) / s
        ay = yf.abs()
        width = torch.where(ay == 0, torch.full_like(ay, 2 * math.log1p(0.5)),
                            torch.log1p(1.0 / (ay + 0.5)))       # bin width in T-space, stable
        log_mid = mid - log_s - 2 * F.softplus(mid) + width.log().unsqueeze(-1)
        lp = torch.where(delta > 1e-5, delta.clamp(min=1e-12).log(), log_mid)
        lp = torch.where(at_lo.unsqueeze(-1), log_hi, lp)        # bin extends to -inf at the bound
        cont = torch.logsumexp(mix + lp, -1) + comp[..., self.K]
        if self.K == 0:
            return cont
        match = y.unsqueeze(-1) == self.atoms
        atom = torch.where(match, comp[..., :self.K], torch.full_like(comp[..., :self.K], -1e30))
        return torch.logsumexp(torch.stack([torch.logsumexp(atom, -1), cont], -1), -1)

    @torch.no_grad()
    def sample(self, x: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
        comp, mix, mu, log_s = self._params(x)
        dev = x.device
        c = torch.distributions.Categorical(logits=comp / temperature).sample().cpu()
        m = torch.distributions.Categorical(logits=mix / temperature).sample()
        # float64 on CPU: MPS has no float64, and dt spans 1 ns .. minutes.
        mu_m = mu.gather(-1, m.unsqueeze(-1)).squeeze(-1).cpu().double()
        s_m = log_s.gather(-1, m.unsqueeze(-1)).squeeze(-1).cpu().double().exp()
        u01 = torch.rand(mu_m.shape, dtype=torch.float64).clamp(1e-12, 1 - 1e-12)
        u = (mu_m + s_m * temperature * (u01.log() - (-u01).log1p())).clamp(-30, 30)
        val = torch.round(_T_inv(u))
        if self.lo is not None:
            val = val.clamp(min=self.lo)
        val = val.clamp(-1e12, 1e12).long()
        if self.K:
            atoms = self.atoms.cpu()
            val = torch.where(c < self.K, atoms[c.clamp(max=self.K - 1)], val)
        return val.to(dev)


# ----------------------------------------------------------------------------- model

@dataclass
class ModelConfig:
    d_model: int = 256
    n_layers: int = 6
    n_heads: int = 8
    ctx: int = 256
    dropout: float = 0.1
    n_mix: int = 8
    use_atoms: bool = True
    use_book: bool = True          # ablation: False zeroes every book feature in the input
    atom_share: float = 0.005
    max_atoms: int = 32
    atoms: dict = field(default_factory=dict)  # field -> list of atom values (fit on train data)
    encoding: tuple = (0, False)   # action encoding trained on: (pointer ranks, whole_fill)
    regime: bool = False           # day-so-far input features (features.py); False = v1 inputs
    flow: bool = False             # order-flow memory inputs (features.py, v3); needs regime
    market: bool = False           # clock-time own + ETF inputs (market.py, v4); needs flow

    def __post_init__(self) -> None:
        if self.flow and not self.regime:
            raise ValueError("flow features come after the day-so-far ones: set regime=True")
        if self.market and not self.flow:
            raise ValueError("market features come after the flow ones: set flow=True")

    @property
    def n_cont(self) -> int:
        if self.market:
            return N_FULL
        return N_ALL if self.flow else N_CONT if self.regime else N_BASE

    @property
    def size_lo(self) -> int:
        return 0 if self.encoding[1] else 1      # whole_fill: 0 = the whole resting order


def _book_cols(n_cont: int) -> list[int]:
    cols = list(BOOK_COLS)
    if n_cont > N_BASE:  # day-so-far spread and mid-move averages are book information too
        cols += [N_BASE + h * N_REG + k for h in range(len(HALF_LIVES)) for k in REGIME_BOOK]
    if n_cont > N_CONT:  # the mid's distance from its moving average
        cols += FLOW_BOOK
    if n_cont > N_ALL:   # the mid's clock-time moves
        cols += OWN_COLS
    return cols


class LOBModel(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        self.e_ts = nn.Embedding(N_TS, d)
        self.e_p = nn.Embedding(P_LO + P_HI + 1, d)
        self.e_q = nn.Embedding(Q_MAX + 1, d)
        self.e_spread = nn.Embedding(SPREAD_MAX + 1, d)
        self.e_cont = nn.Linear(cfg.n_cont, d)
        self._book = _book_cols(cfg.n_cont)
        self.pos = nn.Parameter(torch.zeros(cfg.ctx, d))
        nn.init.normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(d, cfg.n_heads, 4 * d, cfg.dropout, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.trunk_layers = nn.TransformerEncoder(layer, cfg.n_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.o_ts = nn.Embedding(N_TS, d)
        self.o_p = nn.Embedding(P_LO + P_HI + 1, d)
        self.o_q = nn.Embedding(Q_MAX + 1, d)
        self.o_size = nn.Embedding(SIZE_BUCKETS, d)
        self.head_ts = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(), nn.Linear(d, N_TS))
        A = cfg.atoms if cfg.use_atoms else {}
        self.head_p = SpikeHead(d, A.get("p", []), cfg.n_mix, lo=None)
        self.head_q = SpikeHead(d, A.get("q", []), cfg.n_mix, lo=0)
        self.head_size = SpikeHead(d, A.get("size", []), cfg.n_mix, lo=cfg.size_lo)
        self.head_dt = SpikeHead(d, A.get("dt", []), cfg.n_mix, lo=0)
        mask = torch.triu(torch.full((cfg.ctx, cfg.ctx), float("-inf")), diagonal=1)
        self.register_buffer("causal", mask, persistent=False)

    def trunk(self, cat: torch.Tensor, cont: torch.Tensor) -> torch.Tensor:
        cont = cont[..., :self.cfg.n_cont]
        if not self.cfg.use_book:
            cont = cont.clone()
            cont[..., self._book] = 0
            cat = cat.clone()
            cat[..., 3] = 0
        T = cat.shape[1]
        x = (self.e_ts(cat[..., 0]) + self.e_p(cat[..., 1]) + self.e_q(cat[..., 2])
             + self.e_spread(cat[..., 3]) + self.e_cont(cont.float()) + self.pos[:T])
        x = self.trunk_layers(x, mask=self.causal[:T, :T], is_causal=True)
        return self.norm(x)

    def _cond(self, h, ts, p, q, size):
        c1 = h + self.o_ts(ts)
        c2 = c1 + self.o_p(torch.clamp(p, -P_LO, P_HI) + P_LO)
        c3 = c2 + self.o_q(torch.clamp(q, 0, Q_MAX))
        c4 = c3 + self.o_size(size_bucket(size))
        return c1, c2, c3, c4

    def nll(self, cat, cont, target) -> dict[str, torch.Tensor]:
        """Mean negative log-likelihood (nats) per field; target = next actions [B, T, 6]."""
        h = self.trunk(cat, cont)
        typ, side = target[..., TYPE], target[..., SIDE]
        ts = (typ.clamp(1, 5) - 1) * 2 + (side > 0).long()
        p, q, size, dt = target[..., PRICE], target[..., QPOS], target[..., SIZE], target[..., DT]
        c1, c2, c3, c4 = self._cond(h, ts, p, q, size)
        out = {"ts": F.cross_entropy(self.head_ts(h).float().flatten(0, 1), ts.flatten())}
        out["p"] = -self.head_p.log_prob(c1, p).mean()
        mq = (typ == EXEC) | (((typ == CANCEL) | (typ == DELETE)) & (p >= 0))
        out["q"] = -(self.head_q.log_prob(c2, q) * mq).sum() / mq.sum().clamp(min=1)
        ms = typ != DELETE
        out["size"] = -(self.head_size.log_prob(c3, size) * ms).sum() / ms.sum().clamp(min=1)
        out["dt"] = -self.head_dt.log_prob(c4, dt).mean()
        # nats per event: masked fields weighted by how often they are scored
        out["event"] = (out["ts"] + out["p"] + out["q"] * mq.float().mean()
                        + out["size"] * ms.float().mean() + out["dt"])
        return out

    @torch.no_grad()
    def sample_next(self, h: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
        """h [B, d] -> actions [B, 6] (dt, type, side, price field, qpos, size)."""
        ts = torch.distributions.Categorical(logits=self.head_ts(h).float() / temperature).sample()
        c1 = h + self.o_ts(ts)
        p = self.head_p.sample(c1, temperature)
        c2 = c1 + self.o_p(torch.clamp(p, -P_LO, P_HI) + P_LO)
        q = self.head_q.sample(c2, temperature)
        typ = ts // 2 + 1
        level_q = (typ == EXEC) | (((typ == CANCEL) | (typ == DELETE)) & (p >= 0))
        q = torch.where(level_q, q, torch.zeros_like(q))
        c3 = c2 + self.o_q(torch.clamp(q, 0, Q_MAX))
        size = self.head_size.sample(c3, temperature)
        c4 = c3 + self.o_size(size_bucket(size))
        dt = self.head_dt.sample(c4, temperature)
        side = torch.where(ts % 2 == 1, 1, -1)
        return torch.stack([dt, typ, side, p, q, size], -1)


def fit_atoms(actions: np.ndarray, share: float, max_atoms: int) -> dict[str, list[int]]:
    typ = actions[:, TYPE]
    return {
        "p": choose_atoms(actions[:, PRICE], share, max_atoms),
        "q": choose_atoms(actions[np.isin(typ, [CANCEL, DELETE, EXEC]), QPOS], share, max_atoms),
        "size": choose_atoms(actions[typ != DELETE, SIZE], share, max_atoms),
        "dt": choose_atoms(actions[:, DT], share, max_atoms),
    }


# ----------------------------------------------------------------------------- data

class EventData:
    """Features + next-action targets for a set of days; samples random ctx-long windows.

    Each day contributes the rows between open+margin and close-margin. Sampling picks a ticker
    uniformly, then a day in proportion to its length, then a uniform start.
    """

    def __init__(self, parts: list[tuple], ctx: int, encoding: tuple = (0, False)) -> None:
        self.ctx = ctx
        self.parts = parts  # (ticker, cat int16, cont float16, actions int64)
        self.tickers = sorted({p[0] for p in parts})
        self.encoding = tuple(encoding)  # action encoding of every part

    @staticmethod
    def featurize(d: Day, margin_s: int = 300, market=None, out_dir=None,
                  chunk: int = 1_000_000) -> tuple:
        """(ticker, cat int16, cont float16, actions int64) for the modeled rows of a day.
        out_dir: write the arrays to .npy files there and return them memory-mapped (read-only),
        so the training set does not have to fit in RAM (all of v4's inputs are ~9 GB). The
        values are the same either way; chunking only bounds the float64 temporaries."""
        rows = np.flatnonzero(d.model_rows(margin_s))
        if len(rows) and rows[-1] - rows[0] + 1 != len(rows):
            raise ValueError(f"{d.ticker} {d.date}: non-modeled rows inside the session")
        r0, r1 = int(rows[0]), int(rows[-1]) + 1
        n = r1 - r0
        names = {k: Path(out_dir) / f"{d.ticker}_{d.date}_{k}.npy" for k in ("cat", "cont", "act")} \
            if out_dir is not None else None

        def alloc(k, dtype, shape):  # filled in place: no concatenation copy at the end
            if names is None:
                return np.empty(shape, dtype)
            return np.lib.format.open_memmap(names[k], mode="w+", dtype=dtype, shape=shape)

        cat_out = alloc("cat", np.int16, (n, N_CAT))
        cont_out = alloc("cont", np.float16, (n, N_FULL))
        act_out = alloc("act", d.actions.dtype, (n, d.actions.shape[1]))
        act_out[:] = d.actions[r0:r1]
        # The stream starts at the first row of the day, so the day-so-far features of row r0
        # already include the open; rows before r0 are featurised and dropped.
        st = FeatState()
        for s in range(0, r1, chunk):
            e = min(s + chunk, r1)
            cat, cont, st = token_features(d.actions[s:e], d.msgs[s:e], d.book[s:e], st, market)
            k = max(r0 - s, 0)
            if k < e - s:
                o = s + k - r0
                cat_out[o:o + e - s - k] = cat[k:].astype(np.int16)
                cont_out[o:o + e - s - k] = cont[k:].astype(np.float16)
            del cat, cont
        if names is None:
            return d.ticker, cat_out, cont_out, act_out
        for a in (cat_out, cont_out, act_out):
            a.flush()
        del cat_out, cont_out, act_out
        return (d.ticker, *(np.load(names[k], mmap_mode="r") for k in ("cat", "cont", "act")))

    @classmethod
    def from_npz(cls, paths, ctx: int, margin_s: int = 300, market_dir=None,
                 out_dir=None) -> "EventData":
        """Load, featurise and free one day at a time (a GOOG day's book array is ~3 GB).
        market_dir: also the ETF inputs, from market/<ETF>_<date>.npz files there.
        out_dir: keep the features in memory-mapped files there instead of RAM (featurize)."""
        from .market import Market
        parts, encodings, markets = [], set(), {}
        for p in paths:
            d = Day.load(p)
            mk = None
            if market_dir is not None:
                mk = markets.get(d.date) or markets.setdefault(d.date, Market.load(d.date, market_dir))
            parts.append(cls.featurize(d, margin_s, mk, out_dir))
            encodings.add(d.encoding)
            del d
        if len(encodings) != 1:
            raise ValueError(f"days use different action encodings {encodings}: rerun build_actions.py")
        return cls(parts, ctx, encodings.pop())

    def all_actions(self) -> np.ndarray:
        return np.concatenate([p[3] for p in self.parts])

    def n_events(self) -> int:
        return sum(len(p[1]) for p in self.parts)

    def batch(self, B: int, rng: np.random.Generator, device) -> tuple[torch.Tensor, ...]:
        cats, conts, tgts = [], [], []
        for _ in range(B):
            tk = self.tickers[rng.integers(len(self.tickers))]
            cand = [p for p in self.parts if p[0] == tk]
            w = np.array([len(p[1]) for p in cand], dtype=float)
            _, cat, cont, act = cand[rng.choice(len(cand), p=w / w.sum())]
            s = int(rng.integers(0, len(cat) - self.ctx - 1))
            cats.append(cat[s:s + self.ctx])
            conts.append(cont[s:s + self.ctx])
            tgts.append(act[s + 1:s + 1 + self.ctx])
        f = lambda x, dt: torch.from_numpy(np.stack(x)).to(device=device, dtype=dt)  # noqa: E731
        return f(cats, torch.long), f(conts, torch.float32), f(tgts, torch.long)


# ----------------------------------------------------------------------------- training

def pick_device(device: str | None = None) -> torch.device:
    if device:
        return torch.device(device)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


@torch.no_grad()
def evaluate(model: LOBModel, data: EventData, n_batches: int, B: int, device, seed: int = 123) -> dict:
    model.eval()
    rng = np.random.default_rng(seed)
    acc: dict[str, float] = {}
    for _ in range(n_batches):
        out = model.nll(*data.batch(B, rng, device))
        for k, v in out.items():
            acc[k] = acc.get(k, 0.0) + float(v) / n_batches
    model.train()
    return acc


def train(cfg: ModelConfig, train_data: EventData, val_data: EventData, out_path: str | Path,
          steps: int = 20_000, batch: int = 64, lr: float = 3e-4, warmup: int = 500,
          eval_every: int = 500, eval_batches: int = 20, device=None, seed: int = 0,
          resume: bool = True, log=print) -> dict:
    """Writes <out>.pt (best validation step), <out>_last.pt (final weights) and, at every
    evaluation, <out>_state.pt: everything needed to continue (weights, optimizer, schedule,
    scaler, random states). If that file exists and is from the same run, training continues
    from it, so an interrupted run loses at most `eval_every` steps; on CPU the result is
    bit-identical to an uninterrupted run. The state file is removed when the run finishes."""
    device = pick_device(device)
    out_path = Path(out_path)
    state_path = out_path.with_name(out_path.stem + "_state" + out_path.suffix)
    torch.manual_seed(seed)
    model = LOBModel(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warmup) * (
        0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps))))
    use_amp = device.type == "cuda"
    amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)
    rng = np.random.default_rng(seed)
    best, history, start, minutes, val = float("inf"), [], 1, 0.0, None
    run = {"cfg": asdict(cfg), "steps": steps, "batch": batch, "lr": lr, "warmup": warmup,
           "seed": seed, "train_events": train_data.n_events(), "val_events": val_data.n_events()}
    if resume and state_path.exists():
        st = torch.load(state_path, map_location="cpu", weights_only=False)
        if st["run"] != run:
            log(f"{state_path.name} is from a different run (other settings or data): starting over")
        else:
            model.load_state_dict(st["model"])
            opt.load_state_dict(st["opt"])
            sched.load_state_dict(st["sched"])
            scaler.load_state_dict(st["scaler"])
            rng.bit_generator.state = st["rng"]
            torch.set_rng_state(st["torch_rng"])
            if device.type == "cuda" and st["cuda_rng"] is not None \
                    and len(st["cuda_rng"]) == torch.cuda.device_count():
                torch.cuda.set_rng_state_all(st["cuda_rng"])
            best, history, val = st["best"], st["history"], st["val"]
            start, minutes = st["step"] + 1, st["minutes"]
            log(f"resumed from {state_path.name} at step {st['step']}")
    t0 = time.time() - 60 * minutes
    n_params = sum(p.numel() for p in model.parameters())
    log(f"params {n_params / 1e6:.2f}M  train events {train_data.n_events():,}  "
        f"val events {val_data.n_events():,}  device {device}")
    for step in range(start, steps + 1):
        cat, cont, tgt = train_data.batch(batch, rng, device)
        with torch.autocast(device.type, dtype=amp_dtype, enabled=use_amp):
            loss = model.nll(cat, cont, tgt)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss["event"]).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        if step % eval_every == 0 or step == steps:
            val = evaluate(model, val_data, eval_batches, batch, device)
            rec = {"step": step, "train_event": loss["event"].detach().item(),
                   "minutes": round((time.time() - t0) / 60, 2)}
            rec |= {f"val_{k}": round(v, 4) for k, v in val.items()}
            history.append(rec)
            log(" ".join(f"{k}={v:.4g}" if isinstance(v, float) else f"{k}={v}" for k, v in rec.items()))
            if val["event"] < best:
                best = val["event"]
                save(model, out_path, {"step": step, "val": val, "history": history})
            _atomic_save({
                "run": run, "step": step, "minutes": (time.time() - t0) / 60, "best": best,
                "val": val, "history": history, "model": model.state_dict(),
                "opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                "rng": rng.bit_generator.state, "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
            }, state_path)
    # Also keep the final weights: price NLL on the fixed validation batches swings by ~0.15
    # nats between evaluations (a few far-from-touch prices), so "best" can be an early step.
    save(model, out_path.with_name(out_path.stem + "_last" + out_path.suffix),
         {"step": steps, "val": val, "history": history})
    state_path.unlink(missing_ok=True)
    return {"best_val_event": best, "history": history}


def _atomic_save(obj, path: str | Path) -> None:
    """Write to a temporary file, then rename: an interruption never leaves a half-written file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(path)


def save(model: LOBModel, path: str | Path, extra: dict) -> None:
    _atomic_save({"cfg": asdict(model.cfg), "state": model.state_dict(), **extra}, path)


def load(path: str | Path, device=None) -> LOBModel:
    ck = torch.load(path, map_location="cpu", weights_only=False)
    model = LOBModel(ModelConfig(**ck["cfg"]))
    model.load_state_dict(ck["state"])
    return model.to(pick_device(device)).eval()


# ----------------------------------------------------------------------------- generation

def attach_market(g, date: str, market_dir: str = "data/market"):
    """Give a market model (cfg.market) the ETF quotes of `date`; other generators unchanged."""
    if isinstance(g, ModelGenerator) and g.model.cfg.market:
        from .market import Market
        g.market = Market.load(date, market_dir)
    return g


class ModelGenerator(Generator):
    """Batched rollouts: one forward pass per event for a batch of windows, each window with
    its own simulator. Actions the simulator cannot apply (cancel on an empty side) are
    redrawn from the same distribution."""

    def __init__(self, model: LOBModel, name: str = "model", temperature: float = 1.0,
                 batch: int = 250, max_redraws: int = 50, market=None) -> None:
        self.model, self.name, self.temperature = model, name, temperature
        self.batch, self.max_redraws = batch, max_redraws
        self.market = market  # market.Market for windows without their own (market models)

    @classmethod
    def load(cls, path, device=None, name: str = "model", **kw) -> "ModelGenerator":
        return cls(load(path, device), name=name, **kw)

    @torch.no_grad()
    def generate(self, windows: list[Window], n_events: int, seed: int = 0) -> list[Generated]:
        out: list[Generated] = []
        for k in range(0, len(windows), self.batch):
            out += self._chunk(windows[k:k + self.batch], n_events, seed + k)
        return out

    def _chunk(self, ws: list[Window], n_events: int, seed: int) -> list[Generated]:
        model, ctx = self.model, self.model.cfg.ctx
        dev = next(model.parameters()).device
        torch.manual_seed(seed)
        B = len(ws)
        bad = {w.encoding for w in ws} - {tuple(model.cfg.encoding)}
        if bad:
            raise ValueError(f"{self.name} was trained on action encoding {model.cfg.encoding} "
                             f"but the windows use {bad}: rebuild the actions to match")
        cats, conts, states, sims, mks = [], [], [], [], []
        for w in ws:
            if len(w.cond_actions) < ctx:
                raise ValueError(f"window {w.i}: {len(w.cond_actions)} context events < ctx {ctx}")
            if w.feat_state is None:
                raise ValueError(f"window {w.i} has no feature state: make_windows(feat_states=True)")
            mk = (w.market if w.market is not None else self.market) if model.cfg.market else None
            if model.cfg.market and mk is None:
                raise ValueError(f"{self.name} reads the market: set window.market or "
                                 "ModelGenerator.market (market.Market of the day)")
            mks.append(mk)
            # Features of the whole context, continuing the day's stream exactly as in training.
            c, x, st = token_features(w.cond_actions, w.cond_msgs, w.cond_book, w.feat_state, mk)
            cats.append(c[-ctx:])
            conts.append(x[-ctx:])
            states.append(st)
            sims.append(Simulator(SimBook.from_orders(w.state), w.t_prev))
        stream = BatchState(states, mks)
        cat = torch.from_numpy(np.stack(cats)).to(dev)
        cont = torch.from_numpy(np.stack(conts)).to(dev)
        msgs = [[] for _ in range(B)]
        books = [[] for _ in range(B)]
        acts_out = np.empty((B, n_events, 6), dtype=np.int64)
        for step in range(n_events):
            h = model.trunk(cat, cont)[:, -1]
            acts = model.sample_next(h, self.temperature).cpu().numpy()
            applied = np.empty((B, 6), dtype=np.int64)
            snaps = np.empty((B, 4 * sims[0].levels), dtype=np.int64)
            new_msgs = np.empty((B, 6), dtype=np.int64)
            for b in range(B):
                res, tries = sims[b].step(*acts[b].tolist()), 0
                while res is None:
                    tries += 1
                    if tries > self.max_redraws:
                        raise RuntimeError(f"{self.name}: window {ws[b].i} cannot place an action")
                    a = model.sample_next(h[b:b + 1], self.temperature)[0].cpu().numpy()
                    res = sims[b].step(*a.tolist())
                msg, snap = res
                msgs[b].append(msg)
                books[b].append(snap)
                applied[b] = sims[b].last_action
                acts_out[b, step] = applied[b]
                snaps[b] = snap
                new_msgs[b] = msg
            # One vectorised step for the B new rows: each row continues its own window's stream.
            new_c, new_x = stream.step(applied, new_msgs, snaps)
            cat = torch.cat([cat[:, 1:], torch.from_numpy(new_c).to(dev)[:, None]], 1)
            cont = torch.cat([cont[:, 1:], torch.from_numpy(new_x).to(dev)[:, None]], 1)
        return [Generated(np.array(msgs[b], dtype=np.int64), np.array(books[b], dtype=np.int64),
                          sims[b].repairs, sims[b].skipped, acts_out[b]) for b in range(B)]

In [ ]:
%%writefile /content/lobgen/src/lobgen/shock.py
"""Shock test: after a large market order, does the book react the way the real one did?

This is the question a trader asks of a simulator ("if I trade into it, does it react like the
market?"), and the one the detective cannot answer: the detective scores runs from ordinary
moments, while the hard part of market behaviour is the reaction to an aggressive order.

Two experiments, both measured in trader units (ticks, shares, milliseconds):

  real shocks   Every large aggressive order of the test day (top 1% by size, the first of any
                cluster) is left in the context, and each generator continues from the real
                book right after it. Its continuation is compared with what really followed:
                the mid-price path (signed, so + means the price kept moving in the order's
                direction), the spread, the depth refilling on the side that was hit, the
                follow-on aggressive volume, and how fast events arrive.
  injection     A market order of chosen size is inserted at ordinary moments of the day,
                where no such order really happened (a counterfactual: there is no real
                continuation). The model's impact = run with the order minus run without it,
                on the same windows. Real orders of similar size, binned, give the reference
                impact-vs-size curve (computed on every aggressive order of the day).

A market order shows up in the data as several visible executions (LOBSTER type 4) with the
same timestamp against resting orders on one side: one group = one aggressive order. Its
direction is the opposite of the resting side (LOBSTER direction -1 = a sell order was hit =
buyer-initiated).
"""

from __future__ import annotations

import warnings
from dataclasses import dataclass, replace

import numpy as np

from .data import Day, Window, windows_at
from .sim import BUY, EXEC, SELL, TICK, SimBook, Simulator

EVENTS = (1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000)       # events after the shock
TIMES_NS = (1_000_000, 10_000_000, 100_000_000, 1_000_000_000, 10_000_000_000)  # 1 ms .. 10 s
NS = 1_000_000_000


@dataclass
class Shock:
    first: int    # day row of the first execution
    last: int     # day row of the last execution
    t: int        # ns since midnight
    sign: int     # +1 buyer-initiated (asks were hit), -1 seller-initiated
    size: int     # shares
    levels: int   # how many ticks the same-side best moved during the order


def aggressor_groups(msgs: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """(first row, last row) of every aggressive order: runs of consecutive visible executions
    with the same timestamp against the same side."""
    m = np.asarray(msgs)
    ex = m[:, 1] == EXEC
    cont = np.zeros(len(m), dtype=bool)
    cont[1:] = ex[1:] & ex[:-1] & (m[1:, 0] == m[:-1, 0]) & (m[1:, 5] == m[:-1, 5])
    first = np.flatnonzero(ex & ~cont)
    is_last = ex & ~np.append(cont[1:], False)
    return first, np.flatnonzero(is_last)


def all_orders(day: Day) -> list[Shock]:
    """Every aggressive order of the day (no filtering)."""
    m, b = day.msgs, day.book
    first, last = aggressor_groups(m)
    csum = np.concatenate([[0], np.cumsum(m[:, 3])])
    sizes = csum[last + 1] - csum[first]
    out = []
    for f, l, s in zip(first.tolist(), last.tolist(), sizes.tolist()):
        sign = -int(m[f, 5])
        col = 0 if sign > 0 else 2                    # best ask moves when asks are hit
        before = b[f - 1, col] if f > 0 else b[f, col]
        moved = abs(int(b[l, col] - before)) // TICK if b[l, col + 1] > 0 else -1  # -1: side emptied
        out.append(Shock(f, l, int(m[f, 0]), sign, int(s), moved))
    return out


def select_shocks(day: Day, quantile: float = 0.99, min_size: int | None = None,
                  cond_len: int = 500, gen_len: int = 2000, gap: int = 1000,
                  margin_s: int = 300, max_n: int | None = None, seed: int = 0) -> list[Shock]:
    """Large aggressive orders: size >= the day's `quantile` of order sizes (or >= min_size),
    away from the open and close, with room for the context and the continuation. Of orders
    less than `gap` events apart only the first is kept: the later ones are part of the real
    reaction a generator has to produce."""
    orders = all_orders(day)
    sizes = np.array([o.size for o in orders])
    thr = min_size if min_size is not None else float(np.quantile(sizes, quantile))
    t0, t1 = (34_200 + margin_s) * NS, (57_600 - margin_s) * NS
    kept, last_row = [], -10 ** 12
    for o in orders:
        if o.size < thr:
            continue
        if o.first - last_row < gap:
            last_row = o.last  # extend the cluster
            continue
        last_row = o.last
        if t0 <= o.t < t1 and o.first >= cond_len and o.last + 1 + gen_len <= len(day.msgs):
            kept.append(o)
    if max_n is not None and len(kept) > max_n:
        idx = np.sort(np.random.default_rng(seed).choice(len(kept), max_n, replace=False))
        kept = [kept[i] for i in idx]
    return kept


def shock_windows(day: Day, shocks: list[Shock], cond_len: int = 500,
                  gen_len: int = 2000) -> list[Window]:
    """Context ends right after each shock's last execution, so the shock is the last thing a
    generator sees."""
    return windows_at(day, [s.last + 1 for s in shocks], cond_len, gen_len)


def inject(w: Window, sign: int, size: int, dt_ns: int = 1_000) -> tuple[Window, dict]:
    """Insert a market order of `size` shares (sign +1 buy, -1 sell) right after the window's
    context: it executes against the opposite side in price-time order, like a real
    marketable order, and becomes the last events of the context. Returns the new window and
    the reference values for measure()."""
    if not w.encoding[1]:
        raise ValueError("injection writes whole fills as size 0: needs the whole_fill encoding")
    sim = Simulator(SimBook.from_orders(w.state), w.t_prev)
    resting = SELL if sign > 0 else BUY
    bk = sim.book
    pre = w.cond_book[-1]
    msgs, books, acts = [], [], []
    left = int(size)
    while left > 0 and bk.n_levels(resting):
        price = bk.best(resting)
        ref = bk.ref_at(resting, price, 0, from_front=True)
        have = bk.side[resting][price].q[ref]
        take = min(have, left)
        msg, snap = sim.step(dt_ns if not msgs else 0, EXEC, resting, 0, 0,
                             0 if take == have else take)
        msgs.append(msg)
        books.append(snap)
        acts.append(sim.last_action)
        left -= take
    nw = replace(w, state=sim.book.to_orders(), t_prev=sim.t,
                 cond_msgs=np.vstack([w.cond_msgs, np.array(msgs, dtype=np.int64)]),
                 cond_book=np.vstack([w.cond_book, np.array(books, dtype=np.int64)]),
                 cond_actions=np.vstack([w.cond_actions, np.array(acts, dtype=np.int64)]))
    return nw, ref_point(pre, w.t_prev + dt_ns, sign, size - left)


def _mid_ticks(row: np.ndarray) -> float:
    if row[1] > 0 and row[3] > 0:
        return (row[0] + row[2]) / (2 * TICK)
    return np.nan


def ref_point(book_before: np.ndarray, t: int, sign: int, size: int) -> dict:
    """What a response is measured against: the book just before the order, its time and side."""
    col = 1 if sign > 0 else 3                      # volume at the best of the side that is hit
    return {"t": int(t), "sign": int(sign), "size": int(size),
            "mid": _mid_ticks(book_before), "depth": int(book_before[col])}


def shock_ref(day: Day, s: Shock) -> dict:
    return ref_point(day.book[s.first - 1], s.t, s.sign, s.size)


def _ffill_mid(book: np.ndarray, start: float) -> np.ndarray:
    two = (book[:, 1] > 0) & (book[:, 3] > 0)
    mid = np.where(two, (book[:, 0] + book[:, 2]) / (2 * TICK), np.nan)
    full = np.concatenate([[start], mid])
    idx = np.where(~np.isnan(full), np.arange(len(full)), 0)
    np.maximum.accumulate(idx, out=idx)
    return full[idx]                                  # [N+1]: after 0..N continuation events


def measure(msgs: np.ndarray, book: np.ndarray, last_ctx_book: np.ndarray, ref: dict,
            events=EVENTS, times=TIMES_NS) -> dict[str, np.ndarray]:
    """Response of one continuation (real or generated) to the order described by `ref`.

    move_*    signed mid change from just before the order, ticks (+ = continued in the
              order's direction); after k events (move_ev) or t after the order (move_t; nan
              when the run ends earlier, i.e. that horizon is not covered)
    spread_*  bid-ask spread, ticks
    depth_ev  visible volume at the best of the side that was hit, shares
    flow_ev   follow-on aggressive volume in the order's direction, shares (buys - sells,
              signed by the order)
    ms_ev     milliseconds from the order to the k-th event (how fast the market reacts)
    """
    m, b = np.asarray(msgs), np.asarray(book)
    s = ref["sign"]
    n = len(m)
    mid = _ffill_mid(b, _mid_ticks(last_ctx_book))
    spread = np.concatenate([[last_ctx_book[0] - last_ctx_book[2]], b[:, 0] - b[:, 2]]) / TICK
    two = np.concatenate([[last_ctx_book[1] > 0 and last_ctx_book[3] > 0],
                          (b[:, 1] > 0) & (b[:, 3] > 0)])
    spread = np.where(two, spread, np.nan)
    col = 1 if s > 0 else 3
    depth = np.concatenate([[last_ctx_book[col]], b[:, col]]).astype(float)
    ex = m[:, 1] == EXEC
    flow = np.concatenate([[0], np.cumsum(np.where(ex, -m[:, 5] * m[:, 3], 0))]) * s
    k = np.asarray(events)
    ok = k <= n
    kk = np.minimum(k, n)
    t_ev = np.concatenate([[ref["t"]], m[:, 0]])
    out = {
        "move_ev": np.where(ok, s * (mid[kk] - ref["mid"]), np.nan),
        "spread_ev": np.where(ok, spread[kk], np.nan),
        "depth_ev": np.where(ok, depth[kk], np.nan),
        "flow_ev": np.where(ok, flow[kk], np.nan),
        "ms_ev": np.where(ok, (t_ev[kk] - ref["t"]) / 1e6, np.nan),
    }
    tq = ref["t"] + np.asarray(times, dtype=np.int64)
    idx = np.searchsorted(m[:, 0], tq, side="right")   # number of events at or before t
    cov = t_ev[-1] >= tq                                # the run reaches t: state there is known
    out["move_t"] = np.where(cov, s * (mid[idx] - ref["mid"]), np.nan)
    out["spread_t"] = np.where(cov, spread[idx], np.nan)
    return out


def stack(rows: list[dict]) -> dict[str, np.ndarray]:
    return {k: np.stack([r[k] for r in rows]) for k in rows[0]}


def summarize(x: np.ndarray, n_boot: int = 1000, seed: int = 0, min_cover: float = 0.95) -> dict:
    """Mean over shocks with a 95% bootstrap interval, per horizon (columns). x: [n_shocks, H]
    (already averaged over samples of the same shock). A horizon covered by fewer than
    `min_cover` of the runs is reported as nan: the runs that reach it are a biased subset."""
    x = np.asarray(x, dtype=float)
    cover = np.mean(~np.isnan(x), 0)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), (n_boot, len(x)))
    with warnings.catch_warnings():   # all-nan columns are reported as nan below
        warnings.simplefilter("ignore", RuntimeWarning)
        mean = np.nanmean(x, 0)
        boots = np.stack([np.nanmean(x[i], 0) for i in idx])
        lo, hi = np.nanpercentile(boots, 2.5, 0), np.nanpercentile(boots, 97.5, 0)
    bad = cover < min_cover
    f = lambda a: [None if (b or np.isnan(v)) else round(float(v), 4) for v, b in zip(a, bad)]  # noqa: E731
    return {"mean": f(mean), "lo": f(lo), "hi": f(hi), "cover": [round(float(c), 3) for c in cover]}


def tracking(gen: np.ndarray, real: np.ndarray, gen2: np.ndarray | None = None) -> dict:
    """Does the generator know *which* shocks were followed by big moves? Per horizon:
    corr(generated, real) across shocks; with a second independent sample per shock also
    corr(generated, generated'). For a generator that matched the true conditional law the two
    would be equal (both = share of the move's variance that the context predicts), so
    corr(gen, real) << corr(gen, gen') means the model is confident about things that do not
    happen, and corr(gen, real) > corr(gen, gen') means it misses predictable structure."""
    def corr(a, b):
        out = []
        for j in range(a.shape[1]):
            ok = ~np.isnan(a[:, j]) & ~np.isnan(b[:, j])
            if ok.sum() < 10 or a[ok, j].std() == 0 or b[ok, j].std() == 0:
                out.append(None)
            else:
                out.append(round(float(np.corrcoef(a[ok, j], b[ok, j])[0, 1]), 4))
        return out
    res = {"corr_real": corr(gen, real)}
    if gen2 is not None:
        res["corr_self"] = corr(gen, gen2)
    return res


def size_bins(day: Day, edges, times=TIMES_NS, margin_s: int = 300) -> dict:
    """Real impact vs order size from every aggressive order of the day: mean signed mid move
    from just before the order to t after it, per size bin, measured on the real stream."""
    m, b = day.msgs, day.book
    t = m[:, 0]
    mid = _ffill_mid(b, np.nan)[1:]
    lo_t, hi_t = (34_200 + margin_s) * NS, (57_600 - margin_s) * NS
    orders = [o for o in all_orders(day) if lo_t <= o.t < hi_t and o.first > 0]
    sizes = np.array([o.size for o in orders])
    out = {"edges": list(map(int, edges)), "bins": []}
    for a, c in zip(edges[:-1], edges[1:]):
        sel = [o for o, s in zip(orders, sizes) if a <= s < c]
        rows = []
        for o in sel:
            pre = mid[o.first - 1]
            idx = np.searchsorted(t, o.t + np.array(times), side="right") - 1
            rows.append(o.sign * (mid[idx] - pre))
        x = np.array(rows) if rows else np.zeros((0, len(times)))
        out["bins"].append({"lo": int(a), "hi": int(c), "n": len(sel),
                            "median_size": float(np.median(sizes[(sizes >= a) & (sizes < c)]))
                            if len(sel) else None,
                            **(summarize(x) if len(sel) >= 5 else {})})
    return out

In [ ]:
%%writefile /content/lobgen/src/lobgen/cli.py
"""lobgen command line.

  lobgen filter  RAW.gz --tickers GOOG INTC --out data/filtered/2025-12-12.itch
  lobgen convert data/filtered/2025-12-12.itch --ticker GOOG --date 2025-12-12
  lobgen validate data/lobster/GOOG_2025-12-12_34200000_57600000_10.npz
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np

from .itch import filter_file
from .lobster import convert, write_lobster
from .validate import check_book_transitions, check_messages, summary


def _dump(obj, path: Path | None = None) -> None:
    text = json.dumps(obj, indent=2, default=str)
    print(text)
    if path:
        path.write_text(text + "\n")


def cmd_filter(a) -> None:
    out = Path(a.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    rep = filter_file(a.raw, a.tickers, out, allow_truncated=a.allow_truncated)
    _dump(rep, out.with_suffix(".filter.json"))


def cmd_convert(a) -> None:
    msgs, book, rep = convert(a.filtered, a.ticker, a.date, a.levels, a.start, a.end)
    paths = write_lobster(msgs, book, a.out_dir, a.ticker, a.date, a.levels, a.start, a.end,
                          csv=a.csv)
    d = rep.as_dict()
    d["events"] = len(msgs)
    d["files"] = [str(p) for p in paths]
    _dump(d, paths[0].with_suffix(".convert.json"))


def cmd_validate(a) -> None:
    z = np.load(a.npz)
    msgs, book, levels = z["messages"], z["book"], int(z["levels"])
    res = {
        "messages": check_messages(msgs),
        "book_transitions": check_book_transitions(msgs, book, levels),
        "summary": summary(msgs, book),
    }
    _dump(res, Path(a.npz).with_suffix(".validate.json"))
    ok = (res["book_transitions"]["violations"] == 0
          and not any(res["messages"].values()))
    raise SystemExit(0 if ok else 1)


def main() -> None:
    p = argparse.ArgumentParser(prog="lobgen")
    sub = p.add_subparsers(required=True)

    f = sub.add_parser("filter", help="stream a gzipped ITCH day, keep chosen tickers")
    f.add_argument("raw")
    f.add_argument("--tickers", nargs="+", required=True)
    f.add_argument("--out", required=True)
    f.add_argument("--allow-truncated", action="store_true",
                   help="stop cleanly at EOF of a partial download (testing only)")
    f.set_defaults(fn=cmd_filter)

    c = sub.add_parser("convert", help="filtered ITCH -> LOBSTER csv + npz")
    c.add_argument("filtered")
    c.add_argument("--ticker", required=True)
    c.add_argument("--date", required=True, help="YYYY-MM-DD")
    c.add_argument("--levels", type=int, default=10)
    c.add_argument("--start", type=int, default=34_200, help="window start, s after midnight")
    c.add_argument("--end", type=int, default=57_600, help="window end, s after midnight")
    c.add_argument("--out-dir", default="data/lobster")
    c.add_argument("--csv", action="store_true",
                   help="also write full-day LOBSTER csv pair (~2.4 GB per liquid stock-day)")
    c.set_defaults(fn=cmd_convert)

    v = sub.add_parser("validate", help="invariant checks + summary stats")
    v.add_argument("npz")
    v.set_defaults(fn=cmd_validate)

    a = p.parse_args()
    a.fn(a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/lobgen/scripts/build_actions.py
"""Encode every converted day into simulator actions and prove the round trip is exact.

For each data/lobster/{T}_{date}_..._10.npz:
  1. rebuild the order-level book at 09:30 from the filtered ITCH stream (pre-open orders);
  2. encode all LOBSTER rows into actions with a strict simulator;
  3. decode the actions with fresh order ids and compare every message and book row.
Writes {stem}_actions.npz (actions, inexact, init_orders) next to the day file and
results/sim_replay.json. Exit 1 if any non-inexact row fails to reproduce.

Run: uv run python scripts/build_actions.py [--workers 2]
"""

import argparse
import json
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np

from lobgen.lobster import preopen_orders
from lobgen.sim import PTR_RANKS, encode, verify_decode

LOB = Path("data/lobster")
FILTERED = Path("data/filtered")


def one(npz: str, pointer: int = PTR_RANKS, whole_fill: bool = True) -> tuple[str, dict]:
    p = Path(npz)
    ticker, date = p.name.split("_")[:2]
    t0 = time.time()
    init = preopen_orders(FILTERED / f"{date}.itch.gz", ticker)
    z = np.load(p)
    msgs, book, levels = z["messages"], z["book"], int(z["levels"])
    actions, inexact = encode(msgs, init, levels=levels, pointer=pointer, whole_fill=whole_fill)
    np.savez_compressed(p.with_name(p.stem + "_actions.npz"), actions=actions, inexact=inexact,
                        init_orders=init, pointer=pointer, whole_fill=whole_fill)
    res = verify_decode(msgs, book, init, actions, inexact, levels)
    res["init_orders"] = len(init)
    res["pointer"], res["whole_fill"] = pointer, whole_fill
    ex = actions[:, 1] == 4
    res["whole_fill_share"] = round(float(np.mean(actions[ex, 5] == 0)), 4) if ex.any() else None
    res["pointer_share"] = round(float(np.mean(actions[:, 3][np.isin(actions[:, 1], (2, 3))] < 0)), 4)
    res["seconds"] = round(time.time() - t0, 1)
    return f"{ticker}_{date}", res


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--pointer", type=int, default=PTR_RANKS,
                    help="cancels/deletes of the N newest orders use a pointer (0 = the v1 encoding)")
    ap.add_argument("--no-whole-fill", action="store_true",
                    help="executions keep their share count (with --pointer 0: the v1 encoding)")
    a = ap.parse_args()
    days = sorted(str(p) for p in LOB.glob("*_34200000_57600000_10.npz"))
    results = {}
    with ProcessPoolExecutor(a.workers) as ex:
        for name, res in ex.map(one, days, [a.pointer] * len(days),
                                [not a.no_whole_fill] * len(days)):
            results[name] = res
            print(name, json.dumps(res), flush=True)
    Path("results").mkdir(exist_ok=True)
    Path("results/sim_replay.json").write_text(json.dumps(results, indent=2) + "\n")
    bad = sum(r["message_mismatches"] + r["id_mismatches"] + r["book_mismatches"]
              for r in results.values())
    print("ALL EXACT" if not bad else f"{bad} mismatches")
    sys.exit(1 if bad else 0)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/lobgen/scripts/train.py
"""Train the event transformer (step 4). Runs locally (CPU / Apple MPS) or in Colab (CUDA).

Train Mon-Wed, validate Thu, Fri stays untouched for the detective. Atoms for the spike-aware
heads are fit on the training days only. Writes the best-validation checkpoint and a JSON log.

  uv run --group model python scripts/train.py --out checkpoints/full.pt
  # ablations (step 5):
  uv run --group model python scripts/train.py --no-atoms --out checkpoints/no_atoms.pt
  uv run --group model python scripts/train.py --no-book  --out checkpoints/no_book.pt
  uv run --group model python scripts/train.py --ctx 32   --out checkpoints/ctx32.pt
"""

import argparse
import json
import tempfile
from pathlib import Path

import numpy as np

from lobgen.model import EventData, ModelConfig, fit_atoms, train

STEM = "{t}_{d}_34200000_57600000_10.npz"


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--lob-dir", default="data/lobster")
    ap.add_argument("--tickers", nargs="+", default=["GOOG", "INTC"])
    ap.add_argument("--train-dates", nargs="+", default=["2025-12-08", "2025-12-09", "2025-12-10"])
    ap.add_argument("--val-dates", nargs="+", default=["2025-12-11"])
    ap.add_argument("--out", default="checkpoints/full.pt")
    ap.add_argument("--steps", type=int, default=20_000)
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--ctx", type=int, default=256)
    ap.add_argument("--d-model", type=int, default=256)
    ap.add_argument("--layers", type=int, default=6)
    ap.add_argument("--heads", type=int, default=8)
    ap.add_argument("--n-mix", type=int, default=8)
    ap.add_argument("--dropout", type=float, default=0.1)
    ap.add_argument("--no-atoms", action="store_true", help="ablation: pure continuous heads")
    ap.add_argument("--no-book", action="store_true", help="ablation: hide the book from the input")
    ap.add_argument("--atom-share", type=float, default=0.005)
    ap.add_argument("--max-atoms", type=int, default=48)
    ap.add_argument("--no-regime", action="store_true",
                    help="ablation: no day-so-far input features (v1 inputs)")
    ap.add_argument("--flow", action="store_true",
                    help="v3: also the order-flow memory inputs (trade-sign averages over trades)")
    ap.add_argument("--market", action="store_true",
                    help="v4: also clock-time own-mid and ETF (QQQ, SMH) inputs; implies --flow")
    ap.add_argument("--market-dir", default="data/market")
    ap.add_argument("--eval-every", type=int, default=500)
    ap.add_argument("--eval-batches", type=int, default=20)
    ap.add_argument("--device", default=None)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--disk-features", action="store_true",
                    help="keep the input features in memory-mapped temp files, not RAM (same values;"
                         " v4's ~9 GB of inputs do not fit a 12.7 GB Colab runtime otherwise)")
    a = ap.parse_args()

    lob = Path(a.lob_dir)
    paths = lambda dates: [lob / STEM.format(t=t, d=d) for t in a.tickers for d in dates]  # noqa: E731
    mdir = a.market_dir if a.market else None
    print(f"loading + featurizing {len(a.tickers)} tickers x {len(a.train_dates) + len(a.val_dates)} days"
          " (a few minutes, no output until done)", flush=True)
    tmp = tempfile.TemporaryDirectory(prefix="lobgen_feat_") if a.disk_features else None
    fdir = tmp.name if tmp else None
    tr = EventData.from_npz(paths(a.train_dates), a.ctx, market_dir=mdir, out_dir=fdir)
    va = EventData.from_npz(paths(a.val_dates), a.ctx, market_dir=mdir, out_dir=fdir)
    acts = np.concatenate([p[3][::10] for p in tr.parts])  # 10% is plenty to find >=0.5% values
    atoms = fit_atoms(acts, a.atom_share, a.max_atoms)
    print("atoms:", json.dumps(atoms))
    cfg = ModelConfig(d_model=a.d_model, n_layers=a.layers, n_heads=a.heads, ctx=a.ctx,
                      dropout=a.dropout, n_mix=a.n_mix, use_atoms=not a.no_atoms,
                      use_book=not a.no_book, atom_share=a.atom_share, max_atoms=a.max_atoms,
                      atoms=atoms, encoding=tr.encoding, regime=not a.no_regime,
                      flow=a.flow or a.market, market=a.market)
    if va.encoding != tr.encoding:
        raise ValueError("train and validation days use different action encodings")
    res = train(cfg, tr, va, a.out, steps=a.steps, batch=a.batch, lr=a.lr,
                eval_every=a.eval_every, eval_batches=a.eval_batches, device=a.device, seed=a.seed)
    log = Path(a.out).with_suffix(".json")
    log.write_text(json.dumps({"args": vars(a), "atoms": atoms, **res}, indent=1) + "\n")
    print("best val nats/event", res["best_val_event"], "->", a.out)
    if tmp:
        del tr, va  # drop the memory maps before removing their files
        tmp.cleanup()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/lobgen/scripts/run_detective.py
"""Detective AUC vs run length for every generator, plus the two real-data references.

Test day Fri 2025-12-12. Windows: n stratified across the day (5 min from open/close),
cond_len real events of context, gen_len events generated. Every generator continues from the
real order-level book at the window start. References:
  same_day   real runs of neighbouring windows (pairs of adjacent windows, ~20 s apart) - what a
             perfect conditional generator would score; should be ~0.5.
  real_<d>   real runs of day d at the same time of day - a generator that knows "a typical
             day" but not *this* day. Mon 12-08 and Thu 12-11.
  replay     the real run re-played through the simulator - 0.5 by construction (sanity check).
Baselines are trained on Mon-Wed (12-08..12-10).

Results are merged into results/detective.json (so model runs can be added later with
--model) and drawn to results/detective_{ticker}.png.

Run: uv run --group bench python scripts/run_detective.py [--tickers GOOG INTC] [--n 1000]
"""

import argparse
import gc
import json
import time
from collections import Counter
from pathlib import Path

import numpy as np

from lobgen.baselines import HistoryBootstrap, Markov, PooledIID, RealOtherDay, Replay, train_pool
from lobgen.data import Day, make_windows
from lobgen.detective import detect
from lobgen.plots import plot_detective

LOB = Path("data/lobster")
OUT = Path("results")
STEM = "{t}_{d}_34200000_57600000_10.npz"
TEST, TRAIN, OTHERS = "2025-12-12", ["2025-12-08", "2025-12-09", "2025-12-10"], ["2025-12-08", "2025-12-11"]
LENGTHS = [5, 10, 20, 50, 100, 200, 500, 1000]


def repairs_per_event(gen, n_events) -> dict:
    tot = sum((g.repairs for g in gen), Counter())
    n = len(gen) * n_events
    return {k: round(v / n, 5) for k, v in tot.most_common()} | {
        "skipped_per_event": round(sum(g.skipped for g in gen) / n, 5)}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--tickers", nargs="+", default=["GOOG", "INTC"])
    ap.add_argument("--n", type=int, default=1000)
    ap.add_argument("--cond", type=int, default=500)
    ap.add_argument("--gen", type=int, default=1000)
    ap.add_argument("--lengths", type=int, nargs="+", default=LENGTHS)
    ap.add_argument("--only", nargs="*", help="generator names to (re)run; default all")
    ap.add_argument("--model", help="checkpoint of a trained model (adds generator 'model')")
    ap.add_argument("--model-name", default="model")
    ap.add_argument("--device", default=None)
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    OUT.mkdir(exist_ok=True)
    path = OUT / "detective.json"
    allres = json.loads(path.read_text()) if path.exists() else {}
    want = (lambda name: True) if not a.only else (lambda name: name in a.only)

    for t in a.tickers:
        res = allres.setdefault(t, {})
        meta = res.setdefault("_meta", {})
        meta.update({"test": TEST, "train": TRAIN, "n_windows": a.n, "cond_len": a.cond,
                     "gen_len": a.gen, "seed": a.seed, "repairs": meta.get("repairs", {})})
        test = Day.load(LOB / STEM.format(t=t, d=TEST))
        W = make_windows(test, a.n, a.cond, a.gen, seed=a.seed)
        real = [(w.real_msgs, w.real_book) for w in W]

        def run(name, fake, real_=real, rg=None, fg=None):
            t0 = time.time()
            res[name] = detect(real_, fake, a.lengths, rg, fg, seed=a.seed)
            print(f"{t} {name:18s} " + " ".join(f"{L}:{res[name][L]['auc']:.3f}"
                                                for L in a.lengths)
                  + f"  ({time.time() - t0:.0f}s)", flush=True)

        if want("same_day"):
            k = np.arange(len(W) // 2)
            run("same_day", real[1::2][:len(k)], real[0::2][:len(k)], k, k)

        for d in OTHERS:
            name = f"real_{d}"
            if want(name):
                other = Day.load(LOB / STEM.format(t=t, d=d), actions=False)
                gen = RealOtherDay(other).generate(W, a.gen)
                del other
                gc.collect()
                run(name, [(g.msgs, g.book) for g in gen])

        gens = []
        if want("replay"):
            gens.append(Replay())
        if want("history"):
            gens.append(HistoryBootstrap())
        if any(want(n) for n in ("pooled", "markov")):
            pool = train_pool([Day.load(LOB / STEM.format(t=t, d=d), book=False) for d in TRAIN])
            if want("pooled"):
                gens.append(PooledIID(pool))
            if want("markov"):
                gens.append(Markov(pool))
        if a.model:
            from lobgen.model import ModelGenerator, attach_market
            gens.append(attach_market(ModelGenerator.load(a.model, device=a.device,
                                                          name=a.model_name), TEST))

        for g in gens:
            t0 = time.time()
            out = g.generate(W, a.gen, seed=a.seed)
            print(f"{t} {g.name:18s} generated in {time.time() - t0:.0f}s", flush=True)
            meta["repairs"][g.name] = repairs_per_event(out, a.gen)
            run(g.name, [(o.msgs, o.book) for o in out])
            del out
        path.write_text(json.dumps(allres, indent=1) + "\n")
        plot_detective({k: v for k, v in res.items() if not k.startswith("_")}, t,
                       OUT / f"detective_{t}.png")
        del test, W, real
        gc.collect()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/lobgen/scripts/shock_test.py
"""Shock test (src/lobgen/shock.py): real large market orders of the test day, and injected ones.

Part 1, real shocks. Every top-1% aggressive order of Fri 2025-12-12 (first of each cluster,
up to --max-shocks) ends a window's context; each generator continues from the real book right
after it, --samples times. Compared with what really followed: signed mid move (ticks) at
1 ms .. 10 s and 1 .. 2000 events, spread, depth on the hit side, follow-on aggressive volume,
time to the k-th event. Tracking: corr(generated move, real move) across shocks, against
corr(sample 1, sample 2) of the same generator (see shock.tracking).

Part 2, injection. A market order of each size in --inject-sizes (default: the day's 90th,
99th and 99.9th percentile order size and 3x the last) is inserted at --inject-n ordinary
moments; impact = mean move with the order - mean move without it (same windows, same seeds).
The real reference is every aggressive order of the day, binned by size.

Run: uv run --group bench --group model python scripts/shock_test.py --model v2=checkpoints/v2.pt
Writes results/shock.json (merged per ticker and generator), results/shock_{T}.png.
"""

import argparse
import gc
import json
import time
from pathlib import Path

import numpy as np

from lobgen.baselines import HistoryBootstrap, Markov, train_pool
from lobgen.data import Day, make_windows
from lobgen.shock import (EVENTS, TIMES_NS, inject, measure, ref_point, select_shocks,
                          shock_ref, shock_windows, size_bins, stack, summarize, tracking)

LOB = Path("data/lobster")
STEM = "{t}_{d}_34200000_57600000_10.npz"
TEST, TRAIN = "2025-12-12", ["2025-12-08", "2025-12-09", "2025-12-10"]
METRICS = ["move_t", "spread_t", "move_ev", "spread_ev", "depth_ev", "flow_ev", "ms_ev"]


def nice(x: float) -> int:
    """Round to two significant digits."""
    e = 10 ** max(int(np.floor(np.log10(max(x, 1)))) - 1, 0)
    return int(round(x / e) * e)


def run_gen(g, W, refs, n_events, seed, chunk=200) -> dict[str, np.ndarray]:
    """Generate in chunks and measure at once (a 2000-event GOOG run with its book is ~0.6 MB)."""
    rows = []
    for k in range(0, len(W), chunk):
        out = g.generate(W[k:k + chunk], n_events, seed=seed + k)
        for w, r, o in zip(W[k:k + chunk], refs[k:k + chunk], out):
            rows.append(measure(o.msgs, o.book, w.cond_book[-1], r))
        del out
    return stack(rows)


def summary(per_shock: dict) -> dict:
    return {m: summarize(per_shock[m]) for m in METRICS}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--tickers", nargs="+", default=["GOOG", "INTC"])
    ap.add_argument("--model", nargs="*", default=[], help="name=checkpoint pairs")
    ap.add_argument("--baselines", nargs="*", default=["history", "markov"])
    ap.add_argument("--quantile", type=float, default=0.99)
    ap.add_argument("--max-shocks", type=int, default=300)
    ap.add_argument("--samples", type=int, default=2)
    ap.add_argument("--gen", type=int, default=2000)
    ap.add_argument("--inject-n", type=int, default=100)
    ap.add_argument("--inject-gen", type=int, default=1000)
    ap.add_argument("--inject-sizes", type=int, nargs="*", default=None)
    ap.add_argument("--no-inject", action="store_true")
    ap.add_argument("--market-dir", default="data/market", help="ETF quotes for market models")
    ap.add_argument("--out-dir", default="results")
    ap.add_argument("--device", default=None)
    a = ap.parse_args()
    out_dir = Path(a.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "shock.json"
    allres = json.loads(out_path.read_text()) if out_path.exists() else {}

    for t in a.tickers:
        t0 = time.time()
        day = Day.load(LOB / STEM.format(t=t, d=TEST))
        shocks = select_shocks(day, a.quantile, gen_len=a.gen, max_n=a.max_shocks)
        W = shock_windows(day, shocks, 500, a.gen)
        refs = [shock_ref(day, s) for s in shocks]
        res = allres.setdefault(t, {})
        sizes = np.array([s.size for s in shocks])
        res["_meta"] = {"test": TEST, "n_shocks": len(shocks), "quantile": a.quantile,
                        "min_size": int(sizes.min()), "median_size": float(np.median(sizes)),
                        "buy_share": float(np.mean([s.sign > 0 for s in shocks])),
                        "levels_moved_share": float(np.mean([s.levels > 0 for s in shocks])),
                        "samples": a.samples, "gen_len": a.gen,
                        "events": list(EVENTS), "times_ms": [x / 1e6 for x in TIMES_NS]}
        real = stack([measure(w.real_msgs, w.real_book, w.cond_book[-1], r) for w, r in zip(W, refs)])
        res["real"] = summary(real)
        print(f"{t}: {len(shocks)} shocks >= {sizes.min()} shares (median {np.median(sizes):.0f}), "
              f"real move at {[x / 1e6 for x in TIMES_NS]} ms: {res['real']['move_t']['mean']}",
              flush=True)

        # Real impact vs size from every aggressive order of the day.
        from lobgen.shock import all_orders
        osz = np.array([o.size for o in all_orders(day)])
        q = np.quantile(osz, [0.5, 0.9, 0.99, 0.999])
        edges = sorted({1, *map(int, q), int(osz.max()) + 1})
        res["real_by_size"] = size_bins(day, edges)
        inj_sizes = a.inject_sizes or sorted({nice(q[1]), nice(q[2]), nice(q[3]), nice(3 * q[3])})
        Wc = [] if a.no_inject else make_windows(day, a.inject_n, 500, a.inject_gen, seed=1)

        gens = []
        if any(b in a.baselines for b in ("history",)):
            gens.append(HistoryBootstrap())
        if "markov" in a.baselines:
            gens.append(Markov(train_pool([Day.load(LOB / STEM.format(t=t, d=d), book=False)
                                           for d in TRAIN])))
        if a.model:
            from lobgen.model import ModelGenerator, attach_market
            gens += [attach_market(ModelGenerator.load(p, device=a.device, name=n), TEST,
                                   a.market_dir)
                     for n, p in (m.split("=", 1) for m in a.model)]
        del day
        gc.collect()

        for g in gens:
            tg = time.time()
            samples = [run_gen(g, W, refs, a.gen, seed=1000 * s) for s in range(a.samples)]
            mean = {m: np.nanmean(np.stack([s[m] for s in samples]), 0) for m in METRICS}
            entry = summary(mean)
            entry["tracking"] = {m: tracking(samples[0][m], real[m],
                                             samples[1][m] if a.samples > 1 else None)
                                 for m in ("move_t", "move_ev")}
            if Wc:
                signs = [1 if i % 2 == 0 else -1 for i in range(len(Wc))]
                ctrl_refs = [ref_point(w.cond_book[-1], w.t_prev + 1_000, s, 0)
                             for w, s in zip(Wc, signs)]
                ctrl = run_gen(g, Wc, ctrl_refs, a.inject_gen, seed=7)
                inj = {"sizes": inj_sizes, "control": summarize(ctrl["move_t"]), "by_size": []}
                for size in inj_sizes:
                    pairs = [inject(w, s, size) for w, s in zip(Wc, signs)]
                    Wi, ri = [p[0] for p in pairs], [p[1] for p in pairs]
                    got = run_gen(g, Wi, ri, a.inject_gen, seed=7)
                    inj["by_size"].append({"size": size, "filled": float(np.mean([r["size"] for r in ri])),
                                           "move_t": summarize(got["move_t"]),
                                           "impact_t": summarize(got["move_t"] - ctrl["move_t"]),
                                           "spread_t": summarize(got["spread_t"])})
                entry["inject"] = inj
            res[g.name] = entry
            mt = entry["move_t"]["mean"]
            print(f"{t} {g.name:10s} move_t {mt}  corr {entry['tracking']['move_t']}  "
                  f"({time.time() - tg:.0f}s)", flush=True)
            out_path.write_text(json.dumps(allres, indent=1) + "\n")
        out_path.write_text(json.dumps(allres, indent=1) + "\n")
        from lobgen.plots import plot_shock
        plot_shock(res, t, out_dir / f"shock_{t}.png")
        print(f"{t} done in {(time.time() - t0) / 60:.1f} min", flush=True)
        del W, Wc, real
        gc.collect()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/lobgen/scripts/track_test.py
"""Tracking test: given what the market did, does the generated stock follow the real one?

A generator cannot know news before it happens, and neither can anyone else. What it can do is
react to it: news reaches a single stock through the market (market.py), so a model that reads
the market should, given the market's real path, move the stock about as far as it really
moved. This script measures that on the held-out test day (Fri 2025-12-12, which had a
market-wide sell-off around 10:58 ET: GOOG -1.5% and INTC -3.7% on the day).

For --n windows spread over the day, each generator runs --samples times from the real book
for --horizon seconds of clock time. Compared with the real move of the stock over the same
interval (bps):
  corr      correlation across windows of the generator's mean move with the real move
  r2_zero   1 - sum (real - pred)^2 / sum real^2: skill against "the price stays put"
  std       std of single generated moves vs real moves (too calm / too wild)
Generators:
  <model>            a model that does not read the market (v2, v3): it cannot track
  <model>            a market model (v4) given the real market path
  <model>@frozen     the same model, the market frozen at the window start (no news): how much
                     of its tracking comes from the market input
  beta               linear baseline: real move = beta . (QQQ, SMH) move, OLS on the training
                     days at the same horizon; uses the same market information as v4
News scenario (--scenario-bps, default -50): the market model again on the same windows with
QQQ and SMH pushed an extra -50 bps over 10 s from the start (a headline). The model's
response = scenario move - real-market move; it is compared with the real stock's beta, from
the training days, to market moves of at least 10 bps.

Case study: the window that starts 30 s before the test day's worst one-minute QQQ move,
--case-samples runs over --case-seconds, mid paths drawn against the real path
(results/track_case_{T}.png).

Run: uv run --group bench --group model python scripts/track_test.py \
        --model v3=checkpoints/v3.pt v4=checkpoints/v4.pt [--n 60] [--horizon 10]
Needs data/market/ (notebooks/fetch_itch_colab.ipynb, market part). Writes results/track.json.
"""

import argparse
import gc
import json
import time
from dataclasses import replace
from pathlib import Path

import numpy as np

from lobgen.data import NS, Day, make_windows, windows_at
from lobgen.market import Market
from lobgen.shock import _ffill_mid

LOB = Path("data/lobster")
STEM = "{t}_{d}_34200000_57600000_10.npz"
TEST, TRAIN = "2025-12-12", ["2025-12-08", "2025-12-09", "2025-12-10"]


def day_mid(day: Day) -> tuple[np.ndarray, np.ndarray]:
    """(times, log mid) after every row of a day."""
    mid = _ffill_mid(day.book, np.nan)[1:]
    return day.msgs[:, 0], np.log(mid)


def log_mid_at(t: np.ndarray, x: np.ndarray, q) -> np.ndarray:
    k = np.searchsorted(t, np.asarray(q), side="right") - 1
    return np.where(k >= 0, x[np.maximum(k, 0)], np.nan)


def fit_beta(t: str, horizon_s: float, market_dir: str) -> dict:
    """OLS (no intercept) of the stock's horizon return on QQQ's and SMH's, training days,
    non-overlapping intervals 09:35-15:55; also the beta on large market moves only."""
    X, y = [], []
    for d in TRAIN:
        day = Day.load(LOB / STEM.format(t=t, d=d), actions=False)
        ts, x = day_mid(day)
        del day
        mk = Market.load(d, market_dir)
        grid = np.arange(34_500 * NS, 57_300 * NS, int(horizon_s * NS))
        sx = np.diff(log_mid_at(ts, x, grid)) * 1e4
        ex = np.stack([np.diff(s.value(grid)) * 1e4 for s in mk.series], 1)
        ok = ~np.isnan(sx) & ~np.isnan(ex).any(1)
        X.append(ex[ok])
        y.append(sx[ok])
        gc.collect()
    X, y = np.vstack(X), np.concatenate(y)
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    big = np.abs(X[:, 0]) >= 10
    b_big = float(np.sum(X[big, 0] * y[big]) / np.sum(X[big, 0] ** 2)) if big.sum() >= 5 else None
    b_qqq = float(np.sum(X[:, 0] * y) / np.sum(X[:, 0] ** 2))
    return {"beta": beta.tolist(), "n": int(len(y)), "beta_qqq_only": b_qqq,
            "beta_qqq_big_moves": b_big, "n_big": int(big.sum()),
            "r2_in_sample": float(1 - np.sum((y - X @ beta) ** 2) / np.sum(y ** 2))}


def gen_moves(g, W, K, horizon_ns, n_events, seed, chunk=200) -> np.ndarray:
    """[K, len(W)] log-mid move (bps) of each generated run from its start to start + horizon
    (nan when the run ends before then). The K samples of a window share a batch."""
    rep = [w for _ in range(K) for w in W]
    out = []
    for k in range(0, len(rep), chunk):
        res = g.generate(rep[k:k + chunk], n_events, seed=seed + k)
        for w, o in zip(rep[k:k + chunk], res):
            mid = _ffill_mid(o.book, (w.cond_book[-1][0] + w.cond_book[-1][2]) / 2 / 100)
            if o.msgs[-1, 0] < w.t_prev + horizon_ns:
                out.append(np.nan)
                continue
            i = np.searchsorted(o.msgs[:, 0], w.t_prev + horizon_ns, side="right")
            out.append((np.log(mid[i]) - np.log(mid[0])) * 1e4)
        del res
    return np.array(out).reshape(K, len(W))


def gen_paths(g, W, n_events, seed, grid_ns) -> np.ndarray:
    """Mid paths (bps from the start) on a clock grid, one row per run."""
    res = g.generate(W, n_events, seed=seed)
    rows = []
    for w, o in zip(W, res):
        mid = _ffill_mid(o.book, (w.cond_book[-1][0] + w.cond_book[-1][2]) / 2 / 100)
        i = np.searchsorted(o.msgs[:, 0], w.t_prev + grid_ns, side="right")
        p = (np.log(mid[i]) - np.log(mid[0])) * 1e4
        p[w.t_prev + grid_ns > o.msgs[-1, 0]] = np.nan
        rows.append(p)
    return np.array(rows)


def scores(pred: np.ndarray, real: np.ndarray, single: np.ndarray | None = None) -> dict:
    ok = ~np.isnan(pred) & ~np.isnan(real)
    p, r = pred[ok], real[ok]
    out = {"n": int(ok.sum()), "coverage": float(ok.mean()),
           "corr": float(np.corrcoef(p, r)[0, 1]) if ok.sum() > 5 and p.std() > 0 else None,
           "r2_zero": float(1 - np.sum((r - p) ** 2) / np.sum(r ** 2)) if ok.sum() else None,
           "mae_bps": float(np.mean(np.abs(r - p))) if ok.sum() else None,
           "std_real": float(r.std()) if ok.sum() else None}
    if single is not None:
        s = single[~np.isnan(single)]
        out["std_single_run"] = float(s.std()) if len(s) else None
    return out


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--tickers", nargs="+", default=["GOOG", "INTC"])
    ap.add_argument("--model", nargs="*", default=[], help="name=checkpoint pairs")
    ap.add_argument("--n", type=int, default=80)
    ap.add_argument("--horizon", type=float, default=10.0, help="seconds")
    ap.add_argument("--samples", type=int, default=2)
    ap.add_argument("--slack", type=float, default=2.0,
                    help="generate this many times the real event count of the horizon")
    ap.add_argument("--scenario-bps", type=float, default=-50.0)
    ap.add_argument("--scenario-ramp", type=float, default=10.0)
    ap.add_argument("--case-samples", type=int, default=8)
    ap.add_argument("--case-seconds", type=float, default=30.0)
    ap.add_argument("--case-slack", type=float, default=1.5)
    ap.add_argument("--market-dir", default="data/market")
    ap.add_argument("--out-dir", default="results")
    ap.add_argument("--device", default=None)
    a = ap.parse_args()
    from lobgen.model import ModelGenerator
    from lobgen.plots import plot_track_case

    out_dir = Path(a.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "track.json"
    allres = json.loads(out_path.read_text()) if out_path.exists() else {}
    H = int(a.horizon * NS)
    gens = [ModelGenerator.load(p, device=a.device, name=n) for n, p in
            (m.split("=", 1) for m in a.model)]
    for t in a.tickers:
        t_start = time.time()
        res = allres.setdefault(t, {})
        beta = fit_beta(t, a.horizon, a.market_dir)
        res["beta_fit"] = beta
        day = Day.load(LOB / STEM.format(t=t, d=TEST))
        mk = Market.load(TEST, a.market_dir)
        ts, x = day_mid(day)
        W = make_windows(day, a.n, 500, 1000, seed=2)
        t0s = np.array([w.t_prev for w in W])
        real = (log_mid_at(ts, x, t0s + H) - log_mid_at(ts, x, t0s)) * 1e4
        n_real = np.searchsorted(ts, t0s + H) - np.searchsorted(ts, t0s)
        n_events = int(np.ceil(a.slack * np.quantile(n_real, 0.9)))
        etf = np.stack([[(s.value(np.array([t0 + H]))[0] - s.value(np.array([t0]))[0]) * 1e4
                         for s in mk.series] for t0 in t0s])
        res["_meta"] = {"test": TEST, "n_windows": len(W), "horizon_s": a.horizon,
                        "samples": a.samples, "gen_events": n_events,
                        "real_events_median": float(np.median(n_real)),
                        "real_move_std_bps": float(np.nanstd(real)),
                        "qqq_move_std_bps": float(np.std(etf[:, 0]))}
        res["beta"] = scores(etf @ np.array(beta["beta"]), real)
        res["zero"] = scores(np.zeros_like(real), real)
        print(f"{t}: horizon {a.horizon}s, {len(W)} windows, {n_events} events per run; "
              f"beta {np.round(beta['beta'], 2)} -> corr {res['beta']['corr']:.3f}", flush=True)

        # Case-study window: 30 s before the worst one-minute QQQ move of the test day.
        q = mk.series[0]
        grid = np.arange(34_500 * NS, 57_000 * NS, NS)
        r60 = q.value(grid[60:]) - q.value(grid[:-60])
        t_case = int(grid[int(np.argmin(r60))] - 30 * NS)
        row = int(np.searchsorted(ts, t_case))
        case_w = windows_at(day, [row], 500, 1000)[0]
        case_grid = np.arange(0, int(a.case_seconds * NS) + 1, NS // 2)
        n_case = int(a.case_slack * (np.searchsorted(ts, t_case + case_grid[-1]) - row)) + 100
        case = {"t0_s": t_case / NS, "grid_s": (case_grid / NS).tolist(),
                "real": ((log_mid_at(ts, x, case_w.t_prev + case_grid)
                          - log_mid_at(ts, x, case_w.t_prev)) * 1e4).tolist(),
                "qqq": [(q.value(np.array([case_w.t_prev + g]))[0]
                         - q.value(np.array([case_w.t_prev]))[0]) * 1e4 for g in case_grid],
                "beta_qqq": beta["beta_qqq_only"], "runs": {}}
        if res.get("case", {}).get("t0_s") == case["t0_s"]:   # keep other models' runs
            case["runs"].update(res["case"]["runs"])
        del day
        gc.collect()

        for g in gens:
            tg = time.time()
            reads_market = g.model.cfg.market
            for w in W:
                w.market = mk if reads_market else None
            runs = gen_moves(g, W, a.samples, H, n_events, seed=0)
            with np.errstate(all="ignore"):
                res[g.name] = scores(np.nanmean(runs, 0), real, runs[0])
            print(f"{t} {g.name:16s} {res[g.name]}  ({time.time() - tg:.0f}s)", flush=True)
            out_path.write_text(json.dumps(allres, indent=1) + "\n")
            if reads_market:
                for w in W:
                    w.market = mk.frozen(w.t_prev)
                fr = gen_moves(g, W, a.samples, H, n_events, seed=0)
                with np.errstate(all="ignore"):
                    res[f"{g.name}@frozen"] = scores(np.nanmean(fr, 0), real, fr[0])
                print(f"{t} {g.name + '@frozen':16s} {res[f'{g.name}@frozen']}", flush=True)
            if reads_market and a.scenario_bps:
                for w in W:
                    w.market = mk.scenario(w.t_prev, a.scenario_bps, a.scenario_ramp)
                sc = gen_moves(g, W, a.samples, H, n_events, seed=0)   # same seeds as `runs`
                with np.errstate(all="ignore"):
                    resp = np.nanmean(sc - runs, 0)
                ok = ~np.isnan(resp)
                rng = np.random.default_rng(0)
                boot = [np.mean(rng.choice(resp[ok], ok.sum())) for _ in range(1000)]
                res[f"{g.name}@scenario"] = {
                    "scenario_bps": a.scenario_bps, "ramp_s": a.scenario_ramp, "n": int(ok.sum()),
                    "response_bps": float(np.mean(resp[ok])),
                    "response_ci": [float(v) for v in np.percentile(boot, [2.5, 97.5])],
                    "implied_beta": float(np.mean(resp[ok]) / a.scenario_bps),
                    "real_beta_big_moves": beta["beta_qqq_big_moves"],
                    "real_beta_all": beta["beta_qqq_only"]}
                print(f"{t} {g.name}@scenario {res[f'{g.name}@scenario']}", flush=True)
            for w in W:
                w.market = None
            # case study: K runs with the real market (and K with it frozen) in one batch
            conds = [(g.name, mk if reads_market else None)]
            if reads_market:
                conds.append((f"{g.name}@frozen", mk.frozen(case_w.t_prev)))
            cw = [replace(case_w, market=m) for _, m in conds for _ in range(a.case_samples)]
            paths = gen_paths(g, cw, n_case, 99, case_grid)
            for j, (name, _) in enumerate(conds):
                case["runs"][name] = paths[j * a.case_samples:(j + 1) * a.case_samples].tolist()
            out_path.write_text(json.dumps(allres | {t: res | {"case": case}}, indent=1) + "\n")
        res["case"] = case
        out_path.write_text(json.dumps(allres, indent=1) + "\n")
        plot_track_case(case, t, out_dir / f"track_case_{t}.png")
        print(f"{t} done in {(time.time() - t_start) / 60:.1f} min", flush=True)
        del W
        gc.collect()


if __name__ == "__main__":
    main()

In [ ]:
# Prepared data. Reused from Drive (~1 min) when it was built by exactly this code; otherwise
# rebuilt from the filtered ITCH files (~20-30 min), checked, and saved to Drive for next time.
import hashlib, shutil
os.makedirs(f"{REPO}/data/lobster", exist_ok=True)
missing = [d for d in DATES if not os.path.exists(f"{DRIVE}/filtered/{d}.itch.gz")]
assert not missing, f"not in {DRIVE}/filtered: {missing} (run fetch_itch_colab.ipynb first)"
if not os.path.exists(f"{REPO}/data/filtered"):
    os.symlink(f"{DRIVE}/filtered", f"{REPO}/data/filtered")
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/src"
BUILD_SRC = ["itch", "book", "lobster", "validate", "sim", "cli"]   # everything the data depends on
key = hashlib.sha256(b"".join(open(f, "rb").read() for f in
      [f"src/lobgen/{m}.py" for m in BUILD_SRC] + ["scripts/build_actions.py"])).hexdigest()[:12]
CACHE = f"{DRIVE}/prepared_{key}"
if os.path.exists(f"{CACHE}/DONE"):
    shutil.copytree(CACHE, "data/lobster", dirs_exist_ok=True)
    shutil.copy(f"{CACHE}/build_actions.log", "/content/build_actions.log")
    print("prepared data copied from", CACHE)
else:
    for d in DATES:
        for t in TICKERS:
            out = f"data/lobster/{t}_{d}_34200000_57600000_10.npz"
            if not os.path.exists(out):
                subprocess.run(["python", "-m", "lobgen.cli", "convert", f"data/filtered/{d}.itch.gz",
                                "--ticker", t, "--date", d], check=True, stdout=subprocess.DEVNULL)
            print("converted", t, d, flush=True)
    # Encode every day into simulator actions and verify the exact round trip.
    !cd {REPO} && python -u scripts/build_actions.py --workers 1 2>&1 | tee /content/build_actions.log | cut -c1-160
    assert "ALL EXACT" in open("/content/build_actions.log").read(), "round trip failed: stop here"
    for old in os.listdir(DRIVE):            # data built by older code is no longer needed
        if old.startswith("prepared_"):
            shutil.rmtree(f"{DRIVE}/{old}")
    shutil.copytree("data/lobster", CACHE, dirs_exist_ok=True)
    shutil.copy("/content/build_actions.log", CACHE)
    open(f"{CACHE}/DONE", "w").write("ok")
    print("prepared data saved to", CACHE)
assert "ALL EXACT" in open("/content/build_actions.log").read()

In [ ]:
# Market quotes (from fetch_itch_colab.ipynb): small, copied next to the data.
missing = [f"{e}_{d}.npz" for d in DATES for e in ETFS if not os.path.exists(f"{DRIVE}/market/{e}_{d}.npz")]
assert not missing, f"not in {DRIVE}/market: {missing} - run fetch_itch_colab.ipynb first"
shutil.copytree(f"{DRIVE}/market", f"{REPO}/data/market", dirs_exist_ok=True)
print("market quotes:", sorted(os.listdir(f"{REPO}/data/market")))

In [ ]:
# Smoke test (~3 min, one INTC day): 300 steps of a small model on the GPU; losses must be finite.
import json, math
!cd {REPO} && python scripts/train.py --tickers INTC --train-dates 2025-12-10 --val-dates 2025-12-11 --steps 300 --eval-every 100 --eval-batches 5 --d-model 64 --layers 2 --heads 2 --ctx 64 --market --disk-features --out /content/smoke.pt
hist = json.load(open("/content/smoke.json"))["history"]
print(hist[-1])
assert all(math.isfinite(v) for h in hist for k, v in h.items() if k != "step"), "NaN/inf loss: stop"
assert hist[-1]["val_event"] < hist[0]["val_event"], "validation loss did not improve"

In [ ]:
# Training. A run is skipped if its final checkpoint (<name>_last.pt) is already in Drive, and
# continues from <name>_state.pt (saved every 500 steps) if it was interrupted.
# <name>.pt = best validation step, <name>_last.pt = final weights, <name>.json = log.
# Loading + featurizing the days takes a few minutes before the first step: that is not a hang.
# --disk-features: the inputs (~8 GB for v3/v4) go to memory-mapped files on the runtime's disk,
# not RAM (identical values); in RAM they need ~14 GB and the 12.7 GB runtime kills the run
# (it shows only "^C").
# If a run fails or is interrupted, this cell raises, so "Run all" stops here instead of going on.
for name, flags in RUNS.items():
    if os.path.exists(f"{DRIVE}/checkpoints/{name}_last.pt"):
        print("done already:", name)
        continue
    print("training", name, flags, flush=True)
    !cd {REPO} && python -u scripts/train.py --steps {STEPS} {flags} --disk-features --out {DRIVE}/checkpoints/{name}.pt
    assert os.path.exists(f"{DRIVE}/checkpoints/{name}_last.pt"), \
        f"{name} did not finish (error or interrupt above). Re-run this cell to resume it."

In [ ]:
# Detective on the test day (Fri), merged into MyDrive/lobgen/results/detective.json after every
# run. The floors (same-day, other days) do not depend on the action encoding and are already
# computed on the laptop; replay and the baselines do, so they are re-run here.
import shutil
os.makedirs(f"{REPO}/results", exist_ok=True)
if os.path.exists(f"{DRIVE}/results/detective.json"):
    shutil.copy(f"{DRIVE}/results/detective.json", f"{REPO}/results/detective.json")

def done(name, t):
    p = f"{REPO}/results/detective.json"
    return os.path.exists(p) and name in json.load(open(p)).get(t, {})

def save():   # to Drive after every (generator, ticker): an interrupted session loses one at most
    for f in os.listdir(f"{REPO}/results"):
        shutil.copy(f"{REPO}/results/{f}", f"{DRIVE}/results/")

def step(ck):
    return torch.load(ck, map_location="cpu", weights_only=False)["step"]

for name in ["replay", "history", "pooled", "markov"]:
    for t in TICKERS:
        if not done(name, t):
            !cd {REPO} && python scripts/run_detective.py --tickers {t} --only {name}
            save()
            assert done(name, t), f"detective failed for {name} {t}: see the output above"
for name in list(RUNS) + ["full_last"]:
    ck = f"{DRIVE}/checkpoints/{name}.pt"
    assert os.path.exists(ck), f"missing {ck}: run the training cell first"
    best = ck.replace("_last.pt", ".pt")
    if name.endswith("_last") and os.path.exists(best) and step(best) == step(ck):
        print(name, "skipped: the best checkpoint is the final step, same weights")
        continue
    for t in TICKERS:
        if not done(name, t):
            !cd {REPO} && python scripts/run_detective.py --tickers {t} --only {name} --model {ck} --model-name {name}
            save()
            assert done(name, t), f"detective failed for {name} {t}: see the output above"
print("detective done")

In [ ]:
# Shock test (~5 min per model and ticker on a GPU) and tracking test (~15 min for v4 on GOOG).
for f in ("shock.json", "track.json"):
    if os.path.exists(f"{DRIVE}/results/{f}"):
        shutil.copy(f"{DRIVE}/results/{f}", f"{REPO}/results/{f}")

def has(f, name):
    p = f"{REPO}/results/{f}"
    return os.path.exists(p) and all(name in json.load(open(p)).get(t, {}) for t in TICKERS)

for name in SHOCK_MODELS:
    assert os.path.exists(f"{DRIVE}/checkpoints/{name}.pt"), f"missing {name}.pt: train it first"
for name in SHOCK_MODELS:
    ck = f"{DRIVE}/checkpoints/{name}.pt"
    if not has("shock.json", name):
        !cd {REPO} && python -u scripts/shock_test.py --baselines --model {name}={ck}
        save()
        assert has("shock.json", name), f"shock test for {name} failed: see the output above"
for name in TRACK_MODELS:
    ck = f"{DRIVE}/checkpoints/{name}.pt"
    if not has("track.json", name):
        !cd {REPO} && python -u scripts/track_test.py --model {name}={ck}
        save()
        assert has("track.json", name), f"tracking test for {name} failed: see the output above"
print("done: download MyDrive/lobgen/results/ and MyDrive/lobgen/checkpoints/ (v3*, v4*)")